In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/GLAMOR_Project"
DATASET = PROJECT + "/dataset/citation_network.csv"
CODE = PROJECT + "/code"
GLAMOR = PROJECT + "/GLAMOR"
OUTPUTS = PROJECT + "/outputs"

os.makedirs(CODE, exist_ok=True)
os.makedirs(GLAMOR, exist_ok=True)
os.makedirs(OUTPUTS, exist_ok=True)

assert os.path.isfile(DATASET), "Dataset not found"

print("Dataset connected successfully")

Dataset connected successfully


In [ ]:
import pandas as pd

df = pd.read_csv(DATASET)

print("Total records:", len(df))
print("Total columns:", len(df.columns))
print("Columns:", df.columns.tolist())

/tmp/ipykernel_534/2019060200.py:3: DtypeWarning: Columns (0,3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATASET)


Total records: 4609327
Total columns: 8
Columns: ['id', 'title', 'year', 'n_citation', 'doc_type', 'reference_count', 'references', 'doi']


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR
!git clone https://github.com/Zafar-southeast/GLAMOR.git .

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
Cloning into '.'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 55 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 43.23 KiB | 670.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR
!python -m pip install -e ".[dev,kg-validation]"

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
Obtaining file:///content/drive/MyDrive/GLAMOR_Project/GLAMOR
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 74.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 130.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 130.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.0 MB/s  0:00:00
  Building editable for glamor-citation (pyproject.toml) ... done
  Created wheel for glamor-citation: filename=glamor_citation-1.3.0-0.editable-py3-n

In [ ]:
!cat configs/glamor/dblp_v12.yaml

seed: 1203
output_dir: artifacts/glamor_dblp_v12

dataset:
  # https://originalstatic.aminer.cn/misc/dblp.v12.7z
  path: data/raw/dblp_v12.json
  format: auto
  train_ratio: 0.80

knowledge_graph:
  # The paper does not publish its ontology namespace. Replace this example
  # namespace with a controlled project URI before publishing an RDF endpoint.
  base_uri: https://example.org/glamor/
  shapes_path: schemas/glamor_shapes.ttl

walks:
  # N is not usable in the GLAMOR paper ("N around alpha"). 450,000 follows
  # the official implementation of the cited WalkLM method.
  num_walks: 450000
  termination_probability: 0.05
  max_steps: 1000000  # Cited WalkLM implementation; alpha normally stops far earlier.
  validation_ratio: 0.15
  corpus_split_seed: 42  # Cited WalkLM implementation.

language_model:
  model_name: distilroberta-base
  seed: 42  # TrainingArguments default used by the cited WalkLM implementation.
  mlm_probability: 0.15
  batch_size: 16
  epochs: 6
  learning_rate: 0.

In [ ]:
import pandas as pd

DATASET = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"

sample = pd.read_csv(DATASET, nrows=3)
print(sample.to_string())

     id                                                                                                                                                            title    year  n_citation    doc_type  reference_count                                                                                                                                                            references                                           doi
0  1091  Preliminary Design of a Network Protocol Learning Tool Based on the Comprehension of High School Students: Design by an Empirical Study Using a Simple Mind Map  2013.0           1  Conference              2.0                                                                                                                                                 2005687710;2018037215  https://doi.org/10.1007/978-3-642-39476-8_19
1  1388                                                                                                        Further Results on Independen

In [ ]:
import pandas as pd
import os

source = DATASET
output = os.path.join(PROJECT, "dataset", "dblp_v12.txt")

total = 0
missing_ids = 0

with open(output, "w", encoding="utf-8") as out:
    for chunk in pd.read_csv(
        source,
        dtype=str,
        keep_default_na=False,
        chunksize=50000
    ):
        for _, row in chunk.iterrows():
            paper_id = row["id"].strip()

            if not paper_id:
                missing_ids += 1
                paper_id = f"missing_id_row_{total}"

            title = row["title"].replace("\n", " ").replace("\r", " ").strip()
            year = row["year"].strip()

            out.write(f"#index{paper_id}\n")
            out.write(f"#*{title}\n")

            if year:
                out.write(f"#t{year}\n")

            references = row["references"].strip()

            if references:
                for ref in references.split(";"):
                    ref = ref.strip()
                    if ref:
                        out.write(f"#%{ref}\n")

            total += 1

print("Records converted:", total)
print("Missing original IDs:", missing_ids)
print("Output file:", output)
print("Output size (GB):", round(os.path.getsize(output) / (1024**3), 3))

Records converted: 4609327
Missing original IDs: 0
Output file: /content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12.txt
Output size (GB): 0.961


In [ ]:
df.shape

(4609327, 8)

In [ ]:
config = "configs/glamor/dblp_v12.yaml"

with open(config, "r") as f:
    content = f.read()

content = content.replace(
    "path: data/raw/dblp_v12.json",
    "path: /content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12.txt"
)

with open(config, "w") as f:
    f.write(content)

print("Full dataset path updated.")

Full dataset path updated.


In [ ]:
!sed -n '1,90p' src/citation_models/cli.py
!sed -n '1,100p' src/citation_models/common/validation.py

from __future__ import annotations

import argparse
import json
import math
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np

from citation_models.common.config import load_config
from citation_models.common.io import load_id_list, load_papers, save_id_list
from citation_models.common.manifest import write_run_manifest
from citation_models.common.metrics import evaluate_rankings
from citation_models.common.ranking import required_ranking_depth
from citation_models.common.split import evaluable_truth, random_paper_split
from citation_models.common.validation import validate_records, validate_split


def _records_and_split(config: dict[str, Any]):
    records = load_papers(config["dataset"]["path"], config["dataset"].get("format", "auto"))
    report = validate_records(records)
    report.raise_for_errors()
    split = random_paper_split(
        records,
        train_ratio=float(config["dataset"].get("train_ratio", 0.8)),
        seed=

In [ ]:
!sed -n '1,55p' src/citation_models/cli.py

from __future__ import annotations

import argparse
import json
import math
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np

from citation_models.common.config import load_config
from citation_models.common.io import load_id_list, load_papers, save_id_list
from citation_models.common.manifest import write_run_manifest
from citation_models.common.metrics import evaluate_rankings
from citation_models.common.ranking import required_ranking_depth
from citation_models.common.split import evaluable_truth, random_paper_split
from citation_models.common.validation import validate_records, validate_split


def _records_and_split(config: dict[str, Any]):
    records = load_papers(config["dataset"]["path"], config["dataset"].get("format", "auto"))
    report = validate_records(records)
    report.raise_for_errors()
    split = random_paper_split(
        records,
        train_ratio=float(config["dataset"].get("train_ratio", 0.8)),
        seed=

In [ ]:
!sed -n '90,145p' src/citation_models/common/io.py

            line = raw_line.rstrip("\n")
            if not line:
                record = finish()
                if record:
                    yield record
            elif line.startswith("#index"):
                current["id"] = line[6:].strip()
            elif line.startswith("#*"):
                current["title"] = line[2:].strip()
            elif line.startswith("#@"):
                current["authors"] = [x.strip() for x in line[2:].split(",") if x.strip()]
            elif line.startswith("#t"):
                current["year"] = line[2:].strip()
            elif line.startswith("#c"):
                current["venue"] = line[2:].strip()
            elif line.startswith("#%"):
                current.setdefault("references", []).append(line[2:].strip())
            elif line.startswith("#!"):
                current["abstract"] = line[2:].strip()
        record = finish()
        if record:
            yield record


def iter_papers(path: str | Path, fmt: str = "auto") -> 

In [ ]:
!sed -n '1,90p' src/citation_models/common/io.py

from __future__ import annotations

import json
from collections.abc import Iterable, Iterator
from pathlib import Path

from .records import PaperRecord


def _iter_json_array(stream, chunk_size: int = 1024 * 1024) -> Iterator[dict]:
    """Incrementally parse a JSON array without loading the archive twice."""
    decoder = json.JSONDecoder()
    buffer = ""
    started = ended = False
    while True:
        chunk = stream.read(chunk_size)
        eof = not chunk
        buffer += chunk
        cursor = 0
        while True:
            while cursor < len(buffer) and buffer[cursor].isspace():
                cursor += 1
            if not started:
                if cursor >= len(buffer):
                    break
                if buffer[cursor] != "[":
                    raise ValueError("Expected a JSON array")
                started = True
                cursor += 1
                continue
            while cursor < len(buffer) and (
                buffer[cursor].isspace() 

In [ ]:
!cat src/citation_models/common/split.py

from __future__ import annotations

import random
from collections.abc import Sequence
from dataclasses import dataclass

from .records import PaperRecord


@dataclass(frozen=True)
class PaperSplit:
    train: tuple[str, ...]
    test: tuple[str, ...]


def random_paper_split(
    records: Sequence[PaperRecord], train_ratio: float = 0.8, seed: int = 1203
) -> PaperSplit:
    if not 0.0 < train_ratio < 1.0:
        raise ValueError("train_ratio must lie strictly between 0 and 1")
    ids = sorted(record.id for record in records)
    random.Random(seed).shuffle(ids)
    boundary = max(1, min(len(ids) - 1, int(len(ids) * train_ratio)))
    return PaperSplit(tuple(sorted(ids[:boundary])), tuple(sorted(ids[boundary:])))


def evaluable_truth(
    query: PaperRecord, candidate_ids: set[str], *, minimum_relevant: int = 1
) -> set[str]:
    truth = set(query.references) & candidate_ids
    return truth if len(truth) >= minimum_relevant else set()



In [ ]:
!head -n 12 /content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12.txt

#index1091
#*Preliminary Design of a Network Protocol Learning Tool Based on the Comprehension of High School Students: Design by an Empirical Study Using a Simple Mind Map
#t2013.0
#%2005687710
#%2018037215
#index1388
#*Further Results on Independence in Direct-Product Graphs.
#t2000.0
#index1674
#*A methodology for the physically accurate visualisation of roman polychrome statuary
#t2011.0
#%1535888970


In [ ]:
!grep -A5 '^dataset:' configs/glamor/dblp_v12.yaml

dataset:
  # https://originalstatic.aminer.cn/misc/dblp.v12.7z
  path: /content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12.txt
  format: aminer
  train_ratio: 0.80



In [ ]:
!sed -n '65,105p' src/citation_models/common/io.py

            first = first.strip()
        if first.startswith("["):
            stream.seek(0)
            yield from _iter_json_array(stream)
            return
        yield json.loads(first.rstrip(","))
        for line in stream:
            line = line.strip().rstrip(",")
            if line and line not in ("]", "["):
                yield json.loads(line)


def _iter_legacy_aminer(path: Path) -> Iterator[dict]:
    current: dict = {}

    def finish() -> dict | None:
        nonlocal current
        if not current:
            return None
        record = current
        current = {}
        return record

    with path.open("r", encoding="utf-8", errors="replace") as stream:
        for raw_line in stream:
            line = raw_line.rstrip("\n")
            if not line:
                record = finish()
                if record:
                    yield record
            elif line.startswith("#index"):
                current["id"] = line[6:].strip()
            elif line.s

In [ ]:
!grep -n -A70 "def load" src/citation_models/common/io.py

125:def load_papers(path: str | Path, fmt: str = "auto") -> list[PaperRecord]:
126-    return list(iter_papers(path, fmt))
127-
128-
129-def save_papers(records: Iterable[PaperRecord], path: str | Path) -> None:
130-    path = Path(path)
131-    path.parent.mkdir(parents=True, exist_ok=True)
132-    with path.open("w", encoding="utf-8") as stream:
133-        for record in records:
134-            stream.write(json.dumps(record.to_dict(), ensure_ascii=False) + "\n")
135-
136-
137:def load_id_list(path: str | Path) -> list[str]:
138-    with Path(path).open("r", encoding="utf-8") as stream:
139-        return [line.strip() for line in stream if line.strip()]
140-
141-
142-def save_id_list(ids: Iterable[str], path: str | Path) -> None:
143-    path = Path(path)
144-    path.parent.mkdir(parents=True, exist_ok=True)
145-    path.write_text("".join(f"{item}\n" for item in ids), encoding="utf-8")


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

import sys
sys.path.insert(0, "src")

from citation_models.common.io import load_papers

print("GLAMOR module imported successfully")

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
GLAMOR module imported successfully


In [ ]:
import pandas as pd
import os

source = DATASET
output = os.path.join(PROJECT, "dataset", "dblp_v12.txt")
temp = output + ".tmp"

total = 0
columns = ["id", "title", "year", "references"]

with open(temp, "w", encoding="utf-8") as out:
    for chunk in pd.read_csv(
        source,
        dtype=str,
        keep_default_na=False,
        chunksize=100000
    ):
        for paper_id, title, year, refs in chunk[columns].itertuples(
            index=False, name=None
        ):
            out.write(f"#index{paper_id.strip()}\n")
            out.write(f"#*{title.strip()}\n")

            if year.strip():
                out.write(f"#t{year.strip()}\n")

            for ref in refs.split(";"):
                if ref.strip():
                    out.write(f"#%{ref.strip()}\n")

            out.write("\n")
            total += 1

os.replace(temp, output)

print("Records converted:", total)
print("Conversion completed.")

Records converted: 4609327
Conversion completed.


In [ ]:
records = load_papers(
    "/content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12.txt",
    "aminer"
)

print("Records loaded:", len(records))

Records loaded: 4609327


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

file = "/content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12.txt"

print("Converted dataset exists:", os.path.exists(file))

Mounted at /content/drive
Converted dataset exists: True


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"

df = pd.read_csv(path, usecols=["id", "title"])

print("Total records:", len(df))
print("Duplicate ID records:", df["id"].duplicated(keep=False).sum())
print("Empty titles:", df["title"].isna().sum() + (df["title"].astype(str).str.strip() == "").sum())

print("\nDuplicate ID examples:")
print(df[df["id"].duplicated(keep=False)].head(20).to_string(index=False))

print("\nRecords with empty titles:")
print(df[df["title"].isna() | (df["title"].astype(str).str.strip() == "")].head(20).to_string(index=False))

/tmp/ipykernel_398/1221043122.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=["id", "title"])


Total records: 4609327
Duplicate ID records: 7494
Empty titles: 9

Duplicate ID examples:
                                      id                                                                                                                               title
                                    ight                                                                                                                                 NaN
                                    ight                                                                                                                                 NaN
ho/mu$ Approximations for Control Design                                                                                                                              2014.0
ho/mu$ Approximations for Control Design                                                                                                                              2013.0
                              2957531214     

In [ ]:
path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"

with open(path, "r", encoding="utf-8", errors="replace") as f:
    for i in range(6):
        print(f"Line {i+1}: {f.readline().rstrip()[:500]}")

Line 1: id,title,year,n_citation,doc_type,reference_count,references,doi
Line 2: 1091,Preliminary Design of a Network Protocol Learning Tool Based on the Comprehension of High School Students: Design by an Empirical Study Using a Simple Mind Map,2013.0,1,Conference,2.0,2005687710;2018037215,https://doi.org/10.1007/978-3-642-39476-8_19
Line 3: 1388,Further Results on Independence in Direct-Product Graphs.,2000.0,1,Journal,,,
Line 4: 1674,A methodology for the physically accurate visualisation of roman polychrome statuary,2011.0,1,Conference,15.0,1535888970;1992876689;1993710814;2035653341;2043970887;2087778487;2094478046;2100468662;2110331535;2120448006;2138624212;2149970020;2150266006;2296384428;2403283736,https://doi.org/10.2312/VAST/VAST11/137-144
Line 5: 1688,"Comparison of GARCH, Neural Network and Support Vector Machine in Financial Time Series Prediction",2009.0,6,Conference,3.0,1560724230;1986968751;2156909104,https://doi.org/10.1007/978-3-642-11164-8_97
Line 6: 5411,COMPARING G

In [ ]:
import csv
from collections import Counter

path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"

counts = Counter()
examples = []

with open(path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    for line_no, row in enumerate(reader, start=2):
        counts[len(row)] += 1
        if len(row) != 8 and len(examples) < 10:
            examples.append((line_no, len(row), row[:10]))

print("Expected columns:", len(header))
print("Column-count distribution:", counts)
print("Problem-row examples:")
for item in examples:
    print(item)


Expected columns: 8
Column-count distribution: Counter({8: 4609213, 2: 52, 7: 52, 1: 9, 3: 1})
Problem-row examples:
(19208, 2, ['17261976', 'A Multiphysics Frequency-Dependent Model of an ${'])
(19209, 1, ['m IP}^{2}{'])
(19210, 7, ['m C}$ Actuator', '2014.0', '5', 'Journal', '5.0', '1984717708;2019425123;2024991751;2042826198;2171074980', 'https://doi.org/10.1109/TIM.2014.2298172'])
(21852, 2, ['19616501', 'An Additive Decomposition of Harmonic Functions in '])
(21853, 7, ['m{I!R}^{3}', '2014.0', '1', 'Conference', '', '', 'https://doi.org/10.1007/978-3-319-09144-0_14'])
(231198, 2, ['213011619', 'Towards Symbolic Strategy Synthesis for leftlangle {leftlangle A '])
(231199, 1, ['ight'])
(231200, 1, ['angle } '])
(231201, 1, ['ight'])
(231202, 7, ['angle-LTL.', '2003.0', '0', '', '', '', ''])


In [ ]:
path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"

with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
    lines = f.readlines()

for i in range(19205, 19213):
    print(f"Line {i+1}: {lines[i].rstrip()}")

Line 19206: 17260778,Towards multimodal expression of laughter,2012.0,16,Conference,7.0,187812487;1495772310;1994884977;2005639253;2042768254;2151627472;2155890799,https://doi.org/10.1007/978-3-642-33197-8_24
Line 19207: 17261878,From Reputation Models and Systems to Reputation Ontologies,2011.0,8,Conference,27.0,21513368;175487544;1246277140;1502455058;1547249671;1555990437;1577346037;1602708608;1970157936;1990717194;2000397925;2072305035;2089561630;2090989797;2098685442;2104609247;2108605997;2117911068;2120312633;2134661798;2137665245;2139211192;2158606791;2162220364;2171548560;2337430557;2404497786,https://doi.org/10.1007/978-3-642-22200-9_10
Line 19208: 17261976,A Multiphysics Frequency-Dependent Model of an ${
Line 19209: m IP}^{2}{
Line 19210: m C}$ Actuator,2014.0,5,Journal,5.0,1984717708;2019425123;2024991751;2042826198;2171074980,https://doi.org/10.1109/TIM.2014.2298172
Line 19211: 17265218,Reasoning with logical proportions,2010.0,40,Conference,15.0,109699022;1489820822;15199

In [ ]:
import csv
import re

source = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"
output = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_repaired.csv"

def is_new_record(line):
    return re.match(r"^\d+,", line) is not None

total = 0
valid = 0
invalid = 0
current = None

with open(source, "r", encoding="utf-8-sig", errors="replace") as src, \
     open(output, "w", encoding="utf-8", newline="") as dst:

    reader = csv.writer(dst)

    header = next(src)
    reader.writerow(next(csv.reader([header])))

    def save_record(record):
        global total, valid, invalid
        if record is None:
            return
        total += 1
        try:
            row = next(csv.reader([record]))
            if len(row) == 8:
                reader.writerow(row)
                valid += 1
            else:
                invalid += 1
        except Exception:
            invalid += 1

    for line in src:
        line = line.rstrip("\r\n")

        if is_new_record(line):
            save_record(current)
            current = line
        elif current is not None:
            current += " " + line.strip()

    save_record(current)

print("Reconstruction completed.")
print("Total reconstructed records:", total)
print("Valid 8-column records:", valid)
print("Invalid records:", invalid)
print("Saved file:", output)

Reconstruction completed.
Total reconstructed records: 4609266
Valid 8-column records: 4609265
Invalid records: 1
Saved file: /content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_repaired.csv


In [ ]:
import csv
import re

source = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"
current = None
invalid_records = []

with open(source, "r", encoding="utf-8-sig", errors="replace") as f:
    next(f)

    for line in f:
        line = line.rstrip("\r\n")

        if re.match(r"^\d+,", line):
            if current is not None:
                row = next(csv.reader([current]))
                if len(row) != 8:
                    invalid_records.append((len(row), current))
            current = line
        elif current is not None:
            current += " " + line.strip()

    if current is not None:
        row = next(csv.reader([current]))
        if len(row) != 8:
            invalid_records.append((len(row), current))

for count, record in invalid_records:
    print("Column count:", count)
    print("Record:", record)


Column count: 3
Record: 1541758250,Fault tolerance in distributed systems,199


In [ ]:
path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"

with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
    lines = f.readlines()

matches = [i for i, line in enumerate(lines)
           if line.startswith("1541758250,")]

for i in matches:
    for j in range(max(0, i-2), min(len(lines), i+5)):
        print(f"Line {j+1}: {lines[j].rstrip()}")

Line 4609369: 1541754030,Underwater love: building Tristão and Isolda's personalities,1999.0,6,Journal,11.0,1505141137;1531461244;1560725174;1570627388;1993584577;2022638915;2037513958;2042850474;2106239555;2165934840;2167877412,https://doi.org/10.1007/3-540-48317-9_11
Line 4609370: 1541829547,SuperIso: A program for calculating the isospin asymmetry of B→K∗γ in the MSSM,2008.0,202,Journal,1.0,2068671056,https://doi.org/10.1016/j.cpc.2007.12.006
Line 4609371: 1541758250,Fault tolerance in distributed systems,199


In [ ]:
import re

path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network.csv"

with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
    next(f)
    unmatched = [
        (i, line.rstrip()[:250])
        for i, line in enumerate(f, start=2)
        if line.strip() and not re.match(r"^\d+,", line)
    ]

print("Unmatched lines:", len(unmatched))
for item in unmatched[:20]:
    print(item)

Unmatched lines: 104
(19209, 'm IP}^{2}{')
(19210, 'm C}$ Actuator,2014.0,5,Journal,5.0,1984717708;2019425123;2024991751;2042826198;2171074980,https://doi.org/10.1109/TIM.2014.2298172')
(21853, 'm{I!R}^{3},2014.0,1,Conference,,,https://doi.org/10.1007/978-3-319-09144-0_14')
(206857, 'u$-SVR.",2007.0,2,Conference,,,')
(231200, 'ight')
(231201, 'angle }')
(231202, 'ight')
(231203, 'angle-LTL.,2003.0,0,,,,')
(241039, 'ceil DEPTH(2 lceil log n')
(241040, 'ceil -3).,1987.0,20,Conference,,,')
(241445, 'm PROFIL}_{R}$ : Toward Preserving Privacy and Functionality in Geosocial Networks,2014.0,5,Journal,30.0,1492124753;1555820810;1566461722;1601001795;1655958391;1781642226;1970606468;1979476035;2011112377;2025181331;2043944888;2054706300;2056773559;20')
(376780, 'ight) –statistical convergence of order alpha",2015.0,4,Journal,4.0,2009041950;2020524104;2045193784;2083616714,https://doi.org/10.1007/s10998-015-0087-y')
(403267, 'u -Support Vector Regression",2001.0,14,Conference,5.0,2147993235;215

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_repaired.csv"

df = pd.read_csv(path)

print("Repaired records:", len(df))
print("Duplicate IDs:", df["id"].duplicated().sum())
print("Empty titles:", df["title"].isna().sum())

Repaired records: 4609265
Duplicate IDs: 3745
Empty titles: 0


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_repaired.csv"
df = pd.read_csv(path)

duplicates = df[df["id"].duplicated(keep=False)].sort_values("id")

print("Duplicate ID groups:", duplicates["id"].nunique())
print("Rows involved:", len(duplicates))
print("\nFirst 10 duplicate groups:")
print(duplicates.groupby("id").head(2).head(20).to_string(index=False))

print("\nCompletely identical duplicate rows:", df.duplicated().sum())

Duplicate ID groups: 3745
Rows involved: 7490

First 10 duplicate groups:
        id                                                                                                            title   year  n_citation doc_type  reference_count                                                                                                                                                                                                                                                                                                                                                                          references                                          doi
2957374109                                                          Hyperspectral Image Denoising using Dictionary Learning 2019.0           0      NaN              NaN                                                                                                                                                                            

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_repaired.csv"
df = pd.read_csv(path)

dup = df[df["id"].duplicated(keep=False)]
for paper_id, group in dup.groupby("id"):
    if len(group.drop_duplicates()) > 1:
        print("Conflicting ID:", paper_id)
        print(group.to_string(index=False))

Conflicting ID: 2960230874
        id                                                                           title   year  n_citation doc_type  reference_count                                                                                                                                                    references                                          doi
2960230874 To enter or not to enter? A competitive analysis with minimum quality standards 2019.0           0  Journal              2.0 2018451260;20328124172487;2145554279;2146707163;2155151262;2156458885;2157321686;2169466597;2519420704;2605327034;2795402696;2841091033;2909184935;2927240970   https://doi.org/10.1007/s10707-019-00375-w
2960230874 To enter or not to enter? A competitive analysis with minimum quality standards 2019.0           0  Journal              2.0                                                                                                                                         2018451260;2032844294 h

In [ ]:
group = df[df["id"] == 2960230874]

for col in df.columns:
    values = group[col].tolist()
    if values[0] != values[1]:
        print("Different column:", col)
        print("Record 1:", values[0])
        print("Record 2:", values[1])

Different column: references
Record 1: 2018451260;20328124172487;2145554279;2146707163;2155151262;2156458885;2157321686;2169466597;2519420704;2605327034;2795402696;2841091033;2909184935;2927240970
Record 2: 2018451260;2032844294
Different column: doi
Record 1: https://doi.org/10.1007/s10707-019-00375-w
Record 2: https://doi.org/10.1016/j.matcom.2019.07.003


In [ ]:
import pandas as pd

source = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_repaired.csv"
output = "/content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_unique.csv"
mapping_file = "/content/drive/MyDrive/GLAMOR_Project/dataset/paper_id_mapping.csv"

df = pd.read_csv(source)

df.insert(0, "original_id", df["id"])
df["id"] = range(10000000000, 10000000000 + len(df))

mapping = df[["original_id", "id"]].drop_duplicates("original_id", keep="first")

df.to_csv(output, index=False)
mapping.to_csv(mapping_file, index=False)

print("Records preserved:", len(df))
print("Unique internal IDs:", df["id"].nunique())
print("Original IDs mapped:", len(mapping))
print("Dataset saved:", output)
print("Mapping saved:", mapping_file)

Records preserved: 4609265
Unique internal IDs: 4609265
Original IDs mapped: 4605520
Dataset saved: /content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_unique.csv
Mapping saved: /content/drive/MyDrive/GLAMOR_Project/dataset/paper_id_mapping.csv


In [ ]:
import pandas as pd

folder = "/content/drive/MyDrive/GLAMOR_Project/dataset/"

mapping_df = pd.read_csv(
    folder + "paper_id_mapping.csv",
    dtype=str
)

id_map = dict(zip(mapping_df["original_id"], mapping_df["id"]))

source = folder + "citation_network_unique.csv"
output = folder + "citation_network_ready.csv"

total_refs = 0
unmatched_refs = 0

first = True

for chunk in pd.read_csv(source, dtype={"original_id": str}, chunksize=50000):

    def update_references(value):
        global total_refs, unmatched_refs

        if pd.isna(value) or not str(value).strip():
            return value

        updated = []

        for ref in str(value).split(";"):
            ref = ref.strip()
            if not ref:
                continue

            total_refs += 1

            if ref in id_map:
                updated.append(id_map[ref])
            else:
                updated.append(ref)
                unmatched_refs += 1

        return ";".join(updated)

    chunk["references"] = chunk["references"].apply(update_references)

    chunk.to_csv(
        output,
        index=False,
        mode="w" if first else "a",
        header=first
    )

    first = False

print("Reference updating completed.")
print("References processed:", total_refs)
print("References without matching IDs:", unmatched_refs)
print("Saved:", output)

Reference updating completed.
References processed: 43065123
References without matching IDs: 2142525
Saved: /content/drive/MyDrive/GLAMOR_Project/dataset/citation_network_ready.csv


In [ ]:
import pandas as pd

folder = "/content/drive/MyDrive/GLAMOR_Project/dataset/"
source = folder + "citation_network_ready.csv"
output = folder + "dblp_v12_ready.txt"

converted = 0

with open(output, "w", encoding="utf-8") as out:
    for chunk in pd.read_csv(source, chunksize=50000):

        for _, row in chunk.iterrows():
            out.write(f"#index{row['id']}\n")

            title = "" if pd.isna(row["title"]) else str(row["title"])
            out.write(f"#*{title}\n")

            if pd.notna(row["year"]):
                out.write(f"#t{row['year']}\n")

            if pd.notna(row["references"]):
                for ref in str(row["references"]).split(";"):
                    if ref.strip():
                        out.write(f"#%{ref.strip()}\n")

            out.write("\n")
            converted += 1

print("Conversion completed.")
print("Records converted:", converted)
print("Saved file:", output)

Conversion completed.
Records converted: 4609265
Saved file: /content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12_ready.txt


In [ ]:
config = "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/configs/glamor/dblp_v12.yaml"

with open(config, "r") as f:
    content = f.read()

content = content.replace(
    "/content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12.txt",
    "/content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12_ready.txt"
)

with open(config, "w") as f:
    f.write(content)

print("Configuration updated to use dblp_v12_ready.txt")

Configuration updated to use dblp_v12_ready.txt


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!nohup glamor validate-data --config configs/glamor/dblp_v12.yaml > /content/drive/MyDrive/GLAMOR_Project/outputs/validation.log 2>&1 &

/content/drive/MyDrive/GLAMOR_Project/GLAMOR


In [ ]:
!tail -30 /content/drive/MyDrive/GLAMOR_Project/outputs/validation.log

In [ ]:
!pgrep -af "glamor validate-data"
!ls -lh /content/drive/MyDrive/GLAMOR_Project/outputs/validation.log

14712 /usr/bin/python3 /usr/local/bin/glamor validate-data --config configs/glamor/dblp_v12.yaml
-rw------- 1 root root 0 Oct 31 16:37 /content/drive/MyDrive/GLAMOR_Project/outputs/validation.log


In [ ]:
!ps -p 14712 -o pid,etime,%cpu,%mem,stat,cmd

    PID     ELAPSED %CPU %MEM STAT CMD


In [ ]:
!dmesg -T 2>/dev/null | grep -i -E "killed process|out of memory|oom" | tail -10

[Sat Oct 31 16:34:50 2026] glamor invoked oom-killer: gfp_mask=0xcc0(GFP_KERNEL), order=0, oom_score_adj=0
[Sat Oct 31 16:34:50 2026]  oom_kill_process+0x10c/0x210
[Sat Oct 31 16:34:50 2026] [  pid  ]   uid  tgid total_vm      rss pgtables_bytes swapents oom_score_adj name
[Sat Oct 31 16:34:50 2026] oom-kill:constraint=CONSTRAINT_MEMCG,nodemask=(null),cpuset=jupyter-children,mems_allowed=0,oom_memcg=/jupyter-children,task_memcg=/jupyter-children,task=glamor,pid=48795,uid=0
[Sat Oct 31 16:34:50 2026] Memory cgroup out of memory: Killed process 48795 (glamor) total-vm:7166748kB, anon-rss:6890048kB, file-rss:13696kB, shmem-rss:0kB, UID:0 pgtables:13840kB oom_score_adj:0
[Sat Oct 31 16:40:45 2026] glamor invoked oom-killer: gfp_mask=0xcc0(GFP_KERNEL), order=0, oom_score_adj=0
[Sat Oct 31 16:40:45 2026]  oom_kill_process+0x10c/0x210
[Sat Oct 31 16:40:46 2026] [  pid  ]   uid  tgid total_vm      rss pgtables_bytes swapents oom_score_adj name
[Sat Oct 31 16:40:46 2026] oom-kill:constraint=CON

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '1,180p' src/citation_models/cli.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
from __future__ import annotations

import argparse
import json
import math
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np

from citation_models.common.config import load_config
from citation_models.common.io import load_id_list, load_papers, save_id_list
from citation_models.common.manifest import write_run_manifest
from citation_models.common.metrics import evaluate_rankings
from citation_models.common.ranking import required_ranking_depth
from citation_models.common.split import evaluable_truth, random_paper_split
from citation_models.common.validation import validate_records, validate_split


def _records_and_split(config: dict[str, Any]):
    records = load_papers(config["dataset"]["path"], config["dataset"].get("format", "auto"))
    report = validate_records(records)
    report.raise_for_errors()
    split = random_paper_split(
        records,
        train_ratio=float(config["data

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!grep -n -A65 -B5 "def command_validate_data" src/citation_models/cli.py
!grep -n -A65 -B5 "def _records_and_split" src/citation_models/cli.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
16-from citation_models.common.ranking import required_ranking_depth
17-from citation_models.common.split import evaluable_truth, random_paper_split
18-from citation_models.common.validation import validate_records, validate_split
19-
20-
21:def _records_and_split(config: dict[str, Any]):
22-    records = load_papers(config["dataset"]["path"], config["dataset"].get("format", "auto"))
23-    report = validate_records(records)
24-    report.raise_for_errors()
25-    split = random_paper_split(
26-        records,
27-        train_ratio=float(config["dataset"].get("train_ratio", 0.8)),
28-        seed=int(config.get("seed", 1203)),
29-    )
30-    validate_split(split.train, split.test, {record.id for record in records})
31-    return records, split, report
32-
33-
34-def _records_and_saved_split(config: dict[str, Any], output: Path):
35-    records = load_papers(config["dataset"]["path"], config["dataset"].get("format", "auto"))
36-    report 

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '1,220p' src/citation_models/common/validation.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
from __future__ import annotations

from collections import Counter
from collections.abc import Iterable
from dataclasses import dataclass, field

from .records import PaperRecord


@dataclass
class ValidationReport:
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    statistics: dict[str, int | float] = field(default_factory=dict)

    @property
    def ok(self) -> bool:
        return not self.errors

    def raise_for_errors(self) -> None:
        if self.errors:
            raise ValueError("Dataset validation failed:\n- " + "\n- ".join(self.errors))


def validate_records(records: Iterable[PaperRecord]) -> ValidationReport:
    records = list(records)
    report = ValidationReport()
    ids = [r.id for r in records]
    duplicate_ids = sorted(x for x, n in Counter(ids).items() if n > 1)
    if duplicate_ids:
        report.errors.append(f"Duplicate paper ids: {duplicate_ids[:20]}

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '1,45p' src/citation_models/cli.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
from __future__ import annotations

import argparse
import json
import math
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np

from citation_models.common.config import load_config
from citation_models.common.io import load_id_list, load_papers, save_id_list
from citation_models.common.manifest import write_run_manifest
from citation_models.common.metrics import evaluate_rankings
from citation_models.common.ranking import required_ranking_depth
from citation_models.common.split import evaluable_truth, random_paper_split
from citation_models.common.validation import validate_records, validate_split


def _records_and_split(config: dict[str, Any]):
    records = load_papers(config["dataset"]["path"], config["dataset"].get("format", "auto"))
    report = validate_records(records)
    report.raise_for_errors()
    split = random_paper_split(
        records,
        train_ratio=float(config["data

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!grep -n -A100 "def load_papers" src/citation_models/common/io.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
125:def load_papers(path: str | Path, fmt: str = "auto") -> list[PaperRecord]:
126-    return list(iter_papers(path, fmt))
127-
128-
129-def save_papers(records: Iterable[PaperRecord], path: str | Path) -> None:
130-    path = Path(path)
131-    path.parent.mkdir(parents=True, exist_ok=True)
132-    with path.open("w", encoding="utf-8") as stream:
133-        for record in records:
134-            stream.write(json.dumps(record.to_dict(), ensure_ascii=False) + "\n")
135-
136-
137-def load_id_list(path: str | Path) -> list[str]:
138-    with Path(path).open("r", encoding="utf-8") as stream:
139-        return [line.strip() for line in stream if line.strip()]
140-
141-
142-def save_id_list(ids: Iterable[str], path: str | Path) -> None:
143-    path = Path(path)
144-    path.parent.mkdir(parents=True, exist_ok=True)
145-    path.write_text("".join(f"{item}\n" for item in ids), encoding="utf-8")


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!grep -n -A100 "def iter_papers" src/citation_models/common/io.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
114:def iter_papers(path: str | Path, fmt: str = "auto") -> Iterator[PaperRecord]:
115-    path = Path(path)
116-    if fmt == "auto":
117-        with path.open("r", encoding="utf-8", errors="replace") as stream:
118-            prefix = stream.read(32).lstrip()
119-        fmt = "json" if prefix.startswith(("{", "[")) else "aminer"
120-    iterator = _iter_json_records(path) if fmt == "json" else _iter_legacy_aminer(path)
121-    for raw in iterator:
122-        yield PaperRecord.from_dict(raw)
123-
124-
125-def load_papers(path: str | Path, fmt: str = "auto") -> list[PaperRecord]:
126-    return list(iter_papers(path, fmt))
127-
128-
129-def save_papers(records: Iterable[PaperRecord], path: str | Path) -> None:
130-    path = Path(path)
131-    path.parent.mkdir(parents=True, exist_ok=True)
132-    with path.open("w", encoding="utf-8") as stream:
133-        for record in records:
134-            stream.write(json.dumps(record.to_dict(), 

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!grep -n -A100 "def random_paper_split" src/citation_models/common/split.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
16:def random_paper_split(
17-    records: Sequence[PaperRecord], train_ratio: float = 0.8, seed: int = 1203
18-) -> PaperSplit:
19-    if not 0.0 < train_ratio < 1.0:
20-        raise ValueError("train_ratio must lie strictly between 0 and 1")
21-    ids = sorted(record.id for record in records)
22-    random.Random(seed).shuffle(ids)
23-    boundary = max(1, min(len(ids) - 1, int(len(ids) * train_ratio)))
24-    return PaperSplit(tuple(sorted(ids[:boundary])), tuple(sorted(ids[boundary:])))
25-
26-
27-def evaluable_truth(
28-    query: PaperRecord, candidate_ids: set[str], *, minimum_relevant: int = 1
29-) -> set[str]:
30-    truth = set(query.references) & candidate_ids
31-    return truth if len(truth) >= minimum_relevant else set()
32-


In [ ]:
import shutil

source = "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/cli.py"
backup = "/content/drive/MyDrive/GLAMOR_Project/code/cli_original_backup.py"

shutil.copy2(source, backup)

print("Original GLAMOR code backed up successfully.")
print("Backup:", backup)

Original GLAMOR code backed up successfully.
Backup: /content/drive/MyDrive/GLAMOR_Project/code/cli_original_backup.py


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '1,125p' src/citation_models/common/validation.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
from __future__ import annotations

from collections import Counter
from collections.abc import Iterable
from dataclasses import dataclass, field

from .records import PaperRecord


@dataclass
class ValidationReport:
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    statistics: dict[str, int | float] = field(default_factory=dict)

    @property
    def ok(self) -> bool:
        return not self.errors

    def raise_for_errors(self) -> None:
        if self.errors:
            raise ValueError("Dataset validation failed:\n- " + "\n- ".join(self.errors))


def validate_records(records: Iterable[PaperRecord]) -> ValidationReport:
    records = list(records)
    report = ValidationReport()
    ids = [r.id for r in records]
    duplicate_ids = sorted(x for x, n in Counter(ids).items() if n > 1)
    if duplicate_ids:
        report.errors.append(f"Duplicate paper ids: {duplicate_ids[:20]}

In [ ]:
import shutil

source = "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/common/validation.py"
backup = "/content/drive/MyDrive/GLAMOR_Project/code/validation_original_backup.py"

shutil.copy2(source, backup)

print("Validation code backup completed.")

Validation code backup completed.


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '1,115p' src/citation_models/common/validation.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
from __future__ import annotations

from collections import Counter
from collections.abc import Iterable
from dataclasses import dataclass, field

from .records import PaperRecord


@dataclass
class ValidationReport:
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    statistics: dict[str, int | float] = field(default_factory=dict)

    @property
    def ok(self) -> bool:
        return not self.errors

    def raise_for_errors(self) -> None:
        if self.errors:
            raise ValueError("Dataset validation failed:\n- " + "\n- ".join(self.errors))


def validate_records(records: Iterable[PaperRecord]) -> ValidationReport:
    records = list(records)
    report = ValidationReport()
    ids = [r.id for r in records]
    duplicate_ids = sorted(x for x, n in Counter(ids).items() if n > 1)
    if duplicate_ids:
        report.errors.append(f"Duplicate paper ids: {duplicate_ids[:20]}

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '1,65p' src/citation_models/common/validation.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
from __future__ import annotations

from collections import Counter
from collections.abc import Iterable
from dataclasses import dataclass, field

from .records import PaperRecord


@dataclass
class ValidationReport:
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    statistics: dict[str, int | float] = field(default_factory=dict)

    @property
    def ok(self) -> bool:
        return not self.errors

    def raise_for_errors(self) -> None:
        if self.errors:
            raise ValueError("Dataset validation failed:\n- " + "\n- ".join(self.errors))


def validate_records(records: Iterable[PaperRecord]) -> ValidationReport:
    records = list(records)
    report = ValidationReport()
    ids = [r.id for r in records]
    duplicate_ids = sorted(x for x, n in Counter(ids).items() if n > 1)
    if duplicate_ids:
        report.errors.append(f"Duplicate paper ids: {duplicate_ids[:20]}

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!grep -n -A35 "^def validate_records" src/citation_models/common/validation.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
25:def validate_records(records: Iterable[PaperRecord]) -> ValidationReport:
26-    records = list(records)
27-    report = ValidationReport()
28-    ids = [r.id for r in records]
29-    duplicate_ids = sorted(x for x, n in Counter(ids).items() if n > 1)
30-    if duplicate_ids:
31-        report.errors.append(f"Duplicate paper ids: {duplicate_ids[:20]}")
32-    empty_titles = sum(not r.title for r in records)
33-    if empty_titles:
34-        report.errors.append(f"{empty_titles} paper records have empty titles")
35-    known = set(ids)
36-    dangling = sum(ref not in known for r in records for ref in r.references)
37-    self_citations = sum(r.id in r.references for r in records)
38-    if dangling:
39-        report.warnings.append(
40-            f"{dangling} references point outside the loaded corpus; they are ignored in evaluation"
41-        )
42-    if self_citations:
43-        report.warnings.append(f"{self_citations} self-refere

In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       5.6Gi       6.6Gi       6.7Mi       750Mi       7.1Gi
Swap:             0B          0B          0B


In [ ]:
!cat /sys/fs/cgroup/memory.max

max


In [ ]:
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/"
    "src/citation_models/common/validation.py"
)

source = path.read_text(encoding="utf-8")

start = source.index("def validate_records(")
end = source.index("\ndef validate_split(", start)

new_function = '''def validate_records(records: Iterable[PaperRecord]) -> ValidationReport:
    report = ValidationReport()

    ids = set()
    duplicate_ids = set()
    authors = set()
    venues = set()
    fields = set()
    concepts = set()

    empty_titles = 0

    for record in records:
        if record.id in ids:
            duplicate_ids.add(record.id)
        ids.add(record.id)

        if not record.title:
            empty_titles += 1

        authors.update(a.id for a in record.authors)
        if record.venue:
            venues.add(record.venue.id)
        fields.update(f.id for f in record.fields)
        concepts.update(c.id for c in record.concepts)

    if duplicate_ids:
        report.errors.append(
            f"Duplicate paper ids: {sorted(duplicate_ids)[:20]}"
        )

    if empty_titles:
        report.errors.append(
            f"{empty_titles} paper records have empty titles"
        )

    dangling = 0
    self_citations = 0
    citation_edges = 0
    for record in records:
        for ref in record.references:
            if ref not in ids:
                dangling += 1
            elif ref == record.id:
                self_citations += 1
            else:
                citation_edges += 1

    if dangling:
        report.warnings.append(
            f"{dangling} references point outside the loaded corpus; "
            "they are ignored in evaluation"
        )

    if self_citations:
        report.warnings.append(
            f"{self_citations} self-referential paper edges will be removed"
        )

    report.statistics = {
        "papers": len(records),
        "authors": len(authors),
        "venues": len(venues),
        "fields": len(fields),
        "concepts": len(concepts),
        "citation_edges_in_corpus": citation_edges,
        "cold_start_missing_authors": sum(
            record.cold_start for record in records
        ),
    }

    return report
'''

path.write_text(source[:start] + new_function + source[end:], encoding="utf-8")

print("Memory optimization applied to validation.py.")

Memory optimization applied to validation.py.


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!python -m py_compile src/citation_models/common/validation.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR


In [ ]:
import shutil

shutil.copy2(
    "/content/drive/MyDrive/GLAMOR_Project/code/validation_original_backup.py",
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/common/validation.py"
)

print("Original validation code restored.")

Original validation code restored.


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '115,230p' src/citation_models/cli.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
    validation_count = max(
        1,
        min(num_walks - 1, math.ceil(num_walks * validation_ratio)),
    )
    return set(np.random.RandomState(seed).permutation(num_walks)[:validation_count].tolist())


def command_prepare_glamor(args: argparse.Namespace) -> None:
    from citation_models.glamor.kg import build_glamor_kg, export_rdf
    from citation_models.glamor.walks import generate_walk_corpus

    config = load_config(args.config)
    records, split, _ = _records_and_split(config)
    by_id = {record.id: record for record in records}
    train_records = [by_id[x] for x in split.train]
    output = Path(config["output_dir"])
    output.mkdir(parents=True, exist_ok=True)
    save_id_list(split.train, output / "train_ids.txt")
    save_id_list(split.test, output / "test_ids.txt")
    write_run_manifest(
        output / "run_manifest.json",
        model="GLAMOR",
        dataset_path=config["dataset"]["path"],
        config=confi

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!grep -n -A100 "def command_prepare_glamor" src/citation_models/cli.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
122:def command_prepare_glamor(args: argparse.Namespace) -> None:
123-    from citation_models.glamor.kg import build_glamor_kg, export_rdf
124-    from citation_models.glamor.walks import generate_walk_corpus
125-
126-    config = load_config(args.config)
127-    records, split, _ = _records_and_split(config)
128-    by_id = {record.id: record for record in records}
129-    train_records = [by_id[x] for x in split.train]
130-    output = Path(config["output_dir"])
131-    output.mkdir(parents=True, exist_ok=True)
132-    save_id_list(split.train, output / "train_ids.txt")
133-    save_id_list(split.test, output / "test_ids.txt")
134-    write_run_manifest(
135-        output / "run_manifest.json",
136-        model="GLAMOR",
137-        dataset_path=config["dataset"]["path"],
138-        config=config,
139-        train_ids=split.train,
140-        test_ids=split.test,
141-    )
142-    graph = build_glamor_kg(train_records)
143-    graph.s

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '90,205p' src/citation_models/cli.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
            "evaluable_queries": evaluable,
            "skipped_without_relevant_training_citations": len(test_records) - evaluable,
            "candidate_papers": len(candidates),
            "map_k": int(evaluation.get("map_k", 10)),
            "ndcg_k": int(evaluation.get("ndcg_k", 100)),
            "recall_ks": list(evaluation.get("recall_ks", [20, 40, 60, 80, 100])),
            "materialized_ranking_depth": required_ranking_depth(
                len(candidates),
                recall_ks=evaluation.get("recall_ks", [20, 40, 60, 80, 100]),
                map_k=int(evaluation.get("map_k", 10)),
                ndcg_k=int(evaluation.get("ndcg_k", 100)),
            ),
            "cold_start_definition": "held-out paper with no author metadata",
        },
        "overall": metrics,
        "cold_start_missing_author": cold_metrics,
    }


def _walk_validation_indices(num_walks: int, validation_ratio: float, seed: int) -> set[int]

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '85,145p' src/citation_models/cli.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
        )
    evaluable = sum(bool(values) for values in truth.values())
    return {
        "protocol": {
            "held_out_queries": len(test_records),
            "evaluable_queries": evaluable,
            "skipped_without_relevant_training_citations": len(test_records) - evaluable,
            "candidate_papers": len(candidates),
            "map_k": int(evaluation.get("map_k", 10)),
            "ndcg_k": int(evaluation.get("ndcg_k", 100)),
            "recall_ks": list(evaluation.get("recall_ks", [20, 40, 60, 80, 100])),
            "materialized_ranking_depth": required_ranking_depth(
                len(candidates),
                recall_ks=evaluation.get("recall_ks", [20, 40, 60, 80, 100]),
                map_k=int(evaluation.get("map_k", 10)),
                ndcg_k=int(evaluation.get("ndcg_k", 100)),
            ),
            "cold_start_definition": "held-out paper with no author metadata",
        },
        "overall": m

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!grep -n -A100 "def build_glamor_kg" src/citation_models/glamor/kg.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
84:def build_glamor_kg(records: Iterable[PaperRecord]) -> AttributedKnowledgeGraph:
85-    """Build the attributed scholarly KG shown in GLAMOR Figure 1."""
86-    records = list(records)
87-    known_papers = {record.id for record in records}
88-    author_paper_counts = Counter(
89-        author.id
90-        for paper in records
91-        for author in {item.id: item for item in paper.authors}.values()
92-    )
93-    field_paper_counts = Counter(
94-        field.id
95-        for paper in records
96-        for field in {item.id: item for item in paper.fields}.values()
97-    )
98-    concept_paper_counts = Counter(
99-        concept.id
100-        for paper in records
101-        for concept in {item.id: item for item in paper.concepts}.values()
102-    )
103-    author_citation_counts: Counter[str] = Counter()
104-    authors_with_citation_counts: set[str] = set()
105-    for paper in records:
106-        if paper.citation_count is

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '70,150p' src/citation_models/glamor/kg.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
        payload = {
            "entities": [asdict(self.entities[x]) for x in sorted(self.entities)],
            "edges": [asdict(x) for x in self.edges],
        }
        with path.open("w", encoding="utf-8") as stream:
            json.dump(payload, stream, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, path: str | Path) -> AttributedKnowledgeGraph:
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        entities = {x["id"]: Entity(**x) for x in payload["entities"]}
        return cls(entities=entities, edges=[Edge(**x) for x in payload["edges"]])


def build_glamor_kg(records: Iterable[PaperRecord]) -> AttributedKnowledgeGraph:
    """Build the attributed scholarly KG shown in GLAMOR Figure 1."""
    records = list(records)
    known_papers = {record.id for record in records}
    author_paper_counts = Counter(
        author.id
        for paper in records
        for author in {item.id: item for

In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!cat configs/glamor/dblp_v12.yaml

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
seed: 1203
output_dir: artifacts/glamor_dblp_v12

dataset:
  # https://originalstatic.aminer.cn/misc/dblp.v12.7z
  path: /content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12_ready.txt
  format: aminer
  train_ratio: 0.80

knowledge_graph:
  # The paper does not publish its ontology namespace. Replace this example
  # namespace with a controlled project URI before publishing an RDF endpoint.
  base_uri: https://example.org/glamor/
  shapes_path: schemas/glamor_shapes.ttl

walks:
  # N is not usable in the GLAMOR paper ("N around alpha"). 450,000 follows
  # the official implementation of the cited WalkLM method.
  num_walks: 450000
  termination_probability: 0.05
  max_steps: 1000000  # Cited WalkLM implementation; alpha normally stops far earlier.
  validation_ratio: 0.15
  corpus_split_seed: 42  # Cited WalkLM implementation.

language_model:
  model_name: distilroberta-base
  seed: 42  # TrainingArguments default used by the cited WalkLM

In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive/GLAMOR_Project/GLAMOR"))

True


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!grep -n -E "^class |^def build_glamor_kg|^def export_rdf|def save|def from_" src/citation_models/glamor/kg.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
28:class Entity:
36:class Edge:
43:class AttributedKnowledgeGraph:
67:    def save(self, path: str | Path) -> None:
84:def build_glamor_kg(records: Iterable[PaperRecord]) -> AttributedKnowledgeGraph:
272:def export_rdf(


In [ ]:
%cd /content/drive/MyDrive/GLAMOR_Project/GLAMOR

!sed -n '43,125p' src/citation_models/glamor/kg.py

/content/drive/MyDrive/GLAMOR_Project/GLAMOR
class AttributedKnowledgeGraph:
    entities: dict[str, Entity]
    edges: list[Edge]

    def __post_init__(self) -> None:
        unknown = {
            endpoint
            for edge in self.edges
            for endpoint in (edge.source, edge.target)
            if endpoint not in self.entities
        }
        if unknown:
            raise ValueError(f"Edges reference unknown entities: {sorted(unknown)[:20]}")
        self.edges = sorted(set(self.edges))

    @cached_property
    def adjacency(self) -> dict[str, list[tuple[str, str]]]:
        result = {node_id: [] for node_id in self.entities}
        for edge in self.edges:
            result[edge.source].append((edge.relation, edge.target))
        for values in result.values():
            values.sort()
        return result

    def save(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
           

In [ ]:
from google.colab import files

path = "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor/kg.py"

files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

path = "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor/walks.py"

files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

path = "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor/textualize.py"

files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from pathlib import Path

root = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR")
backup = Path("/content/drive/MyDrive/GLAMOR_Project/code/memory_fix_backup")
backup.mkdir(parents=True, exist_ok=True)

for name in ["kg.py", "walks.py", "textualize.py"]:
    source = root / "src/citation_models/glamor" / name
    shutil.copy2(source, backup / name)

print("Original source files backed up successfully.")

Original source files backed up successfully.


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor")

# Update textualize.py
text_file = root / "textualize.py"
text = text_file.read_text()

old = '''    entity_texts = entity_texts or {
        node_id: textualize_entity(entity) for node_id, entity in graph.entities.items()
    }
    pieces = [entity_texts[walk[0][0]]]
    for node_id, incoming_relation in walk[1:]:
        if incoming_relation is None:
            raise ValueError("Every non-start node must have an incoming relation")
        pieces.append(textualize_edge(incoming_relation))
        pieces.append(entity_texts[node_id])
'''

new = '''    pieces = [textualize_entity(graph.entities[walk[0][0]])]
    for node_id, incoming_relation in walk[1:]:
        if incoming_relation is None:
            raise ValueError("Every non-start node must have an incoming relation")
        pieces.append(textualize_edge(incoming_relation))
        pieces.append(textualize_entity(graph.entities[node_id]))
'''

if old not in text:
    raise RuntimeError("Expected textualize.py code was not found. No changes made.")

text_file.write_text(text.replace(old, new))

# Update walks.py
walk_file = root / "walks.py"
text = walk_file.read_text()

old = '''    # GLAMOR states that rule-based F(.) is executed once per entity.
    entity_texts = {
        node_id: textualize_entity(entity) for node_id, entity in graph.entities.items()
    }
'''

if old not in text:
    raise RuntimeError("Expected walks.py code was not found.")

text = text.replace(old, "")

old_call = '''        yield textualize_walk(graph, walk, entity_texts)
'''
new_call = '''        yield textualize_walk(graph, walk)
'''

if old_call not in text:
    raise RuntimeError("Expected walk call was not found.")

walk_file.write_text(text.replace(old_call, new_call))

print("Updated textualize.py and walks.py successfully.")
print("Entity text is now generated only when needed.")

Updated textualize.py and walks.py successfully.
Entity text is now generated only when needed.


In [ ]:
import py_compile
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor"
)

for filename in ["textualize.py", "walks.py"]:
    py_compile.compile(str(folder / filename), doraise=True)
    print(filename, "— syntax OK")

print("Both updated files passed compilation.")

textualize.py — syntax OK
walks.py — syntax OK
Both updated files passed compilation.


In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/cli.py"
)

backup = Path(
    "/content/drive/MyDrive/GLAMOR_Project/code/memory_fix_backup/cli.py"
)

backup.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, backup)

print("cli.py original version backed up successfully.")

cli.py original version backed up successfully.


In [ ]:
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models"
)

terms = [
    "graph.entities",
    "graph.edges",
    "graph.adjacency",
    "AttributedKnowledgeGraph",
    "build_glamor_kg",
    "generate_walk_corpus"
]

for term in terms:
    print(f"\n--- {term} ---")
    for file in root.rglob("*.py"):
        try:
            for line_no, line in enumerate(
                file.read_text(encoding="utf-8").splitlines(), 1
            ):
                if term in line:
                    print(f"{file.relative_to(root)}:{line_no}: {line.strip()}")
        except UnicodeDecodeError:
            pass


--- graph.entities ---
cli.py:154: "entities": len(graph.entities),
cli.py:156: "entity_kinds": dict(Counter(entity.kind for entity in graph.entities.values())),
glamor/kg.py:307: for entity in (graph.entities[node_id] for node_id in sorted(graph.entities)):
glamor/recommend.py:14: ids = sorted(graph.entities)
glamor/recommend.py:15: labels = [graph.entities[node_id].name or node_id for node_id in ids]
glamor/recommend.py:68: and graph.entities.get(entity_id("paper", paper_id), None) is not None
glamor/recommend.py:69: and graph.entities[entity_id("paper", paper_id)].kind == "paper"
glamor/textualize.py:55: pieces = [textualize_entity(graph.entities[walk[0][0]])]
glamor/textualize.py:60: pieces.append(textualize_entity(graph.entities[node_id]))
glamor/walks.py:22: if not graph.entities:
glamor/walks.py:24: if start_node is not None and start_node not in graph.entities:
glamor/walks.py:27: current = start_node or rng.choice(sorted(graph.entities))

--- graph.edges ---
cli.py:155: "edge

In [ ]:
def load_papers(path, fmt="auto"):
    return list(iter_papers(path, fmt))

In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/common/validation.py"
)

backup = Path(
    "/content/drive/MyDrive/GLAMOR_Project/code/memory_fix_backup/validation.py"
)

shutil.copy2(source, backup)

print("Original validation.py backed up successfully.")

Original validation.py backed up successfully.


In [ ]:
from pathlib import Path

file = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/common/validation.py"
)

new_code = r'''
from __future__ import annotations

import sqlite3
import tempfile
from collections.abc import Iterable
from dataclasses import dataclass, field

from .records import PaperRecord


@dataclass
class ValidationReport:
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    statistics: dict[str, int | float] = field(default_factory=dict)

    @property
    def ok(self) -> bool:
        return not self.errors

    def raise_for_errors(self) -> None:
        if self.errors:
            raise ValueError("Dataset validation failed:\n- " + "\n- ".join(self.errors))


def validate_records(records: Iterable[PaperRecord]) -> ValidationReport:
    report = ValidationReport()

    with tempfile.NamedTemporaryFile(suffix=".sqlite", delete=False) as temp:
        db_path = temp.name

    try:
        connection = sqlite3.connect(db_path)
        connection.execute("PRAGMA journal_mode=OFF")
        connection.execute("PRAGMA synchronous=OFF")
        connection.execute("PRAGMA temp_store=FILE")

        connection.executescript("""
            CREATE TABLE papers (
                id TEXT PRIMARY KEY,
                empty_title INTEGER NOT NULL,
                cold_start INTEGER NOT NULL
            );
            CREATE TABLE authors (id TEXT PRIMARY KEY);
            CREATE TABLE venues (id TEXT PRIMARY KEY);
            CREATE TABLE fields (id TEXT PRIMARY KEY);
            CREATE TABLE concepts (id TEXT PRIMARY KEY);
            CREATE TABLE refs (
                source TEXT NOT NULL,
                target TEXT NOT NULL
            );
        """)

        paper_count = 0
        duplicate_count = 0
        duplicate_examples = []
        empty_titles = 0

        for record in records:
            paper_count += 1
            paper_id = str(record.id)
            empty = not record.title

            cursor = connection.execute(
                "INSERT OR IGNORE INTO papers VALUES (?, ?, ?)",
                (paper_id, int(empty), int(record.cold_start))
            )

            if cursor.rowcount == 0:
                duplicate_count += 1
                if paper_id not in duplicate_examples and len(duplicate_examples) < 20:
                    duplicate_examples.append(paper_id)

            if empty:
                empty_titles += 1

            for author in record.authors:
                connection.execute(
                    "INSERT OR IGNORE INTO authors VALUES (?)", (str(author.id),)
                )

            if record.venue:
                connection.execute(
                    "INSERT OR IGNORE INTO venues VALUES (?)", (str(record.venue.id),)
                )

            for item in record.fields:
                connection.execute(
                    "INSERT OR IGNORE INTO fields VALUES (?)", (str(item.id),)
                )

            for item in record.concepts:
                connection.execute(
                    "INSERT OR IGNORE INTO concepts VALUES (?)", (str(item.id),)
                )

            connection.executemany(
                "INSERT INTO refs VALUES (?, ?)",
                [(paper_id, str(ref)) for ref in record.references]
            )

            if paper_count % 10000 == 0:
                connection.commit()

        connection.commit()

        if duplicate_count:
            report.errors.append(
                f"Duplicate paper ids: {duplicate_examples}"
            )

        if empty_titles:
            report.errors.append(
                f"{empty_titles} paper records have empty titles"
            )

        dangling = connection.execute("""
            SELECT COUNT(*)
            FROM refs r
            LEFT JOIN papers p ON p.id = r.target
            WHERE p.id IS NULL
        """).fetchone()[0]

        self_citations = connection.execute("""
            SELECT COUNT(DISTINCT source)
            FROM refs
            WHERE source = target
        """).fetchone()[0]

        citation_edges = connection.execute("""
            SELECT COUNT(*)
            FROM refs r
            JOIN papers p ON p.id = r.target
            WHERE r.source != r.target
        """).fetchone()[0]

        if dangling:
            report.warnings.append(
                f"{dangling} references point outside the loaded corpus; "
                "they are ignored in evaluation"
            )

        if self_citations:
            report.warnings.append(
                f"{self_citations} self-referential paper edges will be removed"
            )

        report.statistics = {
            "papers": paper_count,
            "authors": connection.execute("SELECT COUNT(*) FROM authors").fetchone()[0],
            "venues": connection.execute("SELECT COUNT(*) FROM venues").fetchone()[0],
            "fields": connection.execute("SELECT COUNT(*) FROM fields").fetchone()[0],
            "concepts": connection.execute("SELECT COUNT(*) FROM concepts").fetchone()[0],
            "citation_edges_in_corpus": citation_edges,
            "cold_start_missing_authors": connection.execute(
                "SELECT COUNT(*) FROM papers WHERE cold_start = 1"
            ).fetchone()[0],
        }

        connection.close()
        return report

    finally:
        import os
        if os.path.exists(db_path):
            os.remove(db_path)


def validate_split(train_ids, test_ids, known_ids):
    train, test = set(train_ids), set(test_ids)
    overlap = train & test
    if overlap:
        raise ValueError(f"Train/test leakage: {len(overlap)} paper ids overlap")
    unknown = (train | test) - known_ids
    if unknown:
        raise ValueError(f"Split contains unknown ids: {sorted(unknown)[:20]}")
    missing = known_ids - (train | test)
    if missing:
        raise ValueError(f"Split omits known ids: {sorted(missing)[:20]}")
    if not train or not test:
        raise ValueError("Both train and test splits must be non-empty")
'''

file.write_text(new_code, encoding="utf-8")
print("Disk-based validation code installed.")

Disk-based validation code installed.


In [ ]:
from pathlib import Path
import shutil

root = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/common")
backup = Path("/content/drive/MyDrive/GLAMOR_Project/code/memory_fix_backup")

for name in ["io.py", "split.py"]:
    shutil.copy2(root / name, backup / name)
    print(name, "backed up successfully.")

print("Both original files are محفوظ safely.")

io.py backed up successfully.
split.py backed up successfully.
Both original files are محفوظ safely.


In [ ]:
import sys
import os
from pathlib import Path

REPO = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR")

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from citation_models.common.io import iter_papers

print("GLAMOR package imported successfully.")
print("Streaming reader is available.")

GLAMOR package imported successfully.
Streaming reader is available.


In [ ]:
import py_compile
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/common"
)

py_compile.compile(
    str(root / "validation.py"),
    doraise=True
)

from citation_models.common.validation import validate_records

print("Updated validation.py imported successfully.")
print("Disk-based validation function is ready.")

Updated validation.py imported successfully.
Disk-based validation function is ready.


In [ ]:
from itertools import islice
from citation_models.common.io import iter_papers
from citation_models.common.validation import validate_records

dataset = "/content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12_ready.txt"

sample = list(islice(iter_papers(dataset, "aminer"), 100))

report = validate_records(sample)

print("Records tested:", len(sample))
print("Validation passed:", report.ok)
print("Errors:", report.errors)
print("Warnings:", report.warnings)
print("Statistics:", report.statistics)

Records tested: 100
Validation passed: True
Errors: []
Warnings: ['620 references point outside the loaded corpus; they are ignored in evaluation']
Statistics: {'papers': 100, 'authors': 0, 'venues': 0, 'fields': 0, 'concepts': 0, 'citation_edges_in_corpus': 0, 'cold_start_missing_authors': 100}


In [ ]:
import sys
import random
from pathlib import Path

REPO = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR")
sys.path.insert(0, str(REPO / "src"))

from citation_models.common.io import iter_papers
from citation_models.common.config import load_config

config_path = REPO / "configs/glamor/dblp_v12.yaml"
config = load_config(config_path)

dataset = config["dataset"]["path"]
output = Path(config["output_dir"])
output.mkdir(parents=True, exist_ok=True)

# Read only paper IDs, not complete paper objects
ids = [record.id for record in iter_papers(dataset, "aminer")]

print("Paper IDs loaded:", len(ids))

# Preserve the original GLAMOR splitting algorithm
ids.sort()
random.Random(1203).shuffle(ids)

boundary = max(1, min(len(ids) - 1, int(len(ids) * 0.8)))

train_ids = sorted(ids[:boundary])
test_ids = sorted(ids[boundary:])

# Save separately without creating PaperRecord lists
with (output / "train_ids.txt").open("w") as f:
    f.writelines(f"{paper_id}\n" for paper_id in train_ids)

with (output / "test_ids.txt").open("w") as f:
    f.writelines(f"{paper_id}\n" for paper_id in test_ids)

print("Training papers:", len(train_ids))
print("Testing papers:", len(test_ids))
print("Split files saved:", output)

Paper IDs loaded: 4609265
Training papers: 3687412
Testing papers: 921853
Split files saved: artifacts/glamor_dblp_v12


In [ ]:
%%writefile /content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor/kg.py

Overwriting /content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor/kg.py


In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor/kg.py"
)

backup = Path(
    "/content/drive/MyDrive/GLAMOR_Project/code/memory_fix_backup/kg.py"
)

shutil.copy2(backup, source)

print("Original kg.py restored successfully.")

Original kg.py restored successfully.


In [ ]:
from pathlib import Path
import hashlib

original = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/src/citation_models/glamor/kg.py"
)

backup = Path(
    "/content/drive/MyDrive/GLAMOR_Project/code/memory_fix_backup/kg.py"
)

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("Original file exists:", original.exists())
print("Backup exists:", backup.exists())
print("Files identical:", file_hash(original) == file_hash(backup))
print("Original file size:", original.stat().st_size, "bytes")

Original file exists: True
Backup exists: True
Files identical: True
Original file size: 12789 bytes


In [ ]:
from pathlib import Path
import sys

REPO = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR")
sys.path.insert(0, str(REPO / "src"))

from citation_models.common.io import iter_papers

dataset = "/content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12_ready.txt"

for i, paper in enumerate(iter_papers(dataset, "aminer")):
    print("Paper ID:", paper.id)
    print("Title:", paper.title)
    print("Year:", paper.year)
    print("References:", len(paper.references))
    print("Authors:", len(paper.authors))
    print("Venue:", paper.venue)
    print("-" * 40)

    if i == 4:
        break

Paper ID: 10000000000
Title: Preliminary Design of a Network Protocol Learning Tool Based on the Comprehension of High School Students: Design by an Empirical Study Using a Simple Mind Map
Year: None
References: 2
Authors: 0
Venue: None
----------------------------------------
Paper ID: 10000000001
Title: Further Results on Independence in Direct-Product Graphs.
Year: None
References: 0
Authors: 0
Venue: None
----------------------------------------
Paper ID: 10000000002
Title: A methodology for the physically accurate visualisation of roman polychrome statuary
Year: None
References: 15
Authors: 0
Venue: None
----------------------------------------
Paper ID: 10000000003
Title: Comparison of GARCH, Neural Network and Support Vector Machine in Financial Time Series Prediction
Year: None
References: 3
Authors: 0
Venue: None
----------------------------------------
Paper ID: 10000000004
Title: COMPARING GNG3D AND QUADRIC ERROR METRICS METHODS TO SIMPLIFY 3D MESHES
Year: None
References: 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

print("MyDrive accessible:", os.path.exists("/content/drive/MyDrive"))

print("\nFolders in MyDrive:")
print(os.listdir("/content/drive/MyDrive")[:50])

Mounted at /content/drive
MyDrive accessible: True

Folders in MyDrive:
['Google Meet - Meet - nwf-nsun-wih 2024-10-20 20-25-41 (2).mp4', 'Google Meet - Meet - nwf-nsun-wih 2024-10-20 20-25-41 (1).mp4', 'Google Meet - Meet - nwf-nsun-wih 2024-10-20 20-25-41.mp4', 'Basic Electronics unit1.pdf', 'pc report.docx', 'pc report-DESKTOP-K5UTO53-DESKTOP-K5UTO53.docx', 'pc report 2025 (1).docx', 'pc report 2025.docx', '1-f1b500f4-3cef-4dc4-bfe2-a25d8470d3f4 (2).pdf', 'IMG_20250915_175003.jpg', '24STUCHH010408_infy.jpg.pdf', '24STUCHH010408_leetcode.jpg.pdf', 'Unit-3_Network basic & ESP 32.gdoc', 'AI certification course  (1).pdf', 'smart bus transport.pdf', 'Colab Notebooks', 'final report (408).pdf', 'dairy and observation (408) (1) (1).xlsx', 'dairy and observation (408) (1).xlsx', 'finasl ppt 408 (1) (1).pptx', 'finasl ppt 408 (1).pptx', 'final report(24STUCHH010408) (1).docx', 'final report(24STUCHH010408).docx', '24STUCHH010408 FINAL REPORT.pdf', '24STUCHH010408 (6).rar', '24STUCHH010408 (

In [ ]:
import os

root = "/content/drive/MyDrive"

matches = []

for current, folders, files in os.walk(root):
    folders[:] = [f for f in folders if f != ".Trash"]
    if "GLAMOR_Project" in folders:
        matches.append(os.path.join(current, "GLAMOR_Project"))

print("Project folder locations:")
for path in matches:
    print(path)

if not matches:
    print("GLAMOR_Project not found in MyDrive.")

Project folder locations:
/content/drive/MyDrive/GLAMOR_Project


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")

paths = {
    "Original dataset": PROJECT / "dataset/citation_network.csv",
    "Converted dataset": PROJECT / "dataset/dblp_v12_ready.txt",
    "GLAMOR repository": PROJECT / "GLAMOR",
    "Training IDs": PROJECT / "GLAMOR/configs/glamor/train_ids.txt",
    "SQLite database": PROJECT / "outputs/citation_knowledge_graph.sqlite"
}

for name, path in paths.items():
    print(f"\n{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")
    if path.exists() and path.is_file():
        print(f"Size: {path.stat().st_size / (1024**3):.3f} GB")


Original dataset: FOUND
Size: 1.050 GB

Converted dataset: FOUND
Size: 1.010 GB

GLAMOR repository: FOUND

Training IDs: NOT FOUND

SQLite database: NOT FOUND


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/GLAMOR_Project"

for root, dirs, files in os.walk(PROJECT):
    for filename in files:
        if filename in ["train_ids.txt", "test_ids.txt",
                        "citation_knowledge_graph.sqlite"]:
            path = os.path.join(root, filename)
            print(filename, "FOUND:", path)

train_ids.txt FOUND: /content/drive/MyDrive/GLAMOR_Project/GLAMOR/artifacts/glamor_dblp_v12/train_ids.txt
test_ids.txt FOUND: /content/drive/MyDrive/GLAMOR_Project/GLAMOR/artifacts/glamor_dblp_v12/test_ids.txt


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/GLAMOR_Project"

for root, dirs, files in os.walk(PROJECT):
    for filename in files:
        if filename in [
            "train_ids.txt",
            "test_ids.txt",
            "citation_knowledge_graph.sqlite"
        ]:
            path = os.path.join(root, filename)
            print(filename, ":", path)
            print("Size:", round(os.path.getsize(path) / (1024**2), 2), "MB")

train_ids.txt : /content/drive/MyDrive/GLAMOR_Project/GLAMOR/artifacts/glamor_dblp_v12/train_ids.txt
Size: 42.2 MB
test_ids.txt : /content/drive/MyDrive/GLAMOR_Project/GLAMOR/artifacts/glamor_dblp_v12/test_ids.txt
Size: 10.55 MB


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/GLAMOR_Project"

print("Database files found:")

found = False

for root, dirs, files in os.walk(PROJECT):
    for filename in files:
        if filename.endswith((".sqlite", ".sqlite3", ".db")):
            path = os.path.join(root, filename)
            print(path)
            print("Size:", round(os.path.getsize(path) / (1024**2), 2), "MB")
            found = True

if not found:
    print("No database files found.")

Database files found:
No database files found.


In [ ]:
from pathlib import Path

OUTPUTS = Path("/content/drive/MyDrive/GLAMOR_Project/outputs")

print("Outputs folder exists:", OUTPUTS.exists())

if OUTPUTS.exists():
    for item in OUTPUTS.iterdir():
        print(item.name, "-", round(item.stat().st_size / (1024**2), 2), "MB")

Outputs folder exists: True
validation.log - 0.0 MB


In [ ]:
import os
import sqlite3
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
OUTPUTS = PROJECT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

# Locate the existing training ID file
matches = list(PROJECT.rglob("train_ids.txt"))

if len(matches) != 1:
    raise RuntimeError(f"Expected one train_ids.txt file, found: {len(matches)}")

TRAIN_FILE = matches[0]
DB_PATH = OUTPUTS / "citation_knowledge_graph.sqlite"

# Never overwrite an existing database
if DB_PATH.exists():
    raise RuntimeError(f"Database already exists: {DB_PATH}")

print("Training IDs:", TRAIN_FILE)
print("Database destination:", DB_PATH)

connection = sqlite3.connect(str(DB_PATH))
connection.execute("PRAGMA journal_mode=DELETE")
connection.execute("PRAGMA synchronous=FULL")

connection.execute("""
    CREATE TABLE training_ids (
        paper_id TEXT PRIMARY KEY
    )
""")

connection.execute("""
    CREATE TABLE papers (
        paper_id TEXT PRIMARY KEY,
        title TEXT,
        year TEXT
    )
""")

connection.execute("""
    CREATE TABLE citations (
        source_id TEXT NOT NULL,
        target_id TEXT NOT NULL,
        PRIMARY KEY (source_id, target_id)
    )
""")

connection.execute("""
    CREATE TABLE progress (
        task TEXT PRIMARY KEY,
        last_paper_id TEXT,
        papers_processed INTEGER DEFAULT 0
    )
""")

connection.commit()

# Load the existing training IDs in batches
batch = []
total = 0

with open(TRAIN_FILE, "r", encoding="utf-8") as file:
    for line in file:
        paper_id = line.strip()
        if paper_id:
            batch.append((paper_id,))

        if len(batch) >= 10000:
            connection.executemany(
                "INSERT OR IGNORE INTO training_ids VALUES (?)",
                batch
            )
            connection.commit()
            total += len(batch)
            batch.clear()
            print("Training IDs loaded:", total)

if batch:
    connection.executemany(
        "INSERT OR IGNORE INTO training_ids VALUES (?)",
        batch
    )
    connection.commit()
    total += len(batch)

connection.close()

# Reopen the saved database and verify it
check = sqlite3.connect(str(DB_PATH))
saved_count = check.execute(
    "SELECT COUNT(*) FROM training_ids"
).fetchone()[0]
check.close()

print("\nDATABASE CREATED AND VERIFIED")
print("Training IDs saved:", saved_count)
print("Database exists:", DB_PATH.exists())
print("Database size:", round(DB_PATH.stat().st_size / (1024**2), 2), "MB")

Training IDs: /content/drive/MyDrive/GLAMOR_Project/GLAMOR/artifacts/glamor_dblp_v12/train_ids.txt
Database destination: /content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite
Training IDs loaded: 10000
Training IDs loaded: 20000
Training IDs loaded: 30000
Training IDs loaded: 40000
Training IDs loaded: 50000
Training IDs loaded: 60000
Training IDs loaded: 70000
Training IDs loaded: 80000
Training IDs loaded: 90000
Training IDs loaded: 100000
Training IDs loaded: 110000
Training IDs loaded: 120000
Training IDs loaded: 130000
Training IDs loaded: 140000
Training IDs loaded: 150000
Training IDs loaded: 160000
Training IDs loaded: 170000
Training IDs loaded: 180000
Training IDs loaded: 190000
Training IDs loaded: 200000
Training IDs loaded: 210000
Training IDs loaded: 220000
Training IDs loaded: 230000
Training IDs loaded: 240000
Training IDs loaded: 250000
Training IDs loaded: 260000
Training IDs loaded: 270000
Training IDs loaded: 280000
Training IDs loaded: 29000

In [ ]:
import os
import sys
import sqlite3
from pathlib import Path

# Project paths
PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
REPO = PROJECT / "GLAMOR"
DATASET = PROJECT / "dataset/dblp_v12_ready.txt"
DB_PATH = PROJECT / "outputs/citation_knowledge_graph.sqlite"

# Connect to GLAMOR source code
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from citation_models.common.io import iter_papers

# Verify required files
assert DATASET.exists(), "Converted dataset not found"
assert DB_PATH.exists(), "SQLite database not found"

connection = sqlite3.connect(str(DB_PATH))
connection.execute("PRAGMA journal_mode=DELETE")
connection.execute("PRAGMA synchronous=FULL")

# Add a record-position checkpoint column if needed
columns = [
    row[1] for row in connection.execute(
        "PRAGMA table_info(progress)"
    )
]

if "scanned_records" not in columns:
    connection.execute(
        "ALTER TABLE progress ADD COLUMN scanned_records INTEGER DEFAULT 0"
    )
    connection.commit()

# Recover previous progress
saved = connection.execute(
    "SELECT scanned_records, papers_processed "
    "FROM progress WHERE task = ?",
    ("citation_graph",)
).fetchone()

if saved:
    last_scanned, paper_count = saved
else:
    last_scanned, paper_count = 0, 0

print("Previously checkpointed records:", last_scanned)
print("Previously processed training papers:", paper_count)

# Check for unexpected existing graph data
if paper_count == 0:
    existing = connection.execute(
        "SELECT COUNT(*) FROM papers"
    ).fetchone()[0]

    if existing > 0:
        raise RuntimeError(
            "Graph data exists but progress is missing. Stop and inspect."
        )

def save_checkpoint(record_number, processed_count, paper_id):
    connection.execute("""
        INSERT INTO progress
        (task, last_paper_id, papers_processed, scanned_records)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(task) DO UPDATE SET
            last_paper_id = excluded.last_paper_id,
            papers_processed = excluded.papers_processed,
            scanned_records = excluded.scanned_records
    """, (
        "citation_graph",
        str(paper_id),
        processed_count,
        record_number
    ))
    connection.commit()

try:
    for record_number, paper in enumerate(
        iter_papers(str(DATASET), "aminer"), start=1
    ):
        # Skip records already covered by the last checkpoint
        if record_number <= last_scanned:
            continue

        # Check whether this paper belongs to the training set
        is_training = connection.execute(
            "SELECT 1 FROM training_ids WHERE paper_id = ?",
            (str(paper.id),)
        ).fetchone()

        if not is_training:
            continue

        # Save paper
        connection.execute(
            "INSERT OR IGNORE INTO papers VALUES (?, ?, ?)",
            (str(paper.id), paper.title, paper.year)
        )

        # Save citations only when the target is also in training
        references = paper.references or []

        connection.executemany("""
            INSERT OR IGNORE INTO citations (source_id, target_id)
            SELECT ?, ?
            WHERE EXISTS (
                SELECT 1 FROM training_ids WHERE paper_id = ?
            )
        """, (
            (str(paper.id), str(ref), str(ref))
            for ref in references
            if str(ref) != str(paper.id)
        ))

        paper_count += 1

        # Save a durable checkpoint every 10,000 training papers
        if paper_count % 10000 == 0:
            save_checkpoint(record_number, paper_count, paper.id)

            print(
                "Training papers processed:", paper_count,
                "| Dataset records scanned:", record_number
            )

    # Save final progress
    save_checkpoint(record_number, paper_count, paper.id)

    print("\nCITATION GRAPH BUILD COMPLETED")

except KeyboardInterrupt:
    print("\nProcessing interrupted. Last committed checkpoint is safe.")

finally:
    connection.close()

# Verify the database from disk
check = sqlite3.connect(str(DB_PATH))

print("\nPapers stored:", check.execute(
    "SELECT COUNT(*) FROM papers"
).fetchone()[0])

print("Citation links stored:", check.execute(
    "SELECT COUNT(*) FROM citations"
).fetchone()[0])

print("Saved progress:", check.execute(
    "SELECT scanned_records, papers_processed FROM progress "
    "WHERE task = 'citation_graph'"
).fetchone())

check.close()
print("Database location:", DB_PATH)

Previously checkpointed records: 0
Previously processed training papers: 0
Training papers processed: 10000 | Dataset records scanned: 12500
Training papers processed: 20000 | Dataset records scanned: 25040
Training papers processed: 30000 | Dataset records scanned: 37537
Training papers processed: 40000 | Dataset records scanned: 50050
Training papers processed: 50000 | Dataset records scanned: 62568
Training papers processed: 60000 | Dataset records scanned: 75120
Training papers processed: 70000 | Dataset records scanned: 87702
Training papers processed: 80000 | Dataset records scanned: 100140
Training papers processed: 90000 | Dataset records scanned: 112632
Training papers processed: 100000 | Dataset records scanned: 125113
Training papers processed: 110000 | Dataset records scanned: 137717
Training papers processed: 120000 | Dataset records scanned: 150136
Training papers processed: 130000 | Dataset records scanned: 162567
Training papers processed: 140000 | Dataset records scann

In [ ]:
#checking database integrity
import sqlite3
from pathlib import Path

DB = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"
)

conn = sqlite3.connect(str(DB))

print("Database integrity:",
      conn.execute("PRAGMA integrity_check").fetchone()[0])

print("Papers:",
      conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0])

print("Citation links:",
      conn.execute("SELECT COUNT(*) FROM citations").fetchone()[0])

print("Training IDs:",
      conn.execute("SELECT COUNT(*) FROM training_ids").fetchone()[0])

print("Database size:",
      round(DB.stat().st_size / (1024**3), 2), "GB")

conn.close()

Database integrity: ok
Papers: 3687412
Citation links: 26207545
Training IDs: 3687412
Database size: 2.26 GB


In [ ]:
#prepare recommendation model
import sqlite3

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(DB)

print("PAPER COUNT:",
      conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0])

print("CITATION COUNT:",
      conn.execute("SELECT COUNT(*) FROM citations").fetchone()[0])

print("\nPAPERS WITH OUTGOING CITATIONS:",
      conn.execute(
          "SELECT COUNT(DISTINCT source_id) FROM citations"
      ).fetchone()[0])

print("PAPERS CITED BY OTHER PAPERS:",
      conn.execute(
          "SELECT COUNT(DISTINCT target_id) FROM citations"
      ).fetchone()[0])

print("\nTOP 10 MOST CITED PAPERS:")

for row in conn.execute("""
    SELECT target_id, COUNT(*) AS total
    FROM citations
    GROUP BY target_id
    ORDER BY total DESC
    LIMIT 10
"""):
    print(row)

conn.close()

PAPER COUNT: 3687412
CITATION COUNT: 26207545

PAPERS WITH OUTGOING CITATIONS: 2809282
PAPERS CITED BY OTHER PAPERS: 2204815

TOP 10 MOST CITED PAPERS:
('10002937610', 17666)
('10002691785', 15084)
('10004092588', 12461)
('10002729388', 10921)
('10002785159', 10509)
('10000745098', 10502)
('10002089551', 8509)
('10002468282', 8226)
('10002383571', 7931)
('10000688041', 7635)


In [ ]:
import sqlite3

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(DB)

query = """
SELECT
    p.paper_id,
    p.title,
    COUNT(c.target_id) AS citation_count
FROM citations c
JOIN papers p ON p.paper_id = c.target_id
GROUP BY p.paper_id, p.title
ORDER BY citation_count DESC
LIMIT 10
"""

results = conn.execute(query).fetchall()

print("TOP 10 MOST CITED PAPERS\n")

for i, (paper_id, title, count) in enumerate(results, 1):
    print(f"{i}. Paper ID: {paper_id}")
    print(f"   Title: {title}")
    print(f"   Citations: {count}\n")

conn.close()

TOP 10 MOST CITED PAPERS

1. Paper ID: 10002937610
   Title: Distinctive Image Features from Scale-Invariant Keypoints
   Citations: 17666

2. Paper ID: 10002691785
   Title: LIBSVM: A library for support vector machines
   Citations: 15084

3. Paper ID: 10004092588
   Title: Fuzzy sets
   Citations: 12461

4. Paper ID: 10002729388
   Title: The Nature of Statistical Learning Theory
   Citations: 10921

5. Paper ID: 10002785159
   Title: Histograms of oriented gradients for human detection
   Citations: 10509

6. Paper ID: 10000745098
   Title: Latent dirichlet allocation
   Citations: 10502

7. Paper ID: 10002089551
   Title: Long short-term memory
   Citations: 8509

8. Paper ID: 10002468282
   Title: Image quality assessment: from error visibility to structural similarity
   Citations: 8226

9. Paper ID: 10002383571
   Title: A fast and elitist multiobjective genetic algorithm: NSGA-II
   Citations: 7931

10. Paper ID: 10000688041
   Title: Perceived usefulness, perceived ease of us

In [ ]:
#the searchable title index
import sqlite3
from pathlib import Path

DB = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"
)

conn = sqlite3.connect(str(DB))
conn.execute("PRAGMA synchronous=FULL")

# Create the full-text search index
conn.execute("""
    CREATE VIRTUAL TABLE IF NOT EXISTS papers_fts
    USING fts5(paper_id UNINDEXED, title, tokenize='porter unicode61')
""")
conn.commit()

# Recover the last indexed paper
last_rowid = conn.execute(
    "SELECT COALESCE(MAX(rowid), 0) FROM papers_fts"
).fetchone()[0]

indexed = conn.execute(
    "SELECT COUNT(*) FROM papers_fts"
).fetchone()[0]

print("Already indexed:", indexed)
print("Resuming after database row:", last_rowid)

while True:
    rows = conn.execute("""
        SELECT rowid, paper_id, title
        FROM papers
        WHERE rowid > ?
          AND title IS NOT NULL
          AND TRIM(title) != ''
        ORDER BY rowid
        LIMIT 50000
    """, (last_rowid,)).fetchall()

    if not rows:
        break

    conn.executemany("""
        INSERT OR REPLACE INTO papers_fts(rowid, paper_id, title)
        VALUES (?, ?, ?)
    """, rows)

    last_rowid = rows[-1][0]
    indexed += len(rows)

    conn.commit()

    print("Titles indexed:", indexed,
          "| Last database row:", last_rowid)

print("\nTITLE INDEX BUILD COMPLETED")

print("Total indexed titles:", conn.execute(
    "SELECT COUNT(*) FROM papers_fts"
).fetchone()[0])

print("Database integrity:", conn.execute(
    "PRAGMA integrity_check"
).fetchone()[0])

conn.close()

print("Database saved:", DB.exists())
print("Database size:", round(DB.stat().st_size / (1024**3), 2), "GB")

Already indexed: 0
Resuming after database row: 0
Titles indexed: 50000 | Last database row: 50000
Titles indexed: 100000 | Last database row: 100000
Titles indexed: 150000 | Last database row: 150000
Titles indexed: 200000 | Last database row: 200000
Titles indexed: 250000 | Last database row: 250000
Titles indexed: 300000 | Last database row: 300000
Titles indexed: 350000 | Last database row: 350000
Titles indexed: 400000 | Last database row: 400000
Titles indexed: 450000 | Last database row: 450000
Titles indexed: 500000 | Last database row: 500000
Titles indexed: 550000 | Last database row: 550000
Titles indexed: 600000 | Last database row: 600000
Titles indexed: 650000 | Last database row: 650000
Titles indexed: 700000 | Last database row: 700000
Titles indexed: 750000 | Last database row: 750000
Titles indexed: 800000 | Last database row: 800000
Titles indexed: 850000 | Last database row: 850000
Titles indexed: 900000 | Last database row: 900000
Titles indexed: 950000 | Last data

In [ ]:
import sqlite3

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(DB)

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_citations_target
ON citations(target_id)
""")

conn.commit()

print("Citation index ready.")

conn.close()

Citation index ready.


In [ ]:
import sqlite3

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

def recommend_papers(paper_id, limit=10):
    conn = sqlite3.connect(DB)

    query = """
    WITH related AS (
        SELECT target_id AS paper_id
        FROM citations
        WHERE source_id = ?

        UNION

        SELECT source_id AS paper_id
        FROM citations
        WHERE target_id = ?
    )
    SELECT p.paper_id, p.title
    FROM related r
    JOIN papers p ON p.paper_id = r.paper_id
    WHERE p.paper_id != ?
    LIMIT ?
    """

    results = conn.execute(
        query, (paper_id, paper_id, paper_id, limit)
    ).fetchall()

    conn.close()
    return results


# Test with the paper "Fuzzy sets"
paper_id = "10004092588"

recommendations = recommend_papers(paper_id)

print("RECOMMENDED RESEARCH PAPERS:\n")

for i, (pid, title) in enumerate(recommendations, 1):
    print(f"{i}. {title}")
    print("   Paper ID:", pid)

RECOMMENDED RESEARCH PAPERS:

1. From likelihood uncertainty to fuzziness: a possibility-based approach for building clinical DSSs
   Paper ID: 10000001205
2. Fuzzy Arithmetic for Uncertainty Analysis
   Paper ID: 10000001677
3. An Adequate Representation of Medical Data Based on Partial Set Approximation
   Paper ID: 10000001724
4. Some New Entropies on the Interval-Valued Fuzzy Set
   Paper ID: 10000002510
5. Discernibility Matrix Based Attribute Reduction in Intuitionistic Fuzzy Decision Systems
   Paper ID: 10000003438
6. Applying a Fuzzy Decision Tree Approach to Soil Classification
   Paper ID: 10000003441
7. A New Pseudo-metric for Fuzzy Sets
   Paper ID: 10000003801
8. Lagrange problem in fuzzy reversed posynomial geometric programming
   Paper ID: 10000003902
9. The Membership Function and Its Measurement
   Paper ID: 10000005260
10. NEURO-FUZZY CONTROL OF NONLINEAR SYSTEMS - Application in a Ball and Beam System
   Paper ID: 10000005355


In [ ]:
import sqlite3

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

def recommend_similar_papers(paper_id, limit=10, candidate_limit=500):

    conn = sqlite3.connect(DB)

    # Get references of the selected paper
    seed_refs = {
        row[0] for row in conn.execute(
            "SELECT target_id FROM citations WHERE source_id = ?",
            (paper_id,)
        )
    }

    if not seed_refs:
        conn.close()
        print("No references available for this paper.")
        return []

    # Find papers connected to the selected paper
    candidates = set()

    candidates.update(
        row[0] for row in conn.execute("""
            SELECT target_id FROM citations
            WHERE source_id = ?
            LIMIT ?
        """, (paper_id, candidate_limit))
    )

    candidates.update(
        row[0] for row in conn.execute("""
            SELECT source_id FROM citations
            WHERE target_id = ?
            LIMIT ?
        """, (paper_id, candidate_limit))
    )

    candidates.discard(paper_id)

    if not candidates:
        conn.close()
        print("No connected candidate papers found.")
        return []

    # Fetch candidate references in batches
    candidate_refs = {}

    candidate_list = list(candidates)

    for start in range(0, len(candidate_list), 400):
        batch = candidate_list[start:start + 400]
        placeholders = ",".join("?" for _ in batch)

        rows = conn.execute(f"""
            SELECT source_id, target_id
            FROM citations
            WHERE source_id IN ({placeholders})
        """, batch).fetchall()

        for source, target in rows:
            candidate_refs.setdefault(source, set()).add(target)

    # Calculate Jaccard similarity
    scores = []

    for candidate_id, refs in candidate_refs.items():
        common = len(seed_refs.intersection(refs))
        union = len(seed_refs.union(refs))

        if common > 0 and union > 0:
            similarity = common / union
            scores.append((candidate_id, similarity, common))

    # Sort by similarity
    scores.sort(key=lambda x: (x[1], x[2]), reverse=True)

    # Retrieve titles
    recommendations = []

    for candidate_id, similarity, common in scores[:limit]:
        row = conn.execute(
            "SELECT title FROM papers WHERE paper_id = ?",
            (candidate_id,)
        ).fetchone()

        if row and row[0]:
            recommendations.append(
                (candidate_id, row[0], similarity, common)
            )

    conn.close()
    return recommendations


# Test recommendation system
selected_paper = "10004092588"

results = recommend_similar_papers(selected_paper)

print("TOP 10 SIMILAR RESEARCH PAPERS\n")

for i, (pid, title, score, common) in enumerate(results, 1):
    print(f"{i}. {title}")
    print("   Paper ID:", pid)
    print("   Similarity:", round(score, 4))
    print("   Shared references:", common)
    print()

No references available for this paper.
TOP 10 SIMILAR RESEARCH PAPERS



In [ ]:
import sqlite3

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

def recommend_papers(paper_id, limit=10):
    conn = sqlite3.connect(DB)

    # Check outgoing references
    outgoing = conn.execute(
        "SELECT COUNT(*) FROM citations WHERE source_id = ?",
        (paper_id,)
    ).fetchone()[0]

    if outgoing > 0:
        query = """
        SELECT DISTINCT p.paper_id, p.title
        FROM citations c
        JOIN papers p ON p.paper_id = c.target_id
        WHERE c.source_id = ?
        LIMIT ?
        """
        results = conn.execute(
            query, (paper_id, limit)
        ).fetchall()
        method = "Papers cited by the selected paper"

    else:
        # Fallback: find papers citing the selected paper
        query = """
        SELECT DISTINCT p.paper_id, p.title
        FROM citations c
        JOIN papers p ON p.paper_id = c.source_id
        WHERE c.target_id = ?
        LIMIT ?
        """
        results = conn.execute(
            query, (paper_id, limit)
        ).fetchall()
        method = "Papers citing the selected paper"

    conn.close()

    print("Recommendation method:", method)
    return results


results = recommend_papers("10004092588", 10)

print("\nRECOMMENDED RESEARCH PAPERS:\n")

for i, (pid, title) in enumerate(results, 1):
    print(f"{i}. {title}")
    print("   Paper ID:", pid)

Recommendation method: Papers citing the selected paper

RECOMMENDED RESEARCH PAPERS:

1. From likelihood uncertainty to fuzziness: a possibility-based approach for building clinical DSSs
   Paper ID: 10000001205
2. Fuzzy Arithmetic for Uncertainty Analysis
   Paper ID: 10000001677
3. An Adequate Representation of Medical Data Based on Partial Set Approximation
   Paper ID: 10000001724
4. Some New Entropies on the Interval-Valued Fuzzy Set
   Paper ID: 10000002510
5. Discernibility Matrix Based Attribute Reduction in Intuitionistic Fuzzy Decision Systems
   Paper ID: 10000003438
6. Applying a Fuzzy Decision Tree Approach to Soil Classification
   Paper ID: 10000003441
7. A New Pseudo-metric for Fuzzy Sets
   Paper ID: 10000003801
8. Lagrange problem in fuzzy reversed posynomial geometric programming
   Paper ID: 10000003902
9. The Membership Function and Its Measurement
   Paper ID: 10000005260
10. NEURO-FUZZY CONTROL OF NONLINEAR SYSTEMS - Application in a Ball and Beam System
   Pape

In [ ]:
#Rank recommendations by title similarity
import sqlite3
import re

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

def recommend_by_title_similarity(paper_id, limit=10):

    conn = sqlite3.connect(DB)

    # Get selected paper title
    row = conn.execute(
        "SELECT title FROM papers WHERE paper_id = ?",
        (paper_id,)
    ).fetchone()

    if not row or not row[0]:
        conn.close()
        print("Paper title not found.")
        return []

    title = row[0]

    # Extract searchable words
    words = list(dict.fromkeys(
        re.findall(r"[a-zA-Z0-9]+", title.lower())
    ))

    if not words:
        conn.close()
        print("No searchable title words.")
        return []

    search_query = " OR ".join(
        f'title : "{word}"' for word in words
    )

    # Find papers citing the selected paper,
    # then rank them using title relevance.
    results = conn.execute("""
        SELECT
            f.paper_id,
            f.title,
            bm25(papers_fts) AS relevance
        FROM papers_fts AS f
        WHERE papers_fts MATCH ?
          AND f.paper_id IN (
              SELECT source_id
              FROM citations
              WHERE target_id = ?
          )
        ORDER BY relevance ASC
        LIMIT ?
    """, (search_query, paper_id, limit)).fetchall()

    conn.close()

    print("Selected paper:", title)
    print("Search terms:", ", ".join(words))

    return results


# Test
results = recommend_by_title_similarity("10004092588", 10)

print("\nTOP 10 RANKED RECOMMENDATIONS:\n")

for i, (pid, title, score) in enumerate(results, 1):
    print(f"{i}. {title}")
    print("   Paper ID:", pid)
    print("   Relevance score:", round(score, 5))
    print()

Selected paper: Fuzzy sets
Search terms: fuzzy, sets

TOP 10 RANKED RECOMMENDATIONS:

1. Intuitionistic fuzzy sets and L -fuzzy sets
   Paper ID: 10001112114
   Relevance score: -12.97081

2. Fuzzy Probabilistic Sets and r-Fuzzy Sets
   Paper ID: 10002479158
   Relevance score: -12.97081

3. Picture fuzzy sets and 3-fuzzy sets
   Paper ID: 10004003050
   Relevance score: -12.97081

4. Fuzzy rough sets are intuitionistic L -fuzzy sets
   Paper ID: 10000990169
   Relevance score: -12.59115

5. Type-1 Fuzzy Sets and Intuitionistic Fuzzy Sets
   Paper ID: 10003677683
   Relevance score: -12.59115

6. Soft rough fuzzy sets and soft fuzzy rough sets
   Paper ID: 10001990253
   Relevance score: -12.23308

7. A new extension of fuzzy sets using rough sets: R-fuzzy sets
   Paper ID: 10001684707
   Relevance score: -12.16962

8. Direction fuzzy sets
   Paper ID: 10000730725
   Relevance score: -12.09117

9. Balanced fuzzy sets
   Paper ID: 10000831982
   Relevance score: -12.09117

10. Concavoco

In [ ]:
#hybrid recommmendation system
import sqlite3
import re

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

def hybrid_recommendations(paper_id, limit=10):

    conn = sqlite3.connect(DB)

    row = conn.execute(
        "SELECT title FROM papers WHERE paper_id=?",
        (paper_id,)
    ).fetchone()

    if not row:
        conn.close()
        print("Paper not found.")
        return []

    title = row[0]

    words = list(dict.fromkeys(
        re.findall(r"[a-zA-Z0-9]+", title.lower())
    ))

    stopwords = {
        "a", "an", "the", "of", "in", "on", "and",
        "for", "to", "from", "with", "using"
    }

    words = [w for w in words if w not in stopwords]

    if not words:
        conn.close()
        print("No searchable title words.")
        return []

    search_query = " OR ".join(
        f'title : "{word}"' for word in words
    )

    # Find papers citing the selected paper and matching its title
    title_results = conn.execute("""
        SELECT f.paper_id, f.title, bm25(papers_fts) AS score
        FROM papers_fts f
        WHERE papers_fts MATCH ?
          AND f.paper_id IN (
              SELECT source_id
              FROM citations
              WHERE target_id = ?
          )
        ORDER BY score ASC
        LIMIT 200
    """, (search_query, paper_id)).fetchall()

    if not title_results:
        conn.close()
        print("No matching recommendations found.")
        return []

    # Get citation counts in manageable batches
    popularity = {}

    ids = [r[0] for r in title_results]

    for start in range(0, len(ids), 400):
        batch = ids[start:start + 400]
        placeholders = ",".join("?" for _ in batch)

        rows = conn.execute(f"""
            SELECT target_id, COUNT(*)
            FROM citations
            WHERE target_id IN ({placeholders})
            GROUP BY target_id
        """, batch).fetchall()

        popularity.update(dict(rows))

    conn.close()

    # Rank by title relevance
    title_rank = {
        row[0]: rank
        for rank, row in enumerate(title_results, 1)
    }

    # Rank by citation count
    citation_order = sorted(
        ids,
        key=lambda pid: popularity.get(pid, 0),
        reverse=True
    )

    citation_rank = {
        pid: rank
        for rank, pid in enumerate(citation_order, 1)
    }

    # Reciprocal Rank Fusion
    combined = []

    for pid, paper_title, score in title_results:
        rrf = (
            1 / (60 + title_rank[pid])
            + 1 / (60 + citation_rank[pid])
        )

        combined.append((
            pid,
            paper_title,
            score,
            popularity.get(pid, 0),
            rrf
        ))

    combined.sort(key=lambda x: x[4], reverse=True)

    return combined[:limit]


# Test hybrid recommendation
results = hybrid_recommendations("10004092588", 10)

print("HYBRID RECOMMENDATIONS\n")

for i, (pid, title, bm25, citations, score) in enumerate(results, 1):
    print(f"{i}. {title}")
    print("   Paper ID:", pid)
    print("   Citation count:", citations)
    print("   Combined RRF score:", round(score, 6))
    print()

HYBRID RECOMMENDATIONS

1. Vague sets are intuitionistic fuzzy sets
   Paper ID: 10001410002
   Citation count: 344
   Combined RRF score: 0.029907

2. Intuitionistic fuzzy sets and L -fuzzy sets
   Paper ID: 10001112114
   Citation count: 49
   Combined RRF score: 0.028589

3. Complex fuzzy sets
   Paper ID: 10002057228
   Citation count: 84
   Combined RRF score: 0.027242

4. Soft rough fuzzy sets and soft fuzzy rough sets
   Paper ID: 10001990253
   Citation count: 37
   Combined RRF score: 0.026779

5. Fuzzy rough sets are intuitionistic L -fuzzy sets
   Paper ID: 10000990169
   Citation count: 25
   Combined RRF score: 0.026042

6. Dual Hesitant Fuzzy Sets
   Paper ID: 10001783877
   Citation count: 136
   Combined RRF score: 0.025937

7. A new extension of fuzzy sets using rough sets: R-fuzzy sets
   Paper ID: 10001684707
   Citation count: 30
   Combined RRF score: 0.025795

8. Intuitionistic fuzzy set vs. fuzzy set application in medical pattern recognition
   Paper ID: 1000201

In [ ]:
import sqlite3
from pathlib import Path

DB = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"
)

conn = sqlite3.connect(str(DB))

tables = {
    row[0] for row in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    )
}

print("DATABASE:", DB.exists())

for table in [
    "test_ids",
    "test_queries",
    "test_relevance",
    "evaluation_progress"
]:
    if table in tables:
        count = conn.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]
        print(f"{table}: {count:,} records")
    else:
        print(f"{table}: NOT CREATED")

if "evaluation_progress" in tables:
    print("\nSaved checkpoint:")
    print(conn.execute(
        "SELECT * FROM evaluation_progress"
    ).fetchall())

conn.close()

DATABASE: True
test_ids: NOT CREATED
test_queries: NOT CREATED
test_relevance: NOT CREATED
evaluation_progress: NOT CREATED


In [ ]:
import sqlite3
import sys
import os
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
REPO = PROJECT / "GLAMOR"
DATASET = PROJECT / "dataset/dblp_v12_ready.txt"
DB = PROJECT / "outputs/citation_knowledge_graph.sqlite"

# Locate existing test IDs
test_files = list(PROJECT.rglob("test_ids.txt"))

if len(test_files) != 1:
    raise RuntimeError(f"Expected one test_ids.txt, found {len(test_files)}")

TEST_FILE = test_files[0]

# Import existing dataset reader
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from citation_models.common.io import iter_papers

conn = sqlite3.connect(str(DB))
conn.execute("PRAGMA synchronous=FULL")

conn.executescript("""
CREATE TABLE IF NOT EXISTS test_ids (
    paper_id TEXT PRIMARY KEY
);

CREATE TABLE IF NOT EXISTS test_queries (
    paper_id TEXT PRIMARY KEY,
    title TEXT
);

CREATE TABLE IF NOT EXISTS test_relevance (
    test_paper_id TEXT NOT NULL,
    relevant_paper_id TEXT NOT NULL,
    PRIMARY KEY (test_paper_id, relevant_paper_id)
);

CREATE TABLE IF NOT EXISTS evaluation_progress (
    task TEXT PRIMARY KEY,
    scanned_records INTEGER DEFAULT 0,
    test_papers_processed INTEGER DEFAULT 0
);
""")
conn.commit()

# Load existing test IDs
batch = []

with open(TEST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        pid = line.strip()
        if pid:
            batch.append((pid,))

        if len(batch) >= 10000:
            conn.executemany(
                "INSERT OR IGNORE INTO test_ids VALUES (?)", batch
            )
            conn.commit()
            batch.clear()

if batch:
    conn.executemany(
        "INSERT OR IGNORE INTO test_ids VALUES (?)", batch
    )
    conn.commit()

print("Test IDs loaded:",
      conn.execute("SELECT COUNT(*) FROM test_ids").fetchone()[0])

# Resume from last committed checkpoint
saved = conn.execute("""
    SELECT scanned_records, test_papers_processed
    FROM evaluation_progress
    WHERE task = 'heldout_citations'
""").fetchone()

last_scanned, processed = saved if saved else (0, 0)

print("Previous dataset checkpoint:", last_scanned)
print("Previously processed test papers:", processed)

try:
    for record_number, paper in enumerate(
        iter_papers(str(DATASET), "aminer"), start=1
    ):
        if record_number <= last_scanned:
            continue

        is_test = conn.execute(
            "SELECT 1 FROM test_ids WHERE paper_id = ?",
            (str(paper.id),)
        ).fetchone()

        if not is_test:
            continue

        conn.execute("""
            INSERT OR IGNORE INTO test_queries (paper_id, title)
            VALUES (?, ?)
        """, (str(paper.id), paper.title))

        refs = paper.references or []

        conn.executemany("""
            INSERT OR IGNORE INTO test_relevance
            (test_paper_id, relevant_paper_id)
            SELECT ?, ?
            WHERE EXISTS (
                SELECT 1 FROM training_ids WHERE paper_id = ?
            )
        """, (
            (str(paper.id), str(ref), str(ref))
            for ref in refs
            if str(ref) != str(paper.id)
        ))

        processed += 1

        if processed % 5000 == 0:
            conn.execute("""
                INSERT INTO evaluation_progress
                VALUES ('heldout_citations', ?, ?)
                ON CONFLICT(task) DO UPDATE SET
                    scanned_records = excluded.scanned_records,
                    test_papers_processed = excluded.test_papers_processed
            """, (record_number, processed))

            conn.commit()

            print(
                "Test papers processed:", processed,
                "| Dataset records scanned:", record_number
            )

    # Final checkpoint
    conn.execute("""
        INSERT INTO evaluation_progress
        VALUES ('heldout_citations', ?, ?)
        ON CONFLICT(task) DO UPDATE SET
            scanned_records = excluded.scanned_records,
            test_papers_processed = excluded.test_papers_processed
    """, (record_number, processed))

    conn.commit()
    print("\nTEST-DATA EXTRACTION COMPLETED")

except KeyboardInterrupt:
    print("\nInterrupted. Last committed checkpoint is saved.")

finally:
    conn.close()

# Verify saved results
check = sqlite3.connect(str(DB))

print("\nTest queries:",
      check.execute("SELECT COUNT(*) FROM test_queries").fetchone()[0])

print("Relevant citation links:",
      check.execute("SELECT COUNT(*) FROM test_relevance").fetchone()[0])

print("Database integrity:",
      check.execute("PRAGMA integrity_check").fetchone()[0])

check.close()

Test IDs loaded: 921853
Previous dataset checkpoint: 0
Previously processed test papers: 0
Test papers processed: 5000 | Dataset records scanned: 24800
Test papers processed: 10000 | Dataset records scanned: 49813
Test papers processed: 15000 | Dataset records scanned: 74445
Test papers processed: 20000 | Dataset records scanned: 99411
Test papers processed: 25000 | Dataset records scanned: 124539
Test papers processed: 30000 | Dataset records scanned: 149313
Test papers processed: 35000 | Dataset records scanned: 174901
Test papers processed: 40000 | Dataset records scanned: 199933
Test papers processed: 45000 | Dataset records scanned: 224924
Test papers processed: 50000 | Dataset records scanned: 249783
Test papers processed: 55000 | Dataset records scanned: 274945
Test papers processed: 60000 | Dataset records scanned: 299769
Test papers processed: 65000 | Dataset records scanned: 324825
Test papers processed: 70000 | Dataset records scanned: 349851
Test papers processed: 75000 | D

In [ ]:
import sqlite3

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(DB)

print("TEST PAPERS:",
      conn.execute("SELECT COUNT(*) FROM test_queries").fetchone()[0])

print("RELEVANT LINKS:",
      conn.execute("SELECT COUNT(*) FROM test_relevance").fetchone()[0])

print("TEST PAPERS WITH RELEVANT LINKS:",
      conn.execute("""
          SELECT COUNT(DISTINCT test_paper_id)
          FROM test_relevance
      """).fetchone()[0])

print("DATABASE INTEGRITY:",
      conn.execute("PRAGMA integrity_check").fetchone()[0])

conn.close()

TEST PAPERS: 921853
RELEVANT LINKS: 6557606
TEST PAPERS WITH RELEVANT LINKS: 700900
DATABASE INTEGRITY: ok


In [ ]:
from pathlib import Path

REPO = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR")

files = [
    "src/citation_models/cli.py",
    "src/citation_models/glamor/kg.py",
    "src/citation_models/glamor/walks.py",
    "src/citation_models/glamor/recommend.py",
    "src/citation_models/glamor/language_model.py",
]

for filename in files:
    path = REPO / filename
    print(filename, ":", "FOUND" if path.exists() else "NOT FOUND")

src/citation_models/cli.py : FOUND
src/citation_models/glamor/kg.py : FOUND
src/citation_models/glamor/walks.py : FOUND
src/citation_models/glamor/recommend.py : FOUND
src/citation_models/glamor/language_model.py : FOUND


In [ ]:
#Check GLAMOR training requirements
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
REPO = PROJECT / "GLAMOR"
OUTPUT = REPO / "artifacts/glamor_dblp_v12"

required_files = [
    OUTPUT / "kg.json",
    OUTPUT / "walks_train.txt",
    OUTPUT / "walks_validation.txt",
    REPO / "configs/glamor/dblp_v12.yaml",
]

print("GLAMOR TRAINING FILE CHECK\n")

for path in required_files:
    print(
        path.name,
        ":",
        "FOUND" if path.exists() else "NOT FOUND"
    )
    if path.is_file():
        print("Size:", round(path.stat().st_size / (1024**2), 2), "MB")

print("\nSQLite database remains separate:",
      (PROJECT / "outputs/citation_knowledge_graph.sqlite").exists())

GLAMOR TRAINING FILE CHECK

kg.json : NOT FOUND
walks_train.txt : NOT FOUND
walks_validation.txt : NOT FOUND
dblp_v12.yaml : FOUND
Size: 0.0 MB

SQLite database remains separate: True


In [ ]:
from pathlib import Path
import yaml

CONFIG = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/configs/glamor/dblp_v12.yaml"
)

with open(CONFIG, "r") as f:
    config = yaml.safe_load(f)

print("DATASET SETTINGS:")
print(config.get("dataset"))

print("\nOUTPUT DIRECTORY:")
print(config.get("output_dir"))

print("\nWALK SETTINGS:")
print(config.get("walks"))

print("\nLANGUAGE MODEL SETTINGS:")
print(config.get("language_model"))

DATASET SETTINGS:
{'path': '/content/drive/MyDrive/GLAMOR_Project/dataset/dblp_v12_ready.txt', 'format': 'aminer', 'train_ratio': 0.8}

OUTPUT DIRECTORY:
artifacts/glamor_dblp_v12

WALK SETTINGS:
{'num_walks': 450000, 'termination_probability': 0.05, 'max_steps': 1000000, 'validation_ratio': 0.15, 'corpus_split_seed': 42}

LANGUAGE MODEL SETTINGS:
{'model_name': 'distilroberta-base', 'seed': 42, 'mlm_probability': 0.15, 'batch_size': 16, 'epochs': 6, 'learning_rate': 0.0005, 'weight_decay': 0.01, 'gradient_accumulation_steps': 20, 'fp16': True, 'dataloader_num_workers': 8, 'max_length': 512, 'embedding_batch_size': 64, 'gradient_checkpointing': False, 'resume_from_checkpoint': False}


In [ ]:
import psutil

memory = psutil.virtual_memory()

print("Total RAM:", round(memory.total / (1024**3), 2), "GB")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("Used RAM:", round(memory.used / (1024**3), 2), "GB")

Total RAM: 12.67 GB
Available RAM: 11.13 GB
Used RAM: 1.22 GB


In [ ]:
from pathlib import Path
import yaml

CONFIG = Path(
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/configs/glamor/dblp_v12.yaml"
)

config = yaml.safe_load(CONFIG.read_text())

print("FULL WALK CONFIGURATION:")
print(yaml.dump(config["walks"], sort_keys=False))

print("\nFULL LANGUAGE MODEL CONFIGURATION:")
print(yaml.dump(config["language_model"], sort_keys=False))

FULL WALK CONFIGURATION:
num_walks: 450000
termination_probability: 0.05
max_steps: 1000000
validation_ratio: 0.15
corpus_split_seed: 42


FULL LANGUAGE MODEL CONFIGURATION:
model_name: distilroberta-base
seed: 42
mlm_probability: 0.15
batch_size: 16
epochs: 6
learning_rate: 0.0005
weight_decay: 0.01
gradient_accumulation_steps: 20
fp16: true
dataloader_num_workers: 8
max_length: 512
embedding_batch_size: 64
gradient_checkpointing: false
resume_from_checkpoint: false



In [ ]:
import sys
import os
import inspect
from pathlib import Path

REPO = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR")

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from citation_models.glamor.language_model import (
    fine_tune_graph_language_model,
    GraphLanguageModelEncoder
)

print("TRAINING FUNCTION SIGNATURE:")
print(inspect.signature(fine_tune_graph_language_model))

print("\nTRAINING FUNCTION SOURCE:")
print(inspect.getsource(fine_tune_graph_language_model))

print("\nENCODER SIGNATURE:")
print(inspect.signature(GraphLanguageModelEncoder))

TRAINING FUNCTION SIGNATURE:
(train_corpus: 'str | Path', validation_corpus: 'str | Path', output_dir: 'str | Path', *, model_name: 'str' = 'distilroberta-base', mlm_probability: 'float' = 0.15, batch_size: 'int' = 16, epochs: 'int' = 6, learning_rate: 'float' = 0.0005, weight_decay: 'float' = 0.01, gradient_accumulation_steps: 'int' = 20, max_length: 'int' = 512, fp16: 'bool | None' = None, dataloader_num_workers: 'int' = 8, gradient_checkpointing: 'bool' = False, resume_from_checkpoint: 'bool | str | Path' = False, seed: 'int' = 42) -> 'dict[str, Any]'

TRAINING FUNCTION SOURCE:
def fine_tune_graph_language_model(
    train_corpus: str | Path,
    validation_corpus: str | Path,
    output_dir: str | Path,
    *,
    model_name: str = "distilroberta-base",
    mlm_probability: float = 0.15,
    batch_size: int = 16,
    epochs: int = 6,
    learning_rate: float = 5e-4,
    weight_decay: float = 0.01,
    gradient_accumulation_steps: int = 20,
    max_length: int = 512,
    fp16: bool 

In [ ]:
import inspect
from citation_models.glamor.language_model import fine_tune_graph_language_model

source = inspect.getsource(fine_tune_graph_language_model)

print(source[:6500])

def fine_tune_graph_language_model(
    train_corpus: str | Path,
    validation_corpus: str | Path,
    output_dir: str | Path,
    *,
    model_name: str = "distilroberta-base",
    mlm_probability: float = 0.15,
    batch_size: int = 16,
    epochs: int = 6,
    learning_rate: float = 5e-4,
    weight_decay: float = 0.01,
    gradient_accumulation_steps: int = 20,
    max_length: int = 512,
    fp16: bool | None = None,
    dataloader_num_workers: int = 8,
    gradient_checkpointing: bool = False,
    resume_from_checkpoint: bool | str | Path = False,
    seed: int = 42,
) -> dict[str, Any]:
    """Fine-tune DistilRoBERTa with masked-token cross-entropy (Equation 1)."""
    import torch
    from datasets import load_dataset
    from transformers import (
        AutoModelForMaskedLM,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
        Trainer,
        TrainingArguments,
    )

    train_corpus = Path(train_corpus)
    validation_corpus = Path(validation_corpus)
 

In [ ]:
import inspect
from citation_models.glamor.language_model import fine_tune_graph_language_model

source = inspect.getsource(fine_tune_graph_language_model)

for i, line in enumerate(source.splitlines()[:100], 1):
    print(f"{i}: {line}")

1: def fine_tune_graph_language_model(
2:     train_corpus: str | Path,
3:     validation_corpus: str | Path,
4:     output_dir: str | Path,
5:     *,
6:     model_name: str = "distilroberta-base",
7:     mlm_probability: float = 0.15,
8:     batch_size: int = 16,
9:     epochs: int = 6,
10:     learning_rate: float = 5e-4,
11:     weight_decay: float = 0.01,
12:     gradient_accumulation_steps: int = 20,
13:     max_length: int = 512,
14:     fp16: bool | None = None,
15:     dataloader_num_workers: int = 8,
16:     gradient_checkpointing: bool = False,
17:     resume_from_checkpoint: bool | str | Path = False,
18:     seed: int = 42,
19: ) -> dict[str, Any]:
20:     """Fine-tune DistilRoBERTa with masked-token cross-entropy (Equation 1)."""
21:     import torch
22:     from datasets import load_dataset
23:     from transformers import (
24:         AutoModelForMaskedLM,
25:         AutoTokenizer,
26:         DataCollatorForLanguageModeling,
27:         Trainer,
28:         TrainingAr

In [ ]:
import inspect
from citation_models.glamor.walks import (
    generate_walk_corpus,
    attributed_random_walk
)

print("RANDOM WALK FUNCTION:")
print(inspect.getsource(attributed_random_walk))

print("\nCORPUS GENERATION FUNCTION:")
print(inspect.getsource(generate_walk_corpus))

RANDOM WALK FUNCTION:
def attributed_random_walk(
    graph: AttributedKnowledgeGraph,
    rng: random.Random,
    *,
    termination_probability: float = 0.05,
    max_steps: int = 128,
    start_node: str | None = None,
) -> list[tuple[str, str | None]]:
    if not 0.0 < termination_probability <= 1.0:
        raise ValueError("termination_probability must be in (0, 1]")
    if max_steps < 1:
        raise ValueError("max_steps must be positive")
    if not graph.entities:
        raise ValueError("Cannot walk an empty graph")
    if start_node is not None and start_node not in graph.entities:
        raise ValueError(f"Unknown walk start node: {start_node}")
    adjacency = graph.adjacency
    current = start_node or rng.choice(sorted(graph.entities))
    walk: list[tuple[str, str | None]] = [(current, None)]
    for _ in range(max_steps):
        outgoing = adjacency[current]
        if not outgoing or rng.random() < termination_probability:
            break
        relation, targ

In [ ]:
import inspect
from citation_models.glamor.kg import (
    Entity,
    Edge,
    AttributedKnowledgeGraph,
    build_glamor_kg
)

print("ENTITY STRUCTURE:")
print(inspect.signature(Entity))

print("\nEDGE STRUCTURE:")
print(inspect.signature(Edge))

print("\nGRAPH STRUCTURE:")
print(inspect.signature(AttributedKnowledgeGraph))

print("\nENTITY ID FUNCTION:")
from citation_models.glamor.kg import entity_id
print(inspect.signature(entity_id))

ENTITY STRUCTURE:
(id: 'str', kind: 'str', name: 'str', attributes: 'dict[str, Any]' = <factory>) -> None

EDGE STRUCTURE:
(source: 'str', relation: 'str', target: 'str') -> None

GRAPH STRUCTURE:
(entities: 'dict[str, Entity]', edges: 'list[Edge]') -> None

ENTITY ID FUNCTION:
(kind: 'str', raw_id: 'str') -> 'str'


In [ ]:
import inspect
from citation_models.glamor.textualize import (
    textualize_entity,
    textualize_walk
)

print("ENTITY TEXT FORMAT:")
print(inspect.getsource(textualize_entity))

print("\nWALK TEXT FORMAT:")
print(inspect.getsource(textualize_walk))

ENTITY TEXT FORMAT:
def textualize_entity(entity: Entity) -> str:
    """Rule-based F(.) textualization from GLAMOR Section 2.2.2."""
    attrs = entity.attributes
    if entity.kind == "paper":
        parts = [f"A paper {attrs.get('pid', entity.id)} titled {entity.name}"]
        if attrs.get("year"):
            parts.append(f"published in year {attrs['year']}")
        if attrs.get("citation_count") is not None:
            parts.append(f"with citation count {attrs['citation_count']}")
        if attrs.get("keywords"):
            parts.append("with keywords " + ", ".join(attrs["keywords"]))
        return " ".join(parts)
    if entity.kind == "author":
        text = f"An author {attrs.get('aid', entity.id)} named {entity.name}"
        if attrs.get("affiliation"):
            text += f" affiliated with {attrs['affiliation']}"
        if attrs.get("paper_count") is not None:
            text += f" with paper count {attrs['paper_count']}"
        if attrs.get("citation_count") is n

In [ ]:
import sqlite3
import sys
import os
import random
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
REPO = PROJECT / "GLAMOR"
DB = PROJECT / "outputs/citation_knowledge_graph.sqlite"

PILOT = PROJECT / "outputs/glamor_pilot"
PILOT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from citation_models.glamor.kg import (
    Entity,
    Edge,
    AttributedKnowledgeGraph,
    entity_id
)
from citation_models.glamor.walks import generate_walk_corpus

conn = sqlite3.connect(str(DB))

# Select 1,000 papers that have outgoing citations
source_ids = [
    row[0] for row in conn.execute("""
        SELECT DISTINCT source_id
        FROM citations
        ORDER BY source_id
        LIMIT 1000
    """)
]

if not source_ids:
    conn.close()
    raise RuntimeError("No citation-connected papers found.")

placeholders = ",".join("?" for _ in source_ids)

citation_rows = conn.execute(f"""
    SELECT source_id, target_id
    FROM citations
    WHERE source_id IN ({placeholders})
""", source_ids).fetchall()

# Collect all graph node IDs
paper_ids = set(source_ids)

for source, target in citation_rows:
    paper_ids.add(source)
    paper_ids.add(target)

# Load paper titles and metadata in batches
papers = {}

all_ids = list(paper_ids)

for start in range(0, len(all_ids), 500):
    batch = all_ids[start:start + 500]
    marks = ",".join("?" for _ in batch)

    rows = conn.execute(f"""
        SELECT paper_id, title, year
        FROM papers
        WHERE paper_id IN ({marks})
    """, batch).fetchall()

    for pid, title, year in rows:
        papers[pid] = (title or pid, year)

conn.close()

# Create original GLAMOR graph objects
entities = {}
edges = []

for pid in paper_ids:
    title, year = papers.get(pid, (pid, None))

    attrs = {"pid": pid}

    if year:
        attrs["year"] = year

    entities[entity_id("paper", pid)] = Entity(
        id=entity_id("paper", pid),
        kind="paper",
        name=title,
        attributes=attrs
    )

for source, target in citation_rows:
    edges.append(
        Edge(
            source=entity_id("paper", source),
            relation="cites",
            target=entity_id("paper", target)
        )
    )

graph = AttributedKnowledgeGraph(
    entities=entities,
    edges=edges
)

print("Pilot graph entities:", len(graph.entities))
print("Pilot graph citation edges:", len(graph.edges))

# Generate a small original GLAMOR walk corpus
walks = list(generate_walk_corpus(
    graph,
    num_walks=1000,
    termination_probability=0.15,
    max_steps=20,
    seed=1203
))

# Save separate pilot training and validation files
random.Random(42).shuffle(walks)

split = int(len(walks) * 0.85)

train_file = PILOT / "walks_train.txt"
validation_file = PILOT / "walks_validation.txt"

train_file.write_text(
    "\n".join(walks[:split]) + "\n",
    encoding="utf-8"
)

validation_file.write_text(
    "\n".join(walks[split:]) + "\n",
    encoding="utf-8"
)

print("\nPILOT CORPUS CREATED")
print("Training walks:", len(walks[:split]))
print("Validation walks:", len(walks[split:]))
print("Training file:", train_file)
print("Validation file:", validation_file)
print("Example walk:\n", walks[0])

Pilot graph entities: 7948
Pilot graph citation edges: 7388

PILOT CORPUS CREATED
Training walks: 850
Validation walks: 150
Training file: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/walks_train.txt
Validation file: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/walks_validation.txt
Example walk:
 A paper 10000001677 titled Fuzzy Arithmetic for Uncertainty Analysis. cites. A paper 10002863679 titled Error and uncertainty in modeling and simulation.


In [ ]:
import transformers
import inspect
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)

print("\nTrainingArguments parameters:")
print(inspect.signature(TrainingArguments.__init__))

print("\nSupports overwrite_output_dir:",
      "overwrite_output_dir" in inspect.signature(
          TrainingArguments.__init__
      ).parameters)

print("Supports eval_strategy:",
      "eval_strategy" in inspect.signature(
          TrainingArguments.__init__
      ).parameters)

print("Supports evaluation_strategy:",
      "evaluation_strategy" in inspect.signature(
          TrainingArguments.__init__
      ).parameters)

Transformers version: 5.21.0

TrainingArguments parameters:
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any

In [ ]:
import transformers
import numpy
import huggingface_hub
import inspect
from transformers import TrainingArguments

print("Transformers:", transformers.__version__)
print("NumPy:", numpy.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

params = inspect.signature(TrainingArguments.__init__).parameters

print("overwrite_output_dir supported:",
      "overwrite_output_dir" in params)

print("evaluation_strategy supported:",
      "evaluation_strategy" in params)

print("eval_strategy supported:",
      "eval_strategy" in params)

Transformers: 5.21.0
NumPy: 2.1.3
Hugging Face Hub: 1.33.0
overwrite_output_dir supported: False
evaluation_strategy supported: False
eval_strategy supported: True


In [ ]:
from pathlib import Path

PILOT = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot"
)

MODEL = PILOT / "language_model_compat"

print("Model folder exists:", MODEL.exists())

if MODEL.exists():
    for item in MODEL.iterdir():
        print(item.name)
else:
    print("No model checkpoint saved.")

Model folder exists: False
No model checkpoint saved.


In [ ]:
import os
import sys
import inspect
from pathlib import Path

import torch
import transformers

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
REPO = PROJECT / "GLAMOR"
PILOT = PROJECT / "outputs/glamor_pilot"

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import citation_models.glamor.language_model as lm

# Create a smaller CPU pilot from existing files
train_lines = (PILOT / "walks_train.txt").read_text().splitlines()
valid_lines = (PILOT / "walks_validation.txt").read_text().splitlines()

CPU_DIR = PILOT / "cpu_small_pilot"
CPU_DIR.mkdir(parents=True, exist_ok=True)

(CPU_DIR / "train.txt").write_text(
    "\n".join(train_lines[:100]) + "\n"
)

(CPU_DIR / "validation.txt").write_text(
    "\n".join(valid_lines[:20]) + "\n"
)

# Compatibility with installed Transformers
OriginalTrainingArguments = transformers.TrainingArguments

def compatible_training_arguments(*args, **kwargs):
    supported = inspect.signature(
        OriginalTrainingArguments.__init__
    ).parameters

    kwargs = {
        key: value for key, value in kwargs.items()
        if key in supported
    }

    kwargs["eval_strategy"] = "no"
    kwargs["save_strategy"] = "no"
    kwargs["load_best_model_at_end"] = False
    kwargs["report_to"] = "none"

    return OriginalTrainingArguments(*args, **kwargs)

transformers.TrainingArguments = compatible_training_arguments

OUTPUT = CPU_DIR / "trained_model"

print("Device: CPU")
print("Training walks:", len(train_lines[:100]))
print("Validation walks:", len(valid_lines[:20]))
print("Starting one-epoch pilot training...")

metrics = lm.fine_tune_graph_language_model(
    CPU_DIR / "train.txt",
    CPU_DIR / "validation.txt",
    OUTPUT,
    model_name="distilroberta-base",
    mlm_probability=0.15,
    batch_size=2,
    epochs=1,
    learning_rate=0.00005,
    weight_decay=0.01,
    gradient_accumulation_steps=1,
    max_length=64,
    fp16=False,
    dataloader_num_workers=0,
    gradient_checkpointing=False,
    resume_from_checkpoint=False,
    seed=42
)

print("\nCPU PILOT TRAINING FINISHED")
print("Metrics:", metrics)
print("Model saved:", OUTPUT.exists())

Device: CPU
Training walks: 100
Validation walks: 20
Starting one-epoch pilot training...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from th

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss



CPU PILOT TRAINING FINISHED
Metrics: {'train_runtime': 90.6651, 'train_samples_per_second': 1.103, 'train_steps_per_second': 0.551, 'total_flos': 1275448780584.0, 'train_loss': 3.3753900146484375, 'epoch': 1.0, 'eval_loss': 3.1332340240478516, 'eval_runtime': 3.38, 'eval_samples_per_second': 5.917, 'eval_steps_per_second': 2.959, 'model': {'model_name': 'distilroberta-base', 'model_class': 'RobertaForMaskedLM', 'hidden_size': 768, 'parameter_count': 82170201, 'revision': 'fb53ab8802853c8e4fbdbcd0529f21fc6f459b2b'}}
Model saved: True


In [ ]:
from pathlib import Path

MODEL = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/cpu_small_pilot/trained_model"
)

print("MODEL EXISTS:", MODEL.exists())

if MODEL.exists():
    for item in MODEL.iterdir():
        print(item.name)

MODEL EXISTS: True
config.json
model.safetensors
tokenizer_config.json
special_tokens_map.json
vocab.json
merges.txt
tokenizer.json
training_args.bin


In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

MODEL = "/content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/cpu_small_pilot/trained_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForMaskedLM.from_pretrained(MODEL)

model.eval()

# Use the correct mask token
text = f"A paper about fuzzy sets and {tokenizer.mask_token}"

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

mask_position = (
    inputs["input_ids"][0] == tokenizer.mask_token_id
).nonzero(as_tuple=True)[0].item()

predictions = outputs.logits[0, mask_position].topk(5).indices

print("MODEL TEST SUCCESSFUL")
print("Input:", text)
print("\nTop 5 predicted words:")

for token_id in predictions:
    print("-", tokenizer.decode([token_id]))

MODEL TEST SUCCESSFUL
Input: A paper about fuzzy sets and <mask>

Top 5 predicted words:
-  graphs
-  lists
-  statistics
-  arrays
-  behaviour


In [ ]:
import sqlite3
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from torch.nn.functional import normalize

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
DB = PROJECT / "outputs/citation_knowledge_graph.sqlite"
MODEL = PROJECT / "outputs/glamor_pilot/cpu_small_pilot/trained_model"

# Load the saved pilot model
tokenizer = AutoTokenizer.from_pretrained(str(MODEL))
encoder = AutoModel.from_pretrained(str(MODEL))
encoder.eval()
device = torch.device("cpu")
encoder.to(device)

conn = sqlite3.connect(str(DB))

# Select 20 test papers that have known relevant training papers
query_rows = conn.execute("""
    SELECT q.paper_id, q.title
    FROM test_queries q
    JOIN test_relevance r
      ON q.paper_id = r.test_paper_id
    GROUP BY q.paper_id, q.title
    ORDER BY q.paper_id
    LIMIT 20
""").fetchall()

if not query_rows:
    conn.close()
    raise RuntimeError("No eligible test queries found.")

relevance = {}

for query_id, _ in query_rows:
    relevance[query_id] = {
        row[0] for row in conn.execute("""
            SELECT relevant_paper_id
            FROM test_relevance
            WHERE test_paper_id = ?
        """, (query_id,))
    }

# Build a small candidate pool:
# all relevant papers for these queries + 500 training papers
candidate_ids = set().union(*relevance.values())

negative_ids = conn.execute("""
    SELECT paper_id
    FROM training_ids
    ORDER BY paper_id
    LIMIT 500
""").fetchall()

candidate_ids.update(row[0] for row in negative_ids)

# Retrieve candidate titles in batches
titles = {}

candidate_list = list(candidate_ids)

for start in range(0, len(candidate_list), 400):
    batch = candidate_list[start:start + 400]
    marks = ",".join("?" for _ in batch)

    rows = conn.execute(
        f"SELECT paper_id, title FROM papers WHERE paper_id IN ({marks})",
        batch
    ).fetchall()

    titles.update({
        pid: title or ""
        for pid, title in rows
    })

conn.close()

# Keep only candidates with available titles
candidate_list = [
    pid for pid in candidate_list
    if titles.get(pid, "").strip()
]

def encode_texts(texts, batch_size=16):
    vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=96
        ).to(device)

        with torch.no_grad():
            outputs = encoder(**inputs)

            hidden = outputs.last_hidden_state
            mask = inputs["attention_mask"].unsqueeze(-1)

            pooled = (hidden * mask).sum(dim=1)
            pooled = pooled / mask.sum(dim=1).clamp(min=1)

            vectors.append(normalize(pooled, p=2, dim=1).cpu())

    return torch.cat(vectors, dim=0)

print("Encoding candidate paper titles:", len(candidate_list))

candidate_vectors = encode_texts(
    [titles[pid] for pid in candidate_list]
)

print("Encoding test paper titles:", len(query_rows))

query_vectors = encode_texts(
    [title or "" for _, title in query_rows]
)

# Rank candidates and calculate preliminary metrics
precision_scores = []
recall_scores = []
ap_scores = []
ndcg_scores = []
hit_scores = []

for i, (query_id, _) in enumerate(query_rows):

    scores = torch.matmul(
        candidate_vectors,
        query_vectors[i]
    )

    ranking = torch.argsort(scores, descending=True).tolist()

    ranked_ids = [candidate_list[j] for j in ranking[:10]]
    relevant = relevance[query_id] & set(candidate_list)

    if not relevant:
        continue

    hits = [1 if pid in relevant else 0 for pid in ranked_ids]

    precision_scores.append(sum(hits) / 10)
    recall_scores.append(sum(hits) / len(relevant))
    hit_scores.append(1 if sum(hits) else 0)

    # Average precision at 10
    cumulative = 0
    precision_sum = 0

    for rank, hit in enumerate(hits, 1):
        if hit:
            cumulative += 1
            precision_sum += cumulative / rank

    ap_scores.append(
        precision_sum / min(len(relevant), 10)
    )

    # nDCG at 10
    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, 1)
    )

    ideal_hits = min(len(relevant), 10)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    ndcg_scores.append(dcg / idcg if idcg else 0)

print("\nPRELIMINARY PILOT EVALUATION")
print("Evaluated queries:", len(precision_scores))
print("Candidate papers:", len(candidate_list))
print("Precision@10:", round(np.mean(precision_scores), 4))
print("Recall@10:", round(np.mean(recall_scores), 4))
print("MAP@10:", round(np.mean(ap_scores), 4))
print("nDCG@10:", round(np.mean(ndcg_scores), 4))
print("Hit Rate@10:", round(np.mean(hit_scores), 4))

Some weights of RobertaModel were not initialized from the model checkpoint at /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/cpu_small_pilot/trained_model and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Encoding candidate paper titles: 628
Encoding test paper titles: 20

PRELIMINARY PILOT EVALUATION
Evaluated queries: 20
Candidate papers: 628
Precision@10: 0.045
Recall@10: 0.0664
MAP@10: 0.0665
nDCG@10: 0.1085
Hit Rate@10: 0.3


In [ ]:
#TF-IDF comparison
import sqlite3
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
DB = PROJECT / "outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(str(DB))

# Use the same 20 test queries
query_rows = conn.execute("""
    SELECT q.paper_id, q.title
    FROM test_queries q
    JOIN test_relevance r
      ON q.paper_id = r.test_paper_id
    GROUP BY q.paper_id, q.title
    ORDER BY q.paper_id
    LIMIT 20
""").fetchall()

relevance = {
    qid: {
        row[0] for row in conn.execute("""
            SELECT relevant_paper_id
            FROM test_relevance
            WHERE test_paper_id = ?
        """, (qid,))
    }
    for qid, _ in query_rows
}

# Recreate the same candidate pool
candidate_ids = set().union(*relevance.values())

negative_ids = conn.execute("""
    SELECT paper_id
    FROM training_ids
    ORDER BY paper_id
    LIMIT 500
""").fetchall()

candidate_ids.update(row[0] for row in negative_ids)
candidate_list = list(candidate_ids)

# Retrieve candidate titles
titles = {}

for start in range(0, len(candidate_list), 400):
    batch = candidate_list[start:start + 400]
    marks = ",".join("?" for _ in batch)

    rows = conn.execute(
        f"SELECT paper_id, title FROM papers WHERE paper_id IN ({marks})",
        batch
    ).fetchall()

    titles.update({
        pid: title or ""
        for pid, title in rows
    })

conn.close()

# Keep candidates with titles
candidate_list = [
    pid for pid in candidate_list
    if titles.get(pid, "").strip()
]

# TF-IDF title vectors
documents = [titles[pid] for pid in candidate_list]

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000
)

candidate_vectors = vectorizer.fit_transform(documents)

query_vectors = vectorizer.transform([
    title or "" for _, title in query_rows
])

precision_values = []
recall_values = []
ap_values = []
ndcg_values = []
hit_values = []

for i, (query_id, _) in enumerate(query_rows):

    scores = cosine_similarity(
        query_vectors[i],
        candidate_vectors
    ).flatten()

    ranking = np.argsort(-scores)[:10]
    ranked_ids = [candidate_list[j] for j in ranking]

    relevant = relevance[query_id] & set(candidate_list)

    if not relevant:
        continue

    hits = [
        1 if pid in relevant else 0
        for pid in ranked_ids
    ]

    precision_values.append(sum(hits) / 10)
    recall_values.append(sum(hits) / len(relevant))
    hit_values.append(int(sum(hits) > 0))

    cumulative = 0
    precision_sum = 0

    for rank, hit in enumerate(hits, 1):
        if hit:
            cumulative += 1
            precision_sum += cumulative / rank

    ap_values.append(
        precision_sum / min(len(relevant), 10)
    )

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, 1)
    )

    ideal = min(len(relevant), 10)

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal + 1)
    )

    ndcg_values.append(dcg / idcg if idcg else 0)

print("\nTF-IDF BASELINE RESULTS")
print("Evaluated queries:", len(precision_values))
print("Candidate papers:", len(candidate_list))
print("Precision@10:", round(np.mean(precision_values), 4))
print("Recall@10:", round(np.mean(recall_values), 4))
print("MAP@10:", round(np.mean(ap_values), 4))
print("nDCG@10:", round(np.mean(ndcg_values), 4))
print("Hit Rate@10:", round(np.mean(hit_values), 4))


TF-IDF BASELINE RESULTS
Evaluated queries: 20
Candidate papers: 628
Precision@10: 0.29
Recall@10: 0.44
MAP@10: 0.384
nDCG@10: 0.4788
Hit Rate@10: 0.75


In [ ]:
import sqlite3
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
DB = PROJECT / "outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(str(DB))

# Same 20 test queries
query_rows = conn.execute("""
    SELECT q.paper_id, q.title
    FROM test_queries q
    JOIN test_relevance r ON q.paper_id = r.test_paper_id
    GROUP BY q.paper_id, q.title
    ORDER BY q.paper_id
    LIMIT 20
""").fetchall()

relevance = {
    qid: {
        row[0] for row in conn.execute("""
            SELECT relevant_paper_id
            FROM test_relevance
            WHERE test_paper_id = ?
        """, (qid,))
    }
    for qid, _ in query_rows
}

# Recreate the same candidate pool
candidate_ids = set().union(*relevance.values())

negative_ids = conn.execute("""
    SELECT paper_id
    FROM training_ids
    ORDER BY paper_id
    LIMIT 500
""").fetchall()

candidate_ids.update(row[0] for row in negative_ids)
candidate_list = list(candidate_ids)

# Retrieve candidate titles
titles = {}

for start in range(0, len(candidate_list), 400):
    batch = candidate_list[start:start + 400]
    marks = ",".join("?" for _ in batch)

    rows = conn.execute(
        f"SELECT paper_id, title FROM papers WHERE paper_id IN ({marks})",
        batch
    ).fetchall()

    titles.update({pid: title or "" for pid, title in rows})

# Citation popularity: number of incoming training citations
citation_counts = {}

for start in range(0, len(candidate_list), 400):
    batch = candidate_list[start:start + 400]
    marks = ",".join("?" for _ in batch)

    rows = conn.execute(f"""
        SELECT target_id, COUNT(*)
        FROM citations
        WHERE target_id IN ({marks})
        GROUP BY target_id
    """, batch).fetchall()

    citation_counts.update(dict(rows))

conn.close()

candidate_list = [
    pid for pid in candidate_list
    if titles.get(pid, "").strip()
]

# TF-IDF title similarity
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000
)

candidate_vectors = vectorizer.fit_transform(
    [titles[pid] for pid in candidate_list]
)

query_vectors = vectorizer.transform(
    [title or "" for _, title in query_rows]
)

# Calculate hybrid rankings and metrics
precision_values = []
recall_values = []
ap_values = []
ndcg_values = []
hit_values = []

hybrid_results = {}

for i, (query_id, _) in enumerate(query_rows):

    # Rank by title similarity
    scores = cosine_similarity(
        query_vectors[i], candidate_vectors
    ).flatten()

    title_order = np.argsort(-scores)

    title_rank = {
        candidate_list[j]: rank
        for rank, j in enumerate(title_order, 1)
    }

    # Rank by citation popularity
    popularity_order = sorted(
        candidate_list,
        key=lambda pid: citation_counts.get(pid, 0),
        reverse=True
    )

    citation_rank = {
        pid: rank
        for rank, pid in enumerate(popularity_order, 1)
    }

    # Reciprocal Rank Fusion
    combined = []

    for pid in candidate_list:
        rrf = (
            1 / (60 + title_rank[pid])
            + 1 / (60 + citation_rank[pid])
        )
        combined.append((pid, rrf))

    combined.sort(key=lambda x: x[1], reverse=True)

    ranked_ids = [pid for pid, score in combined[:10]]
    hybrid_results[query_id] = combined[:10]

    relevant = relevance[query_id] & set(candidate_list)

    if not relevant:
        continue

    hits = [int(pid in relevant) for pid in ranked_ids]

    precision_values.append(sum(hits) / 10)
    recall_values.append(sum(hits) / len(relevant))
    hit_values.append(int(sum(hits) > 0))

    # MAP@10
    cumulative = 0
    precision_sum = 0

    for rank, hit in enumerate(hits, 1):
        if hit:
            cumulative += 1
            precision_sum += cumulative / rank

    ap_values.append(
        precision_sum / min(len(relevant), 10)
    )

    # nDCG@10
    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, 1)
    )

    ideal = min(len(relevant), 10)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal + 1)
    )

    ndcg_values.append(dcg / idcg if idcg else 0)

print("\nHYBRID TF-IDF + CITATION RESULTS")
print("Evaluated queries:", len(precision_values))
print("Candidate papers:", len(candidate_list))
print("Precision@10:", round(np.mean(precision_values), 4))
print("Recall@10:", round(np.mean(recall_values), 4))
print("MAP@10:", round(np.mean(ap_values), 4))
print("nDCG@10:", round(np.mean(ndcg_values), 4))
print("Hit Rate@10:", round(np.mean(hit_values), 4))

# Display example recommendations
print("\nEXAMPLE HYBRID RECOMMENDATIONS:")

first_query = query_rows[0][0]

for rank, (pid, score) in enumerate(hybrid_results[first_query], 1):
    print(f"{rank}. {titles[pid]}")
    print("   Paper ID:", pid)
    print("   RRF score:", round(score, 6))


HYBRID TF-IDF + CITATION RESULTS
Evaluated queries: 20
Candidate papers: 628
Precision@10: 0.275
Recall@10: 0.4501
MAP@10: 0.3247
nDCG@10: 0.4472
Hit Rate@10: 0.85

EXAMPLE HYBRID RECOMMENDATIONS:
1. Secret image sharing
   Paper ID: 10001515214
   RRF score: 0.028589
2. Secret image sharing with steganography and authentication
   Paper ID: 10002520390
   RRF score: 0.027118
3. A Tutorial on the Cross-Entropy Method
   Paper ID: 10002450504
   RRF score: 0.026876
4. Improvements of image sharing with steganography and authentication
   Paper ID: 10001206800
   RRF score: 0.025488
5. Sharing secrets in stego images with authentication
   Paper ID: 10001538376
   RRF score: 0.024971
6. Emergence of consensus and shared vocabularies in collaborative tagging systems
   Paper ID: 10002737486
   RRF score: 0.022844
7. Theoretical Improvements in Algorithmic Efficiency for Network Flow Problems
   Paper ID: 10002643757
   RRF score: 0.02235
8. Trust and tacit knowledge sharing and use
   Pa

In [ ]:
import sqlite3
import random
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
DB = PROJECT / "outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(str(DB))

# 1. Select 100 test queries with relevance data
query_rows = conn.execute("""
    SELECT q.paper_id, q.title
    FROM test_queries q
    WHERE EXISTS (
        SELECT 1 FROM test_relevance r
        WHERE r.test_paper_id = q.paper_id
    )
    ORDER BY RANDOM()
    LIMIT 100
""").fetchall()

if not query_rows:
    conn.close()
    raise RuntimeError("No eligible test queries found.")

relevance = {}

for qid, _ in query_rows:
    relevance[qid] = {
        row[0] for row in conn.execute("""
            SELECT relevant_paper_id
            FROM test_relevance
            WHERE test_paper_id = ?
        """, (qid,))
    }

# 2. Create candidate pool from relevant papers
positive_ids = set().union(*relevance.values())

# Add 1,000 randomly selected training papers
negative_rows = conn.execute("""
    SELECT paper_id
    FROM training_ids
    ORDER BY RANDOM()
    LIMIT 1000
""").fetchall()

negative_ids = {
    row[0] for row in negative_rows
} - positive_ids

candidate_ids = positive_ids | negative_ids

# 3. Retrieve candidate titles and citation counts
candidate_list = list(candidate_ids)
titles = {}
citation_counts = {}

for start in range(0, len(candidate_list), 400):
    batch = candidate_list[start:start + 400]
    marks = ",".join("?" for _ in batch)

    rows = conn.execute(
        f"""
        SELECT paper_id, title
        FROM papers
        WHERE paper_id IN ({marks})
        """,
        batch
    ).fetchall()

    titles.update({
        pid: title or "" for pid, title in rows
    })

    rows = conn.execute(
        f"""
        SELECT target_id, COUNT(*)
        FROM citations
        WHERE target_id IN ({marks})
        GROUP BY target_id
        """,
        batch
    ).fetchall()

    citation_counts.update(dict(rows))

conn.close()

candidate_list = [
    pid for pid in candidate_list
    if titles.get(pid, "").strip()
]

# Keep the same relevant set for both methods
print("Queries selected:", len(query_rows))
print("Candidate papers:", len(candidate_list))

# 4. Build TF-IDF vectors
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000
)

candidate_vectors = vectorizer.fit_transform(
    [titles[pid] for pid in candidate_list]
)

query_vectors = vectorizer.transform(
    [title or "" for _, title in query_rows]
)

# 5. Evaluate both methods
results = {
    "TF-IDF": [],
    "Hybrid": []
}

for i, (qid, _) in enumerate(query_rows):

    relevant = relevance[qid] & set(candidate_list)

    if not relevant:
        continue

    # TF-IDF ranking
    scores = cosine_similarity(
        query_vectors[i], candidate_vectors
    ).flatten()

    title_order = np.argsort(-scores)

    title_rank = {
        candidate_list[j]: rank
        for rank, j in enumerate(title_order, 1)
    }

    # Citation popularity ranking
    popularity_order = sorted(
        candidate_list,
        key=lambda pid: citation_counts.get(pid, 0),
        reverse=True
    )

    citation_rank = {
        pid: rank
        for rank, pid in enumerate(popularity_order, 1)
    }

    # Hybrid RRF ranking
    hybrid_scores = {
        pid: (
            1 / (60 + title_rank[pid])
            + 1 / (60 + citation_rank[pid])
        )
        for pid in candidate_list
    }

    rankings = {
        "TF-IDF": sorted(
            candidate_list,
            key=lambda pid: title_rank[pid]
        )[:10],

        "Hybrid": sorted(
            candidate_list,
            key=lambda pid: hybrid_scores[pid],
            reverse=True
        )[:10]
    }

    for method, ranked in rankings.items():

        hits = [
            int(pid in relevant) for pid in ranked
        ]

        precision = sum(hits) / 10
        recall = sum(hits) / len(relevant)
        hit_rate = int(sum(hits) > 0)

        cumulative = 0
        ap_sum = 0

        for rank, hit in enumerate(hits, 1):
            if hit:
                cumulative += 1
                ap_sum += cumulative / rank

        ap = ap_sum / min(len(relevant), 10)

        dcg = sum(
            hit / np.log2(rank + 1)
            for rank, hit in enumerate(hits, 1)
        )

        ideal = min(len(relevant), 10)

        idcg = sum(
            1 / np.log2(rank + 1)
            for rank in range(1, ideal + 1)
        )

        ndcg = dcg / idcg if idcg else 0

        results[method].append(
            [precision, recall, ap, ndcg, hit_rate]
        )

# 6. Display results
print("\n100-QUERY EVALUATION RESULTS")

metric_names = [
    "Precision@10",
    "Recall@10",
    "MAP@10",
    "nDCG@10",
    "Hit Rate@10"
]

for method, values in results.items():
    averages = np.mean(values, axis=0)

    print("\n", method)
    for name, value in zip(metric_names, averages):
        print(name + ":", round(float(value), 4))

print("\nQueries actually evaluated:",
      len(results["TF-IDF"]))

Queries selected: 100
Candidate papers: 1893

100-QUERY EVALUATION RESULTS

 TF-IDF
Precision@10: 0.294
Recall@10: 0.3671
MAP@10: 0.3598
nDCG@10: 0.4705
Hit Rate@10: 0.82

 Hybrid
Precision@10: 0.175
Recall@10: 0.2372
MAP@10: 0.0986
nDCG@10: 0.2005
Hit Rate@10: 0.75

Queries actually evaluated: 100


In [ ]:
import numpy as np

# Check that previous evaluation data is available
required = [
    "query_rows", "relevance", "candidate_list",
    "titles", "citation_counts",
    "query_vectors", "candidate_vectors"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Previous evaluation variables are missing: "
        + ", ".join(missing)
        + ". Tell me before rerunning anything."
    )

# Calculate title and citation ranks for every query
query_rank_data = []

for i, (query_id, _) in enumerate(query_rows):

    relevant = relevance[query_id] & set(candidate_list)

    if not relevant:
        continue

    scores = (
        query_vectors[i] @ candidate_vectors.T
    ).toarray().ravel()

    title_order = np.argsort(-scores)

    title_rank = {
        candidate_list[j]: rank
        for rank, j in enumerate(title_order, 1)
    }

    popularity_order = sorted(
        candidate_list,
        key=lambda pid: citation_counts.get(pid, 0),
        reverse=True
    )

    citation_rank = {
        pid: rank
        for rank, pid in enumerate(popularity_order, 1)
    }

    query_rank_data.append(
        (query_id, relevant, title_rank, citation_rank)
    )

def calculate_metrics(weight):
    values = []

    for query_id, relevant, title_rank, citation_rank in query_rank_data:

        # Weighted Reciprocal Rank Fusion
        combined = {
            pid: (
                weight / (60 + title_rank[pid])
                + (1 - weight) / (60 + citation_rank[pid])
            )
            for pid in candidate_list
        }

        ranked = sorted(
            candidate_list,
            key=lambda pid: combined[pid],
            reverse=True
        )[:10]

        hits = [int(pid in relevant) for pid in ranked]

        precision = sum(hits) / 10
        recall = sum(hits) / len(relevant)
        hit_rate = int(sum(hits) > 0)

        cumulative = 0
        ap_sum = 0

        for rank, hit in enumerate(hits, 1):
            if hit:
                cumulative += 1
                ap_sum += cumulative / rank

        ap = ap_sum / min(len(relevant), 10)

        dcg = sum(
            hit / np.log2(rank + 1)
            for rank, hit in enumerate(hits, 1)
        )

        ideal = min(len(relevant), 10)
        idcg = sum(
            1 / np.log2(rank + 1)
            for rank in range(1, ideal + 1)
        )

        ndcg = dcg / idcg if idcg else 0

        values.append([
            precision, recall, ap, ndcg, hit_rate
        ])

    return np.mean(values, axis=0)

print("HYBRID WEIGHT TUNING")
print("Queries evaluated:", len(query_rank_data))
print("\nTitle weight | Citation weight | Precision | Recall | MAP | nDCG | Hit Rate")

results_tuning = []

for weight in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 1.00]:

    metrics = calculate_metrics(weight)
    results_tuning.append((weight, metrics))

    print(
        f"{weight:.2f}         | "
        f"{1-weight:.2f}            | "
        f"{metrics[0]:.4f}    | "
        f"{metrics[1]:.4f} | "
        f"{metrics[2]:.4f} | "
        f"{metrics[3]:.4f} | "
        f"{metrics[4]:.4f}"
    )

best = max(
    results_tuning,
    key=lambda item: item[1][3]
)

print("\nBEST WEIGHT BY nDCG@10")
print("Title weight:", best[0])
print("Citation weight:", round(1-best[0], 2))
print("nDCG@10:", round(float(best[1][3]), 4))

HYBRID WEIGHT TUNING
Queries evaluated: 100

Title weight | Citation weight | Precision | Recall | MAP | nDCG | Hit Rate
0.50         | 0.50            | 0.1750    | 0.2372 | 0.0986 | 0.2005 | 0.7500
0.60         | 0.40            | 0.2630    | 0.3351 | 0.2110 | 0.3288 | 0.7900
0.70         | 0.30            | 0.2890    | 0.3583 | 0.2750 | 0.3936 | 0.8200
0.80         | 0.20            | 0.2960    | 0.3665 | 0.3194 | 0.4355 | 0.8200
0.90         | 0.10            | 0.2930    | 0.3635 | 0.3495 | 0.4593 | 0.8200
0.95         | 0.05            | 0.2910    | 0.3640 | 0.3523 | 0.4630 | 0.8200
1.00         | 0.00            | 0.2940    | 0.3671 | 0.3598 | 0.4705 | 0.8200

BEST WEIGHT BY nDCG@10
Title weight: 1.0
Citation weight: 0.0
nDCG@10: 0.4705


In [ ]:
import sqlite3
import numpy as np
import random
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
DB = PROJECT / "outputs/citation_knowledge_graph.sqlite"
OUTPUT = PROJECT / "outputs"

# Exclude the previous 100 queries
if "query_rows" not in globals():
    raise RuntimeError(
        "Previous 100-query IDs are unavailable. "
        "Please tell me before proceeding."
    )

previous_ids = [qid for qid, _ in query_rows]

conn = sqlite3.connect(str(DB))

placeholders = ",".join("?" for _ in previous_ids)

# Select 500 different eligible test queries
new_queries = conn.execute(f"""
    SELECT q.paper_id, q.title
    FROM test_queries q
    WHERE EXISTS (
        SELECT 1 FROM test_relevance r
        WHERE r.test_paper_id = q.paper_id
    )
    AND q.paper_id NOT IN ({placeholders})
    ORDER BY RANDOM()
    LIMIT 500
""", previous_ids).fetchall()

if len(new_queries) < 500:
    conn.close()
    raise RuntimeError(
        f"Only {len(new_queries)} new queries were found."
    )

# Retrieve relevance sets
relevance = {}

for qid, _ in new_queries:
    relevance[qid] = {
        row[0] for row in conn.execute("""
            SELECT relevant_paper_id
            FROM test_relevance
            WHERE test_paper_id = ?
        """, (qid,))
    }

positive_ids = set().union(*relevance.values())

# Select 1,000 random training papers as negative candidates
negative_rows = conn.execute("""
    SELECT paper_id
    FROM training_ids
    ORDER BY RANDOM()
    LIMIT 1000
""").fetchall()

candidate_ids = positive_ids | {
    row[0] for row in negative_rows
}

candidate_list = list(candidate_ids)
titles = {}

# Retrieve titles in batches
for start in range(0, len(candidate_list), 400):
    batch = candidate_list[start:start + 400]
    marks = ",".join("?" for _ in batch)

    rows = conn.execute(
        f"""
        SELECT paper_id, title
        FROM papers
        WHERE paper_id IN ({marks})
        """,
        batch
    ).fetchall()

    titles.update({
        pid: title or "" for pid, title in rows
    })

conn.close()

candidate_list = [
    pid for pid in candidate_list
    if titles.get(pid, "").strip()
]

print("New test queries:", len(new_queries))
print("Candidate papers:", len(candidate_list))

# Build TF-IDF vectors
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000
)

candidate_vectors = vectorizer.fit_transform(
    [titles[pid] for pid in candidate_list]
)

query_vectors = vectorizer.transform(
    [title or "" for _, title in new_queries]
)

# Evaluate
all_metrics = []

for i, (qid, _) in enumerate(new_queries):

    relevant = relevance[qid] & set(candidate_list)

    if not relevant:
        continue

    scores = cosine_similarity(
        query_vectors[i],
        candidate_vectors
    ).flatten()

    top_indices = np.argsort(-scores)[:10]
    ranked_ids = [candidate_list[j] for j in top_indices]

    hits = [int(pid in relevant) for pid in ranked_ids]

    precision = sum(hits) / 10
    recall = sum(hits) / len(relevant)
    hit_rate = int(sum(hits) > 0)

    cumulative = 0
    ap_sum = 0

    for rank, hit in enumerate(hits, 1):
        if hit:
            cumulative += 1
            ap_sum += cumulative / rank

    ap = ap_sum / min(len(relevant), 10)

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, 1)
    )

    ideal = min(len(relevant), 10)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    all_metrics.append([
        precision, recall, ap, ndcg, hit_rate
    ])

if not all_metrics:
    raise RuntimeError("No queries could be evaluated.")

averages = np.mean(all_metrics, axis=0)

print("\n500-QUERY TF-IDF EVALUATION")
print("Queries evaluated:", len(all_metrics))
print("Candidate papers:", len(candidate_list))
print("Precision@10:", round(float(averages[0]), 4))
print("Recall@10:", round(float(averages[1]), 4))
print("MAP@10:", round(float(averages[2]), 4))
print("nDCG@10:", round(float(averages[3]), 4))
print("Hit Rate@10:", round(float(averages[4]), 4))

# Save results
result_file = OUTPUT / "tfidf_500_query_evaluation.txt"

result_file.write_text(
    "500-QUERY TF-IDF EVALUATION\n"
    f"Queries evaluated: {len(all_metrics)}\n"
    f"Candidate papers: {len(candidate_list)}\n"
    f"Precision@10: {averages[0]:.4f}\n"
    f"Recall@10: {averages[1]:.4f}\n"
    f"MAP@10: {averages[2]:.4f}\n"
    f"nDCG@10: {averages[3]:.4f}\n"
    f"Hit Rate@10: {averages[4]:.4f}\n",
    encoding="utf-8"
)

print("\nResults saved:", result_file)

New test queries: 500
Candidate papers: 5466

500-QUERY TF-IDF EVALUATION
Queries evaluated: 500
Candidate papers: 5466
Precision@10: 0.232
Recall@10: 0.2929
MAP@10: 0.2694
nDCG@10: 0.3783
Hit Rate@10: 0.786

Results saved: /content/drive/MyDrive/GLAMOR_Project/outputs/tfidf_500_query_evaluation.txt


In [ ]:
import numpy as np
from pathlib import Path

all_metrics = []

for i, (query_id, _) in enumerate(new_queries):

    relevant = relevance[query_id] & set(candidate_list)

    if not relevant:
        continue

    # Corrected: no squeeze(1)
    scores = torch.matmul(
        candidate_vectors,
        query_vectors[i]
    ).numpy()

    ranking = np.argsort(-scores)[:10]

    ranked_ids = [
        candidate_list[j] for j in ranking
    ]

    hits = [int(pid in relevant) for pid in ranked_ids]

    precision = sum(hits) / 10
    recall = sum(hits) / len(relevant)
    hit_rate = int(sum(hits) > 0)

    cumulative = 0
    ap_sum = 0

    for rank, hit in enumerate(hits, 1):
        if hit:
            cumulative += 1
            ap_sum += cumulative / rank

    ap = ap_sum / min(len(relevant), 10)

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, 1)
    )

    ideal = min(len(relevant), 10)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    all_metrics.append([
        precision, recall, ap, ndcg, hit_rate
    ])

averages = np.mean(all_metrics, axis=0)

print("\nGLAMOR PILOT — 500 QUERY EVALUATION")
print("Queries evaluated:", len(all_metrics))
print("Candidate papers:", len(candidate_list))
print("Precision@10:", round(float(averages[0]), 4))
print("Recall@10:", round(float(averages[1]), 4))
print("MAP@10:", round(float(averages[2]), 4))
print("nDCG@10:", round(float(averages[3]), 4))
print("Hit Rate@10:", round(float(averages[4]), 4))

result_file = (
    Path("/content/drive/MyDrive/GLAMOR_Project/outputs")
    / "glamor_500_query_evaluation.txt"
)

result_file.write_text(
    "GLAMOR PILOT 500-QUERY EVALUATION\n"
    f"Queries evaluated: {len(all_metrics)}\n"
    f"Candidate papers: {len(candidate_list)}\n"
    f"Precision@10: {averages[0]:.4f}\n"
    f"Recall@10: {averages[1]:.4f}\n"
    f"MAP@10: {averages[2]:.4f}\n"
    f"nDCG@10: {averages[3]:.4f}\n"
    f"Hit Rate@10: {averages[4]:.4f}\n"
)

print("\nResults saved:", result_file)


GLAMOR PILOT — 500 QUERY EVALUATION
Queries evaluated: 500
Candidate papers: 5466
Precision@10: 0.0758
Recall@10: 0.1002
MAP@10: 0.0765
nDCG@10: 0.1324
Hit Rate@10: 0.45

Results saved: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_500_query_evaluation.txt


In [ ]:
from pathlib import Path

OUTPUT = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs"
)

report = """
GLAMOR PROJECT - MODEL COMPARISON

Evaluation dataset:
500 test queries
5466 candidate papers

Metric             TF-IDF       GLAMOR PILOT
Precision@10       0.2320       0.0758
Recall@10          0.2929       0.1002
MAP@10             0.2694       0.0765
nDCG@10            0.3783       0.1324
Hit Rate@10        0.7860       0.4500

Current best method: TF-IDF

Note:
GLAMOR results are from a small pilot
trained on 100 walks for one epoch.
These are preliminary evaluation results.
"""

file = OUTPUT / "model_comparison_report.txt"
file.write_text(report, encoding="utf-8")

print("Comparison report saved:", file)

Comparison report saved: /content/drive/MyDrive/GLAMOR_Project/outputs/model_comparison_report.txt


In [ ]:
from pathlib import Path
import sqlite3

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")

APP = PROJECT / "application"
TEMPLATES = APP / "templates"
STATIC = APP / "static"
OUTPUTS = PROJECT / "outputs"

# Create application folders
TEMPLATES.mkdir(parents=True, exist_ok=True)
STATIC.mkdir(parents=True, exist_ok=True)

# Verify existing database
DB = OUTPUTS / "citation_knowledge_graph.sqlite"

print("APPLICATION FOLDER:", APP.exists())
print("TEMPLATES FOLDER:", TEMPLATES.exists())
print("STATIC FOLDER:", STATIC.exists())
print("DATABASE:", DB.exists())

if DB.exists():
    conn = sqlite3.connect(str(DB))

    print("\nDATABASE TABLES:")
    tables = conn.execute("""
        SELECT name FROM sqlite_master
        WHERE type='table'
        ORDER BY name
    """).fetchall()

    for table in tables:
        print("-", table[0])

    conn.close()

print("\nAPPLICATION FOLDER CREATED SUCCESSFULLY")

APPLICATION FOLDER: True
TEMPLATES FOLDER: True
STATIC FOLDER: True
DATABASE: True

DATABASE TABLES:
- citations
- evaluation_progress
- papers
- papers_fts
- papers_fts_config
- papers_fts_content
- papers_fts_data
- papers_fts_docsize
- papers_fts_idx
- progress
- test_ids
- test_queries
- test_relevance
- training_ids

APPLICATION FOLDER CREATED SUCCESSFULLY


In [ ]:
!pip install -q flask

In [ ]:
import flask

print("Flask version:", flask.__version__)
print("FLASK READY")

Flask version: 3.1.3
FLASK READY


/tmp/ipykernel_56223/3383746471.py:3: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask version:", flask.__version__)


In [ ]:
from pathlib import Path

APP = Path("/content/drive/MyDrive/GLAMOR_Project/application")

app_code = r'''
import sqlite3
from pathlib import Path
from flask import Flask, request, render_template, jsonify

app = Flask(__name__)

DB = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/"
    "citation_knowledge_graph.sqlite"
)

def get_db():
    conn = sqlite3.connect(str(DB))
    conn.row_factory = sqlite3.Row
    return conn

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/search")
def search():
    query = request.args.get("q", "").strip()

    if not query:
        return jsonify([])

    conn = get_db()
    rows = conn.execute("""
        SELECT paper_id, title, year
        FROM papers
        WHERE papers_fts MATCH ?
        LIMIT 20
    """, (query.replace('"', '""'),)).fetchall()

    conn.close()

    return jsonify([dict(row) for row in rows])

@app.route("/paper/<paper_id>")
def paper_details(paper_id):
    conn = get_db()

    paper = conn.execute("""
        SELECT paper_id, title, year
        FROM papers
        WHERE paper_id = ?
    """, (paper_id,)).fetchone()

    if paper is None:
        conn.close()
        return jsonify({"error": "Paper not found"}), 404

    citations = conn.execute("""
        SELECT COUNT(*)
        FROM citations
        WHERE target_id = ?
    """, (paper_id,)).fetchone()[0]

    conn.close()

    return jsonify({
        "paper_id": paper["paper_id"],
        "title": paper["title"],
        "year": paper["year"],
        "citation_count": citations
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

(APP / "app.py").write_text(app_code, encoding="utf-8")

print("Flask backend created:", (APP / "app.py").exists())
print("Location:", APP / "app.py")

Flask backend created: True
Location: /content/drive/MyDrive/GLAMOR_Project/application/app.py


In [ ]:
from pathlib import Path

APP = Path("/content/drive/MyDrive/GLAMOR_Project/application")
TEMPLATES = APP / "templates"

# Corrected Flask backend
app_code = r'''
import sqlite3
from pathlib import Path
from flask import Flask, request, render_template, jsonify

app = Flask(__name__)

DB = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/"
    "citation_knowledge_graph.sqlite"
)

def get_db():
    conn = sqlite3.connect(str(DB))
    conn.row_factory = sqlite3.Row
    return conn

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/search")
def search():
    query = request.args.get("q", "").strip()

    if not query:
        return jsonify([])

    conn = get_db()

    rows = conn.execute("""
        SELECT paper_id, title, year
        FROM papers
        WHERE title LIKE ?
        LIMIT 20
    """, (f"%{query}%",)).fetchall()

    conn.close()
    return jsonify([dict(row) for row in rows])

@app.route("/paper/<paper_id>")
def paper_details(paper_id):
    conn = get_db()

    paper = conn.execute("""
        SELECT paper_id, title, year
        FROM papers
        WHERE paper_id = ?
    """, (paper_id,)).fetchone()

    if paper is None:
        conn.close()
        return jsonify({"error": "Paper not found"}), 404

    citations = conn.execute("""
        SELECT COUNT(*)
        FROM citations
        WHERE target_id = ?
    """, (paper_id,)).fetchone()[0]

    conn.close()

    return jsonify({
        "paper_id": paper["paper_id"],
        "title": paper["title"],
        "year": paper["year"],
        "citation_count": citations
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

(APP / "app.py").write_text(app_code, encoding="utf-8")

# Create homepage
html_code = r'''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Research Paper Recommendation System</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f4f6fb;
            margin: 0;
            color: #20243a;
        }
        header {
            background: #202b50;
            color: white;
            padding: 30px;
            text-align: center;
        }
        main {
            max-width: 850px;
            margin: 40px auto;
            padding: 25px;
            background: white;
            border-radius: 12px;
        }
        input {
            width: 75%;
            padding: 13px;
            border: 1px solid #bbb;
            border-radius: 6px;
        }
        button {
            padding: 13px 20px;
            background: #334e9a;
            color: white;
            border: 0;
            border-radius: 6px;
            cursor: pointer;
        }
        .paper {
            padding: 15px;
            border-bottom: 1px solid #ddd;
        }
        .paper a {
            color: #334e9a;
            text-decoration: none;
            font-weight: bold;
        }
        #results { margin-top: 25px; }
    </style>
</head>
<body>
<header>
    <h1>Research Paper Recommendation System</h1>
    <p>Find academic research papers using intelligent search</p>
</header>

<main>
    <h2>Search Research Papers</h2>
    <input id="query" placeholder="Enter a paper title or keyword">
    <button onclick="searchPapers()">Search</button>
    <div id="results"></div>
</main>

<script>
async function searchPapers() {
    const query = document.getElementById("query").value.trim();
    const results = document.getElementById("results");

    if (!query) {
        results.textContent = "Please enter a search term.";
        return;
    }

    results.textContent = "Searching...";

    try {
        const response = await fetch(
            "/search?q=" + encodeURIComponent(query)
        );
        const papers = await response.json();

        results.innerHTML = "";

        if (papers.length === 0) {
            results.textContent = "No papers found.";
            return;
        }

        papers.forEach(paper => {
            const div = document.createElement("div");
            div.className = "paper";

            const link = document.createElement("a");
            link.href = "/paper/" + encodeURIComponent(paper.paper_id);
            link.textContent = paper.title || "Untitled paper";

            const info = document.createElement("p");
            info.textContent =
                "Paper ID: " + paper.paper_id +
                " | Year: " + (paper.year || "Not available");

            div.appendChild(link);
            div.appendChild(info);
            results.appendChild(div);
        });

    } catch (error) {
        results.textContent = "Search failed. Please try again.";
    }
}
</script>
</body>
</html>
'''

(TEMPLATES / "index.html").write_text(html_code, encoding="utf-8")

print("Backend updated:", (APP / "app.py").exists())
print("Homepage created:", (TEMPLATES / "index.html").exists())
print("READY FOR TESTING")

Backend updated: True
Homepage created: True
READY FOR TESTING


In [ ]:
import sys
import importlib
from pathlib import Path

APP = Path("/content/drive/MyDrive/GLAMOR_Project/application")
sys.path.insert(0, str(APP))

# Reload the updated app.py file
import app as app_module
importlib.reload(app_module)

client = app_module.app.test_client()

print("FLASK BACKEND TEST")

# Test homepage
response = client.get("/")
print("\nHomepage:", response.status_code)

# Test search
response = client.get("/search?q=fuzzy")
print("\nSearch:", response.status_code)

if response.status_code == 200:
    for paper in response.get_json()[:5]:
        print(paper)
else:
    print(response.get_data(as_text=True)[:1000])

# Test paper details
response = client.get("/paper/10002937610")
print("\nPaper details:", response.status_code)

if response.is_json:
    print(response.get_json())

FLASK BACKEND TEST

Homepage: 200

Search: 200
{'paper_id': '10000000022', 'title': 'Algorithms for the Construction of Digital Convex Fuzzy Hulls.', 'year': None}
{'paper_id': '10000000264', 'title': 'A method for selection of PID control strategy based on fuzzy behaviour modelling.', 'year': None}
{'paper_id': '10000000282', 'title': 'Fuzzy Querying: Issues and Perspectives', 'year': None}
{'paper_id': '10000000284', 'title': 'Fuzzy clustering of the self-organizing map: some applications on financial time series', 'year': None}
{'paper_id': '10000000481', 'title': 'Model of Fuzzy Comprehension Evaluation of Traffic Operations Safety on Urban Ice and Snow Road', 'year': None}

Paper details: 200
{'citation_count': 17666, 'paper_id': '10002937610', 'title': 'Distinctive Image Features from Scale-Invariant Keypoints', 'year': None}


In [ ]:
from pathlib import Path

APP = Path("/content/drive/MyDrive/GLAMOR_Project/application")

app_code = r'''
import sqlite3
import re
from pathlib import Path
from flask import Flask, request, render_template, jsonify
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

app = Flask(__name__)

DB = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/"
    "citation_knowledge_graph.sqlite"
)

def get_db():
    conn = sqlite3.connect(str(DB))
    conn.row_factory = sqlite3.Row
    return conn

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/search")
def search():
    query = request.args.get("q", "").strip()

    if not query:
        return jsonify([])

    conn = get_db()
    rows = conn.execute("""
        SELECT paper_id, title, year
        FROM papers
        WHERE title LIKE ?
        LIMIT 20
    """, (f"%{query}%",)).fetchall()
    conn.close()

    return jsonify([dict(row) for row in rows])

@app.route("/paper/<paper_id>")
def paper_details(paper_id):
    conn = get_db()

    paper = conn.execute("""
        SELECT paper_id, title, year
        FROM papers
        WHERE paper_id = ?
    """, (paper_id,)).fetchone()

    if paper is None:
        conn.close()
        return jsonify({"error": "Paper not found"}), 404

    citations = conn.execute("""
        SELECT COUNT(*)
        FROM citations
        WHERE target_id = ?
    """, (paper_id,)).fetchone()[0]

    conn.close()

    return jsonify({
        "paper_id": paper["paper_id"],
        "title": paper["title"],
        "year": paper["year"],
        "citation_count": citations
    })

@app.route("/recommend/<paper_id>")
def recommend(paper_id):
    conn = get_db()

    selected = conn.execute("""
        SELECT paper_id, title
        FROM papers
        WHERE paper_id = ?
    """, (paper_id,)).fetchone()

    if selected is None:
        conn.close()
        return jsonify({"error": "Paper not found"}), 404

    title = selected["title"] or ""
    words = list(dict.fromkeys(
        word.lower()
        for word in re.findall(r"[A-Za-z0-9]+", title)
        if len(word) > 2
    ))

    if not words:
        conn.close()
        return jsonify([])

    # Retrieve a manageable candidate set using title keywords
    conditions = " OR ".join(["title LIKE ?" for _ in words])

    rows = conn.execute(f"""
        SELECT paper_id, title
        FROM papers
        WHERE paper_id != ?
          AND ({conditions})
          AND title IS NOT NULL
          AND title != ''
        LIMIT 1000
    """, (paper_id, *[f"%{word}%" for word in words])).fetchall()

    conn.close()

    candidates = [
        dict(row) for row in rows
        if row["title"] and row["title"].strip()
    ]

    if not candidates:
        return jsonify([])

    documents = [title] + [
        item["title"] for item in candidates
    ]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=20000
    )

    vectors = vectorizer.fit_transform(documents)

    scores = cosine_similarity(
        vectors[0:1], vectors[1:]
    ).flatten()

    ranked = scores.argsort()[::-1][:10]

    recommendations = []

    for index in ranked:
        if scores[index] <= 0:
            continue

        item = candidates[index]

        recommendations.append({
            "paper_id": item["paper_id"],
            "title": item["title"],
            "similarity_score": round(float(scores[index]), 4)
        })

    return jsonify(recommendations)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

(APP / "app.py").write_text(app_code, encoding="utf-8")

print("Recommendation endpoint added:", (APP / "app.py").exists())
print("Endpoint: /recommend/<paper_id>")

Recommendation endpoint added: True
Endpoint: /recommend/<paper_id>


In [ ]:
#Test the recommendation endpoint
import sys
import importlib
from pathlib import Path

APP = Path("/content/drive/MyDrive/GLAMOR_Project/application")
sys.path.insert(0, str(APP))

# Load the updated backend
import app as app_module
importlib.reload(app_module)

client = app_module.app.test_client()

# Test recommendations for a known paper
paper_id = "10002937610"

response = client.get(f"/recommend/{paper_id}")

print("RECOMMENDATION TEST")
print("Status:", response.status_code)

if response.status_code == 200:
    recommendations = response.get_json()

    print("Number of recommendations:", len(recommendations))

    for i, paper in enumerate(recommendations, 1):
        print(f"\n{i}. {paper['title']}")
        print("Paper ID:", paper["paper_id"])
        print("Similarity:", paper["similarity_score"])
else:
    print(response.get_data(as_text=True))

RECOMMENDATION TEST
Status: 200
Number of recommendations: 10

1. Adaptive and fast scale invariant feature extraction.
Paper ID: 10000003709
Similarity: 0.2019

2. Data hiding method based on local image features
Paper ID: 10000011580
Similarity: 0.1606

3. Real-Time Scale Invariant Object Recognition Using an Artificial Neural Network.
Paper ID: 10000002837
Similarity: 0.1528

4. SURE: surface entropy for distinctive 3d features
Paper ID: 10000001569
Similarity: 0.1378

5. The Cyclic Beat Spectrum: Tempo-Related Audio Features for Time-Scale Invariant Audio Identification.
Paper ID: 10000005118
Similarity: 0.1363

6. A visual pathway for shape-based invariant classification of gray scale images
Paper ID: 10000003188
Similarity: 0.075

7. Multiscale Coarse-graining Invariant Sequences
Paper ID: 10000010762
Similarity: 0.063

8. f-divergence is a generalized invariant measure between distributions.
Paper ID: 10000017406
Similarity: 0.062

9. Occlusion Invariant Face Recognition Using T

In [ ]:
#Connect recommendations to the website
from pathlib import Path

APP = Path("/content/drive/MyDrive/GLAMOR_Project/application")
TEMPLATES = APP / "templates"

html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Research Paper Recommendation System</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f4f6fb;
    margin: 0;
    color: #20243a;
}
header {
    background: #202b50;
    color: white;
    padding: 30px;
    text-align: center;
}
main {
    max-width: 900px;
    margin: 35px auto;
    padding: 25px;
    background: white;
    border-radius: 12px;
}
input {
    width: 70%;
    padding: 13px;
    border: 1px solid #bbb;
    border-radius: 6px;
}
button {
    padding: 13px 20px;
    background: #334e9a;
    color: white;
    border: none;
    border-radius: 6px;
    cursor: pointer;
}
.paper {
    padding: 15px;
    border-bottom: 1px solid #ddd;
}
.paper button {
    margin-top: 8px;
}
h2 { color: #202b50; }
</style>
</head>
<body>

<header>
<h1>Research Paper Recommendation System</h1>
<p>Discover research papers related to your interests</p>
</header>

<main>
<h2>Search Research Papers</h2>

<input id="query" placeholder="Enter paper title or keyword">
<button onclick="searchPapers()">Search</button>

<div id="results"></div>
<div id="details"></div>
<div id="recommendations"></div>
</main>

<script>
async function searchPapers() {
    const q = document.getElementById("query").value.trim();
    const results = document.getElementById("results");

    document.getElementById("details").innerHTML = "";
    document.getElementById("recommendations").innerHTML = "";

    if (!q) {
        results.textContent = "Enter a search term.";
        return;
    }

    results.textContent = "Searching...";

    try {
        const response = await fetch(
            "/search?q=" + encodeURIComponent(q)
        );
        const papers = await response.json();

        results.innerHTML = "<h2>Search Results</h2>";

        if (!papers.length) {
            results.textContent = "No papers found.";
            return;
        }

        papers.forEach(paper => {
            const div = document.createElement("div");
            div.className = "paper";

            const title = document.createElement("strong");
            title.textContent = paper.title;

            const info = document.createElement("p");
            info.textContent = "Paper ID: " + paper.paper_id;

            const button = document.createElement("button");
            button.textContent = "View Paper and Recommendations";
            button.onclick = () => viewPaper(paper.paper_id);

            div.appendChild(title);
            div.appendChild(info);
            div.appendChild(button);
            results.appendChild(div);
        });
    } catch (error) {
        results.textContent = "Search failed.";
    }
}

async function viewPaper(id) {
    const details = document.getElementById("details");
    const recommendations =
        document.getElementById("recommendations");

    details.textContent = "Loading paper details...";
    recommendations.innerHTML = "";

    try {
        const response = await fetch("/paper/" + encodeURIComponent(id));
        const paper = await response.json();

        if (!response.ok) throw new Error("Paper not found");

        details.innerHTML = "<h2>Selected Research Paper</h2>";

        const title = document.createElement("p");
        title.textContent = paper.title;

        const info = document.createElement("p");
        info.textContent =
            "Paper ID: " + paper.paper_id +
            " | Citations: " + paper.citation_count;

        details.appendChild(title);
        details.appendChild(info);

        loadRecommendations(id);
    } catch (error) {
        details.textContent = "Unable to load paper details.";
    }
}

async function loadRecommendations(id) {
    const area = document.getElementById("recommendations");
    area.textContent = "Finding similar papers...";

    try {
        const response = await fetch(
            "/recommend/" + encodeURIComponent(id)
        );

        const papers = await response.json();

        area.innerHTML = "<h2>Top 10 Similar Research Papers</h2>";

        if (!papers.length) {
            area.textContent = "No similar papers found.";
            return;
        }

        papers.forEach((paper, index) => {
            const div = document.createElement("div");
            div.className = "paper";

            const title = document.createElement("strong");
            title.textContent = (index + 1) + ". " + paper.title;

            const info = document.createElement("p");
            info.textContent =
                "Paper ID: " + paper.paper_id +
                " | Similarity: " + paper.similarity_score;

            div.appendChild(title);
            div.appendChild(info);
            area.appendChild(div);
        });
    } catch (error) {
        area.textContent = "Unable to load recommendations.";
    }
}
</script>

</body>
</html>
'''

(TEMPLATES / "index.html").write_text(html, encoding="utf-8")

print("Website updated:", (TEMPLATES / "index.html").exists())
print("Search interface: READY")
print("Paper details interface: READY")
print("Recommendation interface: READY")

Website updated: True
Search interface: READY
Paper details interface: READY
Recommendation interface: READY


In [ ]:
import sys
import importlib
from pathlib import Path

APP = Path("/content/drive/MyDrive/GLAMOR_Project/application")
sys.path.insert(0, str(APP))

import app as app_module
importlib.reload(app_module)

client = app_module.app.test_client()

print("COMPLETE WEBSITE TEST")

# 1. Homepage
home = client.get("/")
print("1. Homepage:", home.status_code)

# 2. Search
search = client.get("/search?q=fuzzy")
print("2. Search:", search.status_code)

papers = search.get_json()

if not papers:
    raise RuntimeError("No search results found.")

print("Search results:", len(papers))

# 3. Paper details
paper_id = papers[0]["paper_id"]
details = client.get(f"/paper/{paper_id}")
print("3. Paper details:", details.status_code)

# 4. Recommendations
recommendations = client.get(f"/recommend/{paper_id}")
print("4. Recommendations:", recommendations.status_code)

if recommendations.status_code == 200:
    recs = recommendations.get_json()
    print("Recommendations returned:", len(recs))

    for i, item in enumerate(recs[:5], 1):
        print(f"{i}. {item['title']}")

print("\nWEBSITE WORKFLOW TEST FINISHED")

COMPLETE WEBSITE TEST
1. Homepage: 200
2. Search: 200
Search results: 20
3. Paper details: 200
4. Recommendations: 200
Recommendations returned: 10
1. An Efficient Algorithm for the Generation of Z-Convex Polyominoes
2. Prefix Table Construction and Conversion
3. O(1) Time Algorithm on BSR for Computing Convex Hull.
4. Automatic pipeline construction for real-time annotation
5. Fuzzy Grading: Fuzzy Logic for Uncertainty Management of Linguistic Evaluations.

WEBSITE WORKFLOW TEST FINISHED


In [ ]:
#website
import sys
import threading
from pathlib import Path
from werkzeug.serving import make_server
from IPython.display import display, HTML
from google.colab.output import eval_js

APP = Path("/content/drive/MyDrive/GLAMOR_Project/application")
sys.path.insert(0, str(APP))

import app as app_module

# Start Flask server
server = make_server(
    "0.0.0.0",
    5000,
    app_module.app
)

thread = threading.Thread(
    target=server.serve_forever,
    daemon=True
)

thread.start()

# Create Colab browser link
url = eval_js("google.colab.kernel.proxyPort(5000)")

print("WEBSITE SERVER STARTED")
print("Website URL:", url)

display(HTML(
    f'<h3><a href="{url}" target="_blank">'
    'OPEN RESEARCH PAPER RECOMMENDATION SYSTEM</a></h3>'
))

WEBSITE SERVER STARTED
Website URL: https://5000-m-s-use1d10-4kg8tkn486h0-d.us-east1.prod.r.colab.dev


In [ ]:
from pathlib import Path
import os

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
OUTPUTS = PROJECT / "outputs"

folders = [
    OUTPUTS / "glamor_pilot",
    OUTPUTS / "glamor_pilot" / "language_model_compat",
    OUTPUTS / "glamor_pilot" / "cpu_small_pilot" / "trained_model"
]

print("GLAMOR TRAINING FILE CHECK\n")

for folder in folders:
    print("\nFOLDER:", folder)
    print("Exists:", folder.exists())

    if folder.exists():
        for item in sorted(folder.iterdir()):
            if item.is_file():
                size_mb = item.stat().st_size / (1024 * 1024)
                print(f"  {item.name}: {size_mb:.2f} MB")
            elif item.is_dir():
                print(f"  {item.name}/")

# Check pilot corpus
for filename in ["walks_train.txt", "walks_validation.txt"]:
    matches = list(OUTPUTS.rglob(filename))

    print("\n", filename)

    if matches:
        for path in matches:
            print("Path:", path)
            print("Size:", round(path.stat().st_size / (1024 * 1024), 2), "MB")
            with path.open("r", encoding="utf-8") as f:
                print("Walks:", sum(1 for _ in f))
    else:
        print("NOT FOUND")

print("\nTRAINING FILE CHECK FINISHED")

GLAMOR TRAINING FILE CHECK


FOLDER: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot
Exists: True
  cpu_small_pilot/
  walks_train.txt: 0.16 MB
  walks_validation.txt: 0.03 MB

FOLDER: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/language_model_compat
Exists: False

FOLDER: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/cpu_small_pilot/trained_model
Exists: True
  config.json: 0.00 MB
  merges.txt: 0.44 MB
  model.safetensors: 313.47 MB
  special_tokens_map.json: 0.00 MB
  tokenizer.json: 3.39 MB
  tokenizer_config.json: 0.00 MB
  training_args.bin: 0.01 MB
  vocab.json: 0.76 MB

 walks_train.txt
Path: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/walks_train.txt
Size: 0.16 MB
Walks: 850

 walks_validation.txt
Path: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_pilot/walks_validation.txt
Size: 0.03 MB
Walks: 150

TRAINING FILE CHECK FINISHED


In [ ]:
import sqlite3

DB = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(DB)

print("FULL GRAPH TRAINING CHECK\n")

print("Papers:", conn.execute(
    "SELECT COUNT(*) FROM papers"
).fetchone()[0])

print("Citation links:", conn.execute(
    "SELECT COUNT(*) FROM citations"
).fetchone()[0])

print("Papers with outgoing citations:", conn.execute(
    "SELECT COUNT(DISTINCT source_id) FROM citations"
).fetchone()[0])

print("Papers cited by others:", conn.execute(
    "SELECT COUNT(DISTINCT target_id) FROM citations"
).fetchone()[0])

print("\nCITATION TABLE INDEXES:")

for row in conn.execute(
    "PRAGMA index_list(citations)"
):
    print(row)

print("\nSAMPLE CITATION LINKS:")

for row in conn.execute("""
    SELECT source_id, target_id
    FROM citations
    LIMIT 5
"""):
    print(row)

conn.close()

print("\nFULL GRAPH CHECK COMPLETED")

FULL GRAPH TRAINING CHECK

Papers: 3687412
Citation links: 26207545
Papers with outgoing citations: 2809282
Papers cited by others: 2204815

CITATION TABLE INDEXES:
(0, 'idx_citations_target', 0, 'c', 0)
(1, 'sqlite_autoindex_citations_1', 1, 'pk', 0)

SAMPLE CITATION LINKS:
('10000000000', '10001185423')
('10000000000', '10001302680')
('10000000002', '10000442607')
('10000000002', '10001069614')
('10000000002', '10001469932')

FULL GRAPH CHECK COMPLETED


In [ ]:
import sqlite3
import random
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
DB = PROJECT / "outputs/citation_knowledge_graph.sqlite"
OUT = PROJECT / "outputs/glamor_large_training"

OUT.mkdir(parents=True, exist_ok=True)

TOTAL_WALKS = 10000
MAX_STEPS = 20
SEED = 42

rng = random.Random(SEED)

conn = sqlite3.connect(str(DB))
conn.execute("PRAGMA busy_timeout=60000")

print("Preparing citation graph index...")

conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_citations_source
    ON citations(source_id)
""")
conn.commit()

max_rowid = conn.execute(
    "SELECT MAX(rowid) FROM citations"
).fetchone()[0]

if not max_rowid:
    conn.close()
    raise RuntimeError("No citation links found.")

def random_start():
    position = rng.randint(1, max_rowid)

    row = conn.execute("""
        SELECT source_id
        FROM citations
        WHERE rowid >= ?
        LIMIT 1
    """, (position,)).fetchone()

    if row is None:
        row = conn.execute("""
            SELECT source_id
            FROM citations
            LIMIT 1
        """).fetchone()

    return row[0] if row else None

def make_walk(start):
    walk = [start]
    current = start

    for _ in range(MAX_STEPS):
        outgoing = conn.execute("""
            SELECT target_id
            FROM citations
            WHERE source_id = ?
        """, (current,)).fetchall()

        if not outgoing:
            break

        if rng.random() < 0.05:
            break

        current = rng.choice(outgoing)[0]
        walk.append(current)

    return walk

raw_file = OUT / "citation_walks_10000.txt"
walks = []
attempts = 0
max_attempts = TOTAL_WALKS * 5

print("Generating citation walks...")

with raw_file.open("w", encoding="utf-8") as f:
    while len(walks) < TOTAL_WALKS and attempts < max_attempts:
        attempts += 1

        start = random_start()
        if not start:
            continue

        walk = make_walk(start)

        if len(walk) < 2:
            continue

        marks = ",".join("?" for _ in set(walk))

        rows = conn.execute(
            f"""
            SELECT paper_id, title
            FROM papers
            WHERE paper_id IN ({marks})
            """,
            list(set(walk))
        ).fetchall()

        title_map = {
            paper_id: title or "Untitled paper"
            for paper_id, title in rows
        }

        if any(pid not in title_map for pid in walk):
            continue

        # Convert citation relationships into training text
        parts = []

        for source, target in zip(walk, walk[1:]):
            parts.append(
                f"A research paper titled "
                f"{title_map[source]} cites a research paper titled "
                f"{title_map[target]}."
            )

        text = " ".join(parts)
        f.write(text + "\n")
        walks.append(text)

        if len(walks) % 500 == 0:
            f.flush()
            print("Walks generated:", len(walks))

conn.close()

print("\nGENERATION FINISHED")
print("Walks generated:", len(walks))
print("Attempts:", attempts)
print("Training corpus:", raw_file)
print("File size (MB):", round(raw_file.stat().st_size / (1024 * 1024), 2))

if len(walks) < TOTAL_WALKS:
    raise RuntimeError(
        "Fewer than 10,000 walks generated. "
        "Check the output before proceeding."
    )

# Create separate training and validation files
rng.shuffle(walks)
boundary = int(len(walks) * 0.9)

train_file = OUT / "walks_train_10000.txt"
valid_file = OUT / "walks_validation_10000.txt"

train_file.write_text(
    "\n".join(walks[:boundary]) + "\n",
    encoding="utf-8"
)

valid_file.write_text(
    "\n".join(walks[boundary:]) + "\n",
    encoding="utf-8"
)

print("\nCORPUS SPLIT")
print("Training walks:", boundary)
print("Validation walks:", len(walks) - boundary)
print("Training file:", train_file)
print("Validation file:", valid_file)

Preparing citation graph index...
Generating citation walks...
Walks generated: 500
Walks generated: 1000
Walks generated: 1500
Walks generated: 2000
Walks generated: 2500
Walks generated: 3000
Walks generated: 3500
Walks generated: 4000
Walks generated: 4500
Walks generated: 5000
Walks generated: 5500
Walks generated: 6000
Walks generated: 6500
Walks generated: 7000
Walks generated: 7500
Walks generated: 8000
Walks generated: 8500
Walks generated: 9000
Walks generated: 9500
Walks generated: 10000

GENERATION FINISHED
Walks generated: 10000
Attempts: 10558
Training corpus: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_large_training/citation_walks_10000.txt
File size (MB): 9.02

CORPUS SPLIT
Training walks: 9000
Validation walks: 1000
Training file: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_large_training/walks_train_10000.txt
Validation file: /content/drive/MyDrive/GLAMOR_Project/outputs/glamor_large_training/walks_validation_10000.txt


In [ ]:
import sys
import inspect
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/GLAMOR_Project")
REPO = PROJECT / "GLAMOR"

sys.path.insert(0, str(REPO / "src"))

from citation_models.glamor import language_model

print("TRAINING FUNCTION CHECK\n")

print("Function available:",
      hasattr(language_model, "fine_tune_graph_language_model"))

if hasattr(language_model, "fine_tune_graph_language_model"):
    function = language_model.fine_tune_graph_language_model

    print("\nFUNCTION PARAMETERS:")
    print(inspect.signature(function))

print("\nTRAINING FILES:")

for name in [
    "walks_train_10000.txt",
    "walks_validation_10000.txt"
]:
    path = PROJECT / "outputs/glamor_large_training" / name
    print(name, ":", path.exists())

print("\nPILOT MODEL:")
model = PROJECT / "outputs/glamor_pilot/cpu_small_pilot/trained_model"
print("Exists:", model.exists())

print("\nTRAINING PREPARATION CHECK COMPLETE")

TRAINING FUNCTION CHECK

Function available: True

FUNCTION PARAMETERS:
(train_corpus: 'str | Path', validation_corpus: 'str | Path', output_dir: 'str | Path', *, model_name: 'str' = 'distilroberta-base', mlm_probability: 'float' = 0.15, batch_size: 'int' = 16, epochs: 'int' = 6, learning_rate: 'float' = 0.0005, weight_decay: 'float' = 0.01, gradient_accumulation_steps: 'int' = 20, max_length: 'int' = 512, fp16: 'bool | None' = None, dataloader_num_workers: 'int' = 8, gradient_checkpointing: 'bool' = False, resume_from_checkpoint: 'bool | str | Path' = False, seed: 'int' = 42) -> 'dict[str, Any]'

TRAINING FILES:
walks_train_10000.txt : True
walks_validation_10000.txt : True

PILOT MODEL:
Exists: True

TRAINING PREPARATION CHECK COMPLETE


In [ ]:
import sys
import inspect
from pathlib import Path

REPO = Path("/content/drive/MyDrive/GLAMOR_Project/GLAMOR")
sys.path.insert(0, str(REPO / "src"))

from citation_models.glamor.language_model import (
    fine_tune_graph_language_model
)

signature = inspect.signature(fine_tune_graph_language_model)

print("GLAMOR TRAINING PARAMETERS\n")

for name, parameter in signature.parameters.items():
    print(f"{name}: {parameter.default}")

print("\nTRANSFORMERS VERSION:")

import transformers
print(transformers.__version__)

print("\nPYTORCH VERSION:")

import torch
print(torch.__version__)

print("\nCUDA AVAILABLE:", torch.cuda.is_available())

GLAMOR TRAINING PARAMETERS

train_corpus: <class 'inspect._empty'>
validation_corpus: <class 'inspect._empty'>
output_dir: <class 'inspect._empty'>
model_name: distilroberta-base
mlm_probability: 0.15
batch_size: 16
epochs: 6
learning_rate: 0.0005
weight_decay: 0.01
gradient_accumulation_steps: 20
max_length: 512
fp16: None
dataloader_num_workers: 8
gradient_checkpointing: False
resume_from_checkpoint: False
seed: 42

TRANSFORMERS VERSION:
4.57.1

PYTORCH VERSION:
2.11.0+cpu

CUDA AVAILABLE: False


In [ ]:

# ============================================================
# GLAMOR FAST CPU EXPERIMENT - RESTORE TRAINING ARGUMENTS
# ============================================================

import os
import sys
import time
import json
import transformers

from google.colab import drive
drive.mount('/content/drive')

# 1. PROJECT PATHS
ROOT = "/content/drive/MyDrive/GLAMOR_Project"
REPO = os.path.join(ROOT, "GLAMOR")
FAST_DIR = os.path.join(ROOT, "outputs/cpu_fast_experiment")

FAST_TRAIN = os.path.join(FAST_DIR, "train_500.txt")
FAST_VALID = os.path.join(FAST_DIR, "validation_100.txt")
MODEL_DIR = os.path.join(FAST_DIR, "trained_model")

# 2. VERIFY FILES
assert os.path.isfile(FAST_TRAIN), "Training file is missing."
assert os.path.isfile(FAST_VALID), "Validation file is missing."

# 3. RESTORE THE REAL TRANSFORMERS CLASS
from transformers.training_args import TrainingArguments as RealTrainingArguments
from transformers.trainer_utils import IntervalStrategy
from transformers.trainer_utils import SaveStrategy

# Undo the earlier function replacement
transformers.TrainingArguments = RealTrainingArguments

# 4. LOAD GLAMOR
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import citation_models.glamor.language_model as lm

# Restore the class in the GLAMOR module too
lm.TrainingArguments = RealTrainingArguments

# 5. PATCH THE ORIGINAL CLASS VALIDATION
original_post_init = RealTrainingArguments.__post_init__

def corrected_post_init(self):
    # Use matching evaluation and saving strategies
    self.load_best_model_at_end = False
    self.eval_strategy = IntervalStrategy.EPOCH
    self.save_strategy = SaveStrategy.EPOCH

    # Continue with the original Transformers validation
    original_post_init(self)

RealTrainingArguments.__post_init__ = corrected_post_init

print("TrainingArguments restored:", isinstance(
    transformers.TrainingArguments, type
))
print("Transformers version:", transformers.__version__)
print("Training and validation files verified.")

# 6. START TRAINING
start_time = time.time()

result = lm.fine_tune_graph_language_model(
    train_corpus=FAST_TRAIN,
    validation_corpus=FAST_VALID,
    output_dir=MODEL_DIR,
    model_name="distilroberta-base",
    mlm_probability=0.15,
    batch_size=2,
    epochs=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    gradient_accumulation_steps=1,
    max_length=64,
    fp16=False,
    dataloader_num_workers=0,
    gradient_checkpointing=False,
    seed=1203
)

elapsed = time.time() - start_time

# 7. SAVE TRAINING REPORT
report = {
    "training_walks": 500,
    "validation_walks": 100,
    "epochs": 1,
    "batch_size": 2,
    "max_length": 64,
    "training_time_minutes": round(elapsed / 60, 2),
    "model_directory": MODEL_DIR
}

with open(os.path.join(FAST_DIR, "training_result.json"), "w") as file:
    json.dump(report, file, indent=4)

print("\n========== TRAINING COMPLETED ==========")
print("Training time:", round(elapsed / 60, 2), "minutes")
print("Model saved at:", MODEL_DIR)
print("Training report saved.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TrainingArguments restored: True
Transformers version: 4.57.1
Training and validation files verified.


Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Epoch,Training Loss,Validation Loss
1,No log,1.385100


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



========== TRAINING COMPLETED ==========
Training time: 9.2 minutes
Model saved at: /content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model
Training report saved.


In [ ]:

import os
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_DIR = "/content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model"

print("Checking saved model...")

assert os.path.isdir(MODEL_DIR), "Model folder not found."

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForMaskedLM.from_pretrained(MODEL_DIR)

model.eval()

text = "A research paper about machine learning uses <mask> methods."

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

mask_position = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

if len(mask_position) == 0:
    raise ValueError("Mask token was not found.")

predictions = outputs.logits[0, mask_position[0]].topk(5).indices

print("\nTop 5 predicted words:")
for token_id in predictions:
    print(tokenizer.decode([token_id.item()]))

print("\nSUCCESS: Saved GLAMOR model loaded and generated predictions.")


Checking saved model...

Top 5 predicted words:
 two
 computational
 statistical
 several
 mathematical

SUCCESS: Saved GLAMOR model loaded and generated predictions.


In [ ]:
#Evaluate recommendation performance

import os
import sqlite3

DB_PATH = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "outputs/citation_knowledge_graph.sqlite"
)

assert os.path.exists(DB_PATH), "SQLite database not found."

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Inspect database tables
cursor.execute("""
    SELECT name FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""")

print("DATABASE TABLES:")
for row in cursor.fetchall():
    print("-", row[0])

# Inspect papers table columns
cursor.execute("PRAGMA table_info(papers)")
columns = cursor.fetchall()

print("\nPAPERS TABLE COLUMNS:")
for column in columns:
    print(column)

# Check how many titles are missing
all_ids = list(dict.fromkeys(list(new_queries) + list(candidate_list)))

print("\nTotal unique paper IDs:", len(all_ids))
print("Titles currently available:", sum(
    bool(titles.get(pid) or titles.get(str(pid)))
    for pid in all_ids
))
print("Titles missing:", sum(
    not bool(titles.get(pid) or titles.get(str(pid)))
    for pid in all_ids
))

conn.close()


DATABASE TABLES:
- citations
- evaluation_progress
- papers
- papers_fts
- papers_fts_config
- papers_fts_content
- papers_fts_data
- papers_fts_docsize
- papers_fts_idx
- progress
- test_ids
- test_queries
- test_relevance
- training_ids

PAPERS TABLE COLUMNS:
(0, 'paper_id', 'TEXT', 0, None, 1)
(1, 'title', 'TEXT', 0, None, 0)
(2, 'year', 'TEXT', 0, None, 0)

Total unique paper IDs: 5966
Titles currently available: 5466
Titles missing: 500


In [ ]:

# ============================================================
# GLAMOR FAST MODEL - CORRECTED 500 QUERY EVALUATION
# Uses query ID/title tuples and SQLite database
# ============================================================

import os
import sqlite3
import numpy as np
import torch
import json
from transformers import AutoTokenizer, AutoModel

# 1. PATHS
ROOT = "/content/drive/MyDrive/GLAMOR_Project"

DB_PATH = os.path.join(
    ROOT, "outputs/citation_knowledge_graph.sqlite"
)

MODEL_DIR = os.path.join(
    ROOT, "outputs/cpu_fast_experiment/trained_model"
)

OUTPUT_FILE = os.path.join(
    ROOT, "outputs/glamor_fast_500_query_evaluation.txt"
)

# 2. CHECK INPUTS
assert os.path.exists(DB_PATH), "SQLite database missing."
assert os.path.exists(MODEL_DIR), "Trained model missing."

assert "new_queries" in globals(), "new_queries is missing."
assert "candidate_list" in globals(), "candidate_list is missing."

# Correctly extract query IDs and titles from tuples
query_pairs = []

for item in new_queries:
    if isinstance(item, (tuple, list)) and len(item) >= 2:
        query_pairs.append((str(item[0]), str(item[1])))
    else:
        raise ValueError(
            "Expected each query to contain a paper ID and title."
        )

candidate_ids = list(dict.fromkeys(str(x) for x in candidate_list))

print("Queries:", len(query_pairs))
print("Candidates:", len(candidate_ids))

# 3. RETRIEVE CANDIDATE TITLES AND RELEVANCE FROM SQLITE
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

all_paper_ids = list(dict.fromkeys(
    [pid for pid, title in query_pairs] + candidate_ids
))

def fetch_in_chunks(sql_prefix, ids, chunk_size=800):
    rows = []
    for start in range(0, len(ids), chunk_size):
        chunk = ids[start:start + chunk_size]
        placeholders = ",".join("?" for _ in chunk)
        cur.execute(
            sql_prefix.format(placeholders=placeholders),
            chunk
        )
        rows.extend(cur.fetchall())
    return rows

paper_rows = fetch_in_chunks(
    "SELECT paper_id, title FROM papers "
    "WHERE paper_id IN ({placeholders})",
    all_paper_ids
)

title_map = {str(pid): title for pid, title in paper_rows}

# Query titles come directly from test_queries tuples
query_title_map = {
    pid: title for pid, title in query_pairs
}

# Candidate titles come from the papers table
missing_candidates = [
    pid for pid in candidate_ids
    if not title_map.get(pid) or not title_map[pid].strip()
]

if missing_candidates:
    raise ValueError(
        f"{len(missing_candidates)} candidate titles are missing. "
        "No candidates have been silently removed."
    )

query_ids = [pid for pid, title in query_pairs]

# Retrieve relevant paper IDs for all 500 queries
relevance_rows = fetch_in_chunks(
    "SELECT test_paper_id, relevant_paper_id "
    "FROM test_relevance "
    "WHERE test_paper_id IN ({placeholders})",
    query_ids
)

relevance_map = {pid: set() for pid in query_ids}

for query_id, relevant_id in relevance_rows:
    relevance_map[str(query_id)].add(str(relevant_id))

print("Candidate titles retrieved:", len(candidate_ids))
print("Query titles retrieved:", len(query_pairs))
print("Relevance records retrieved:", len(relevance_rows))

conn.close()

# 4. LOAD TRAINED MODEL
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModel.from_pretrained(MODEL_DIR)
model.eval()
model.to("cpu")

# 5. CREATE TITLE EMBEDDINGS
all_ids = list(dict.fromkeys(query_ids + candidate_ids))

all_texts = [
    query_title_map[pid] if pid in query_title_map
    else title_map[pid]
    for pid in all_ids
]

if any(not text.strip() for text in all_texts):
    raise ValueError("An empty title was found.")

def encode_texts(texts, batch_size=32):
    vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        with torch.no_grad():
            output = model(**inputs)
            hidden = output.last_hidden_state
            mask = inputs["attention_mask"].unsqueeze(-1)

            pooled = (hidden * mask).sum(dim=1)
            pooled = pooled / mask.sum(dim=1).clamp(min=1)

            pooled = torch.nn.functional.normalize(
                pooled, p=2, dim=1
            )

        vectors.append(pooled.cpu())

        print(
            "Encoded:",
            min(start + batch_size, len(texts)),
            "/",
            len(texts)
        )

    return torch.cat(vectors, dim=0)

print("\nGenerating paper embeddings...")
vectors = encode_texts(all_texts)

id_to_vector = {
    pid: vectors[i] for i, pid in enumerate(all_ids)
}

candidate_vectors = torch.stack([
    id_to_vector[pid] for pid in candidate_ids
])

# 6. CALCULATE RECOMMENDATION METRICS
def calculate_metrics(ranked_ids, relevant_ids, k=10):
    relevant_ids = set(relevant_ids)
    top_ids = ranked_ids[:k]

    hits = [
        1 if pid in relevant_ids else 0
        for pid in top_ids
    ]

    precision = sum(hits) / k

    recall = (
        sum(hits) / len(relevant_ids)
        if relevant_ids else 0
    )

    hit_count = 0
    ap_sum = 0

    for rank, hit in enumerate(hits, start=1):
        if hit:
            hit_count += 1
            ap_sum += hit_count / rank

    denominator = min(len(relevant_ids), k)
    average_precision = (
        ap_sum / denominator if denominator else 0
    )

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, start=1)
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, min(len(relevant_ids), k) + 1)
    )

    ndcg = dcg / idcg if idcg else 0
    hit_rate = 1 if sum(hits) > 0 else 0

    return precision, recall, average_precision, ndcg, hit_rate

# 7. EVALUATE ALL QUERIES
results = []

print("\nEvaluating all queries...")

for index, (query_id, query_title) in enumerate(query_pairs):

    query_vector = id_to_vector[query_id]

    scores = torch.matmul(
        candidate_vectors, query_vector
    ).numpy()

    # Do not recommend the query paper itself
    valid_indices = [
        i for i, pid in enumerate(candidate_ids)
        if pid != query_id
    ]

    ranked_indices = sorted(
        valid_indices,
        key=lambda i: scores[i],
        reverse=True
    )[:10]

    ranked_ids = [
        candidate_ids[i] for i in ranked_indices
    ]

    metrics = calculate_metrics(
        ranked_ids,
        relevance_map[query_id],
        k=10
    )

    results.append(metrics)

    if (index + 1) % 50 == 0:
        print("Evaluated:", index + 1, "/", len(query_pairs))

# 8. FINAL METRICS
average = np.mean(results, axis=0)

metric_names = [
    "Precision@10",
    "Recall@10",
    "MAP@10",
    "nDCG@10",
    "Hit Rate@10"
]

report = [
    "GLAMOR FAST MODEL - 500 QUERY EVALUATION",
    "==========================================",
    f"Queries evaluated: {len(query_pairs)}",
    f"Candidates available: {len(candidate_ids)}",
    f"Relevance records: {len(relevance_rows)}",
    ""
]

for name, value in zip(metric_names, average):
    report.append(
        f"{name}: {value:.4f} ({value * 100:.2f}%)"
    )

report_text = "\n".join(report)

print("\n" + report_text)

# 9. SAVE REPORT
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(report_text)

print("\nEvaluation report saved:")
print(OUTPUT_FILE)


Queries: 500
Candidates: 5466
Candidate titles retrieved: 5466
Query titles retrieved: 500
Relevance records retrieved: 4577


Some weights of RobertaModel were not initialized from the model checkpoint at /content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Generating paper embeddings...
Encoded: 32 / 5966
Encoded: 64 / 5966
Encoded: 96 / 5966
Encoded: 128 / 5966
Encoded: 160 / 5966
Encoded: 192 / 5966
Encoded: 224 / 5966
Encoded: 256 / 5966
Encoded: 288 / 5966
Encoded: 320 / 5966
Encoded: 352 / 5966
Encoded: 384 / 5966
Encoded: 416 / 5966
Encoded: 448 / 5966
Encoded: 480 / 5966
Encoded: 512 / 5966
Encoded: 544 / 5966
Encoded: 576 / 5966
Encoded: 608 / 5966
Encoded: 640 / 5966
Encoded: 672 / 5966
Encoded: 704 / 5966
Encoded: 736 / 5966
Encoded: 768 / 5966
Encoded: 800 / 5966
Encoded: 832 / 5966
Encoded: 864 / 5966
Encoded: 896 / 5966
Encoded: 928 / 5966
Encoded: 960 / 5966
Encoded: 992 / 5966
Encoded: 1024 / 5966
Encoded: 1056 / 5966
Encoded: 1088 / 5966
Encoded: 1120 / 5966
Encoded: 1152 / 5966
Encoded: 1184 / 5966
Encoded: 1216 / 5966
Encoded: 1248 / 5966
Encoded: 1280 / 5966
Encoded: 1312 / 5966
Encoded: 1344 / 5966
Encoded: 1376 / 5966
Encoded: 1408 / 5966
Encoded: 1440 / 5966
Encoded: 1472 / 5966
Encoded: 1504 / 5966
Encoded: 1536 /

In [ ]:

# ============================================================
# GLAMOR + TF-IDF HYBRID RECOMMENDATION EXPERIMENT
# Same 500 queries and 5,466 candidate papers
# ============================================================

import numpy as np
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. CHECK EXISTING EVALUATION DATA
required = [
    "query_pairs", "query_ids", "candidate_ids",
    "query_title_map", "title_map",
    "query_vectors", "candidate_vectors", "relevance_map"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Required evaluation data is missing: "
        + ", ".join(missing)
        + ". Do not retrain. Tell me this error."
    )

print("Queries:", len(query_ids))
print("Candidates:", len(candidate_ids))

# 2. PREPARE TITLES
query_texts = [query_title_map[pid] for pid in query_ids]
candidate_texts = [title_map[pid] for pid in candidate_ids]

# 3. CREATE TF-IDF VECTORS
print("\nCreating TF-IDF vectors...")

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=1
)

candidate_tfidf = vectorizer.fit_transform(candidate_texts)
query_tfidf = vectorizer.transform(query_texts)

# 4. CALCULATE TF-IDF SIMILARITY
print("Calculating TF-IDF similarities...")

tfidf_scores = cosine_similarity(
    query_tfidf, candidate_tfidf
)

# 5. GET GLAMOR SIMILARITY
print("Calculating GLAMOR similarities...")

glamor_scores = (
    query_vectors @ candidate_vectors.T
).cpu().numpy()

# 6. NORMALIZE SCORES FOR EACH QUERY
def normalize_rows(scores):
    minimum = scores.min(axis=1, keepdims=True)
    maximum = scores.max(axis=1, keepdims=True)

    return (scores - minimum) / np.maximum(
        maximum - minimum, 1e-12
    )

tfidf_normalized = normalize_rows(tfidf_scores)
glamor_normalized = normalize_rows(glamor_scores)

# 7. METRIC CALCULATION
def calculate_metrics(ranked_ids, relevant_ids, k=10):
    relevant_ids = set(relevant_ids)
    top_ids = ranked_ids[:k]

    hits = [int(pid in relevant_ids) for pid in top_ids]

    precision = sum(hits) / k
    recall = sum(hits) / len(relevant_ids) if relevant_ids else 0

    hit_count = 0
    ap_sum = 0

    for rank, hit in enumerate(hits, start=1):
        if hit:
            hit_count += 1
            ap_sum += hit_count / rank

    denominator = min(len(relevant_ids), k)
    ap = ap_sum / denominator if denominator else 0

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, start=1)
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, min(len(relevant_ids), k) + 1)
    )

    ndcg = dcg / idcg if idcg else 0
    hit_rate = int(sum(hits) > 0)

    return precision, recall, ap, ndcg, hit_rate

# 8. TEST FIVE WEIGHT COMBINATIONS
weights = [0, 0.25, 0.50, 0.75, 1.0]
experiment_results = []

print("\nTesting hybrid combinations...")

for glamor_weight in weights:

    tfidf_weight = 1.0 - glamor_weight

    combined_scores = (
        tfidf_weight * tfidf_normalized
        + glamor_weight * glamor_normalized
    )

    query_metrics = []

    for i, query_id in enumerate(query_ids):

        # Exclude the query paper itself
        valid_indices = [
            j for j, pid in enumerate(candidate_ids)
            if pid != query_id
        ]

        ranked_indices = sorted(
            valid_indices,
            key=lambda j: combined_scores[i, j],
            reverse=True
        )[:10]

        ranked_ids = [
            candidate_ids[j] for j in ranked_indices
        ]

        metrics = calculate_metrics(
            ranked_ids,
            relevance_map[query_id],
            k=10
        )

        query_metrics.append(metrics)

    average = np.mean(query_metrics, axis=0)

    experiment_results.append({
        "TF-IDF Weight": tfidf_weight,
        "GLAMOR Weight": glamor_weight,
        "Precision@10": average[0],
        "Recall@10": average[1],
        "MAP@10": average[2],
        "nDCG@10": average[3],
        "Hit Rate@10": average[4]
    })

    print(
        f"TF-IDF {tfidf_weight:.0%} | "
        f"GLAMOR {glamor_weight:.0%} | "
        f"Hit Rate: {average[4] * 100:.2f}% | "
        f"nDCG: {average[3] * 100:.2f}%"
    )

# 9. DISPLAY AND SAVE RESULTS
results_df = pd.DataFrame(experiment_results)

print("\n========== HYBRID COMPARISON ==========")
print(results_df.round(4).to_string(index=False))

best_index = results_df["nDCG@10"].idxmax()
best_result = results_df.loc[best_index]

print("\nBEST RESULT BY nDCG@10")
print(best_result.to_string())

OUTPUT_DIR = (
    "/content/drive/MyDrive/GLAMOR_Project/outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(
    OUTPUT_DIR, "hybrid_recommendation_comparison.csv"
)

txt_path = os.path.join(
    OUTPUT_DIR, "hybrid_recommendation_comparison.txt"
)

results_df.to_csv(csv_path, index=False)

with open(txt_path, "w", encoding="utf-8") as f:
    f.write(results_df.to_string(index=False))
    f.write("\n\nBest result by nDCG@10:\n")
    f.write(best_result.to_string())

print("\nResults saved:")
print(csv_path)
print(txt_path)


Queries: 500
Candidates: 5466

Creating TF-IDF vectors...
Calculating TF-IDF similarities...
Calculating GLAMOR similarities...

Testing hybrid combinations...
TF-IDF 100% | GLAMOR 0% | Hit Rate: 79.00% | nDCG: 38.83%
TF-IDF 75% | GLAMOR 25% | Hit Rate: 79.60% | nDCG: 39.27%
TF-IDF 50% | GLAMOR 50% | Hit Rate: 79.40% | nDCG: 38.47%
TF-IDF 25% | GLAMOR 75% | Hit Rate: 74.80% | nDCG: 31.91%
TF-IDF 0% | GLAMOR 100% | Hit Rate: 37.80% | nDCG: 10.05%

========== HYBRID COMPARISON ==========
 TF-IDF Weight  GLAMOR Weight  Precision@10  Recall@10  MAP@10  nDCG@10  Hit Rate@10
          1.00           0.00        0.2354     0.3011  0.2767   0.3883        0.790
          0.75           0.25        0.2382     0.3035  0.2816   0.3927        0.796
          0.50           0.50        0.2302     0.2964  0.2735   0.3847        0.794
          0.25           0.75        0.1830     0.2420  0.2129   0.3191        0.748
          0.00           1.00        0.0586     0.0787  0.0545   0.1005        0.378

In [ ]:

# ============================================================
# HYBRID WEIGHT FINE-TUNING EXPERIMENT
# ============================================================

import numpy as np
import pandas as pd
import os

# 1. VERIFY EXISTING DATA
required = [
    "query_ids", "candidate_ids",
    "tfidf_normalized", "glamor_normalized",
    "relevance_map"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Required variables missing: " + ", ".join(missing)
    )

assert tfidf_normalized.shape == glamor_normalized.shape
assert tfidf_normalized.shape[0] == len(query_ids)
assert tfidf_normalized.shape[1] == len(candidate_ids)

# 2. METRIC FUNCTION
def calculate_metrics(ranked_ids, relevant_ids, k=10):
    relevant_ids = set(relevant_ids)
    top_ids = ranked_ids[:k]

    hits = [int(pid in relevant_ids) for pid in top_ids]

    precision = sum(hits) / k
    recall = sum(hits) / len(relevant_ids) if relevant_ids else 0

    hit_count = 0
    ap_sum = 0

    for rank, hit in enumerate(hits, start=1):
        if hit:
            hit_count += 1
            ap_sum += hit_count / rank

    denominator = min(len(relevant_ids), k)
    ap = ap_sum / denominator if denominator else 0

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, start=1)
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, min(len(relevant_ids), k) + 1)
    )

    ndcg = dcg / idcg if idcg else 0
    hit_rate = int(sum(hits) > 0)

    return precision, recall, ap, ndcg, hit_rate

# 3. TEST NEARBY WEIGHTS
glamor_weights = [0.20, 0.225, 0.25, 0.275, 0.30]

fine_tuning_results = []

print("Starting hybrid weight fine-tuning...")

for gw in glamor_weights:
    tw = 1.0 - gw

    combined = (
        tw * tfidf_normalized
        + gw * glamor_normalized
    )

    query_results = []

    for i, query_id in enumerate(query_ids):
        valid_indices = [
            j for j, pid in enumerate(candidate_ids)
            if pid != query_id
        ]

        ranked_indices = sorted(
            valid_indices,
            key=lambda j: combined[i, j],
            reverse=True
        )[:10]

        ranked_ids = [
            candidate_ids[j] for j in ranked_indices
        ]

        query_results.append(
            calculate_metrics(
                ranked_ids,
                relevance_map[query_id],
                k=10
            )
        )

    avg = np.mean(query_results, axis=0)

    fine_tuning_results.append({
        "TF-IDF Weight": tw,
        "GLAMOR Weight": gw,
        "Precision@10": avg[0],
        "Recall@10": avg[1],
        "MAP@10": avg[2],
        "nDCG@10": avg[3],
        "Hit Rate@10": avg[4]
    })

    print(
        f"TF-IDF {tw:.1%} + GLAMOR {gw:.1%} | "
        f"Hit Rate: {avg[4]*100:.2f}% | "
        f"nDCG: {avg[3]*100:.2f}%"
    )

# 4. DISPLAY BEST COMBINATION
fine_df = pd.DataFrame(fine_tuning_results)

best = fine_df.loc[fine_df["nDCG@10"].idxmax()]

print("\n========== FINE-TUNING RESULTS ==========")
print(fine_df.round(4).to_string(index=False))

print("\nBEST COMBINATION BY nDCG@10")
print(best.to_string())

# 5. SAVE RESULTS
OUT = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "outputs/hybrid_weight_fine_tuning.csv"
)

os.makedirs(os.path.dirname(OUT), exist_ok=True)
fine_df.to_csv(OUT, index=False)

print("\nResults saved to:")
print(OUT)


Starting hybrid weight fine-tuning...
TF-IDF 80.0% + GLAMOR 20.0% | Hit Rate: 79.80% | nDCG: 39.35%
TF-IDF 77.5% + GLAMOR 22.5% | Hit Rate: 80.00% | nDCG: 39.49%
TF-IDF 75.0% + GLAMOR 25.0% | Hit Rate: 79.60% | nDCG: 39.27%
TF-IDF 72.5% + GLAMOR 27.5% | Hit Rate: 79.60% | nDCG: 39.30%
TF-IDF 70.0% + GLAMOR 30.0% | Hit Rate: 79.40% | nDCG: 39.19%

========== FINE-TUNING RESULTS ==========
 TF-IDF Weight  GLAMOR Weight  Precision@10  Recall@10  MAP@10  nDCG@10  Hit Rate@10
         0.800          0.200        0.2384     0.3048  0.2822   0.3935        0.798
         0.775          0.225        0.2396     0.3069  0.2833   0.3949        0.800
         0.750          0.250        0.2382     0.3035  0.2816   0.3927        0.796
         0.725          0.275        0.2380     0.3034  0.2820   0.3930        0.796
         0.700          0.300        0.2372     0.3007  0.2816   0.3919        0.794

BEST COMBINATION BY nDCG@10
TF-IDF Weight    0.775000
GLAMOR Weight    0.225000
Precision@10     0

In [ ]:

import sqlite3
import os

DB_PATH = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "outputs/citation_knowledge_graph.sqlite"
)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Check the number of available test queries
cur.execute("SELECT COUNT(*) FROM test_queries")
total_test_queries = cur.fetchone()[0]

# Check the number of unique test queries
cur.execute("""
    SELECT COUNT(DISTINCT paper_id)
    FROM test_queries
""")
unique_test_queries = cur.fetchone()[0]

# Check the number of test relevance records
cur.execute("SELECT COUNT(*) FROM test_relevance")
total_relevance = cur.fetchone()[0]

# Check the number of training papers
cur.execute("SELECT COUNT(*) FROM training_ids")
training_count = cur.fetchone()[0]

# Check whether the 500 previously evaluated queries
# are stored in the test query table
previous_query_ids = [str(pid) for pid in query_ids]

placeholders = ",".join("?" for _ in previous_query_ids)

cur.execute(
    f"""
    SELECT COUNT(DISTINCT paper_id)
    FROM test_queries
    WHERE paper_id IN ({placeholders})
    """,
    previous_query_ids
)

previously_used = cur.fetchone()[0]

# Count test queries not used in the previous experiment
cur.execute("""
    SELECT COUNT(DISTINCT paper_id)
    FROM test_queries
""")
all_test_ids = {
    str(row[0]) for row in cur.execute(
        "SELECT DISTINCT paper_id FROM test_queries"
    )
}

unused_ids = all_test_ids - set(previous_query_ids)

print("========== INDEPENDENT TEST DATA ==========")
print("Total test query records:", total_test_queries)
print("Unique test queries:", unique_test_queries)
print("Training papers:", training_count)
print("Total relevance records:", total_relevance)
print("Previously used queries found:", previously_used)
print("Unused test queries available:", len(unused_ids))

conn.close()


========== INDEPENDENT TEST DATA ==========
Total test query records: 921853
Unique test queries: 921853
Training papers: 3687412
Total relevance records: 6557606
Previously used queries found: 500
Unused test queries available: 921353


In [ ]:

import sqlite3
import os
import random

DB_PATH = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "outputs/citation_knowledge_graph.sqlite"
)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# 1. Get previously evaluated query IDs
used_ids = set(str(x) for x in query_ids)

# 2. Select 500 unused test queries
cur.execute("""
    SELECT paper_id, title
    FROM test_queries
    ORDER BY paper_id
""")

unused_queries = [
    (str(pid), title)
    for pid, title in cur.fetchall()
    if str(pid) not in used_ids and title and title.strip()
]

random.seed(1203)
independent_queries = random.sample(
    unused_queries, 500
)

independent_ids = [
    pid for pid, title in independent_queries
]

# 3. Retrieve relevance records
relevance = {pid: set() for pid in independent_ids}

for start in range(0, len(independent_ids), 800):
    chunk = independent_ids[start:start + 800]
    placeholders = ",".join("?" for _ in chunk)

    cur.execute(
        f"""
        SELECT test_paper_id, relevant_paper_id
        FROM test_relevance
        WHERE test_paper_id IN ({placeholders})
        """,
        chunk
    )

    for query_id, relevant_id in cur.fetchall():
        relevance[str(query_id)].add(str(relevant_id))

# 4. Check whether relevant papers exist in training papers
all_relevant_ids = set().union(*relevance.values())

available_relevant = set()

relevant_list = list(all_relevant_ids)

for start in range(0, len(relevant_list), 800):
    chunk = relevant_list[start:start + 800]
    placeholders = ",".join("?" for _ in chunk)

    cur.execute(
        f"""
        SELECT paper_id
        FROM papers
        WHERE paper_id IN ({placeholders})
        """,
        chunk
    )

    available_relevant.update(
        str(row[0]) for row in cur.fetchall()
    )

queries_with_relevance = sum(
    bool(relevance[pid]) for pid in independent_ids
)

queries_with_available_relevance = sum(
    bool(relevance[pid] & available_relevant)
    for pid in independent_ids
)

print("========== INDEPENDENT QUERY PREPARATION ==========")
print("New queries selected:", len(independent_queries))
print("Queries with relevance labels:", queries_with_relevance)
print("Unique relevant papers:", len(all_relevant_ids))
print("Relevant papers available in training database:",
      len(available_relevant))
print("Queries with available relevant papers:",
      queries_with_available_relevance)

print("\nSample independent queries:")
for item in independent_queries[:5]:
    print(item)

conn.close()


========== INDEPENDENT QUERY PREPARATION ==========
New queries selected: 500
Queries with relevance labels: 389
Unique relevant papers: 3574
Relevant papers available in training database: 3574
Queries with available relevant papers: 389

Sample independent queries:
('10004270582', 'An Exponential Lower Bound for Proofs in Focused Calculi')
('10001070784', 'Combining multi-objective search and constraint solving for configuring large software product lines')
('10001894083', 'Security of Multivariate Signature Scheme Using Non-commutative Rings')
('10000980236', 'Industrial valves with electric actuators: Technical diagnostics by a moment sensor within smart actuator worm-and-wheel gearbox')
('10003628574', 'Speech perception - perceptual flexibility.')


In [ ]:

# ============================================================
# INDEPENDENT FINAL TEST - GLAMOR + TF-IDF HYBRID
# Fixed weights: 77.5% TF-IDF + 22.5% GLAMOR
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = "/content/drive/MyDrive/GLAMOR_Project"
DB_PATH = os.path.join(ROOT, "outputs/citation_knowledge_graph.sqlite")
MODEL_DIR = os.path.join(
    ROOT, "outputs/cpu_fast_experiment/trained_model"
)
OUTPUT_DIR = os.path.join(ROOT, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. VERIFY PREPARED DATA
assert len(independent_queries) == 500
assert len(relevance) == 500

evaluated_queries = [
    (pid, title)
    for pid, title in independent_queries
    if relevance.get(pid)
]

unlabelled_count = 500 - len(evaluated_queries)

print("Total independent queries:", 500)
print("Queries with relevance labels:", len(evaluated_queries))
print("Queries without labels:", unlabelled_count)

# 2. BUILD CANDIDATE POOL
# Include original candidates AND every known relevant paper.
candidate_pool = set(str(pid) for pid in candidate_ids)

for pid, title in evaluated_queries:
    candidate_pool.update(relevance[pid])

# Remove each query paper from the shared candidate pool.
# It will also be excluded individually during ranking.
candidate_pool = sorted(candidate_pool)

print("Candidate papers in independent test:", len(candidate_pool))

# 3. FETCH ALL CANDIDATE TITLES FROM SQLITE
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

candidate_title_map = {}

for start in range(0, len(candidate_pool), 800):
    chunk = candidate_pool[start:start + 800]
    placeholders = ",".join("?" for _ in chunk)

    cur.execute(
        f"""
        SELECT paper_id, title
        FROM papers
        WHERE paper_id IN ({placeholders})
        """,
        chunk
    )

    for pid, title in cur.fetchall():
        candidate_title_map[str(pid)] = title

conn.close()

missing = [
    pid for pid in candidate_pool
    if not candidate_title_map.get(pid)
    or not candidate_title_map[pid].strip()
]

if missing:
    raise ValueError(
        f"{len(missing)} candidate titles are missing. "
        "No candidates were silently removed."
    )

print("Candidate titles verified:", len(candidate_title_map))

# 4. PREPARE QUERY AND CANDIDATE TITLES
query_ids_ind = [pid for pid, title in evaluated_queries]
query_texts_ind = [title for pid, title in evaluated_queries]

candidate_texts_ind = [
    candidate_title_map[pid] for pid in candidate_pool
]

# 5. CALCULATE TF-IDF SCORES
print("\nCalculating independent TF-IDF scores...")

vectorizer_ind = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=1
)

candidate_tfidf_ind = vectorizer_ind.fit_transform(
    candidate_texts_ind
)
query_tfidf_ind = vectorizer_ind.transform(query_texts_ind)

tfidf_scores_ind = cosine_similarity(
    query_tfidf_ind, candidate_tfidf_ind
)

# 6. LOAD THE TRAINED GLAMOR MODEL
print("\nLoading trained GLAMOR model...")

tokenizer_ind = AutoTokenizer.from_pretrained(MODEL_DIR)
model_ind = AutoModel.from_pretrained(MODEL_DIR)
model_ind.eval()
model_ind.to("cpu")

# 7. GENERATE EMBEDDINGS
def encode_independent(texts, batch_size=32):
    vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer_ind(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        with torch.no_grad():
            output = model_ind(**inputs)
            hidden = output.last_hidden_state
            mask = inputs["attention_mask"].unsqueeze(-1)

            pooled = (hidden * mask).sum(dim=1)
            pooled = pooled / mask.sum(dim=1).clamp(min=1)

            pooled = torch.nn.functional.normalize(
                pooled, p=2, dim=1
            )

        vectors.append(pooled.cpu())

        print(
            "Encoded:",
            min(start + batch_size, len(texts)),
            "/",
            len(texts)
        )

    return torch.cat(vectors, dim=0)

print("\nEncoding candidate papers...")
candidate_embeddings_ind = encode_independent(
    candidate_texts_ind
)

print("\nEncoding independent queries...")
query_embeddings_ind = encode_independent(
    query_texts_ind
)

glamor_scores_ind = torch.matmul(
    query_embeddings_ind,
    candidate_embeddings_ind.T
).numpy()

# 8. NORMALIZE SCORES PER QUERY
def normalize_rows_ind(scores):
    minimum = scores.min(axis=1, keepdims=True)
    maximum = scores.max(axis=1, keepdims=True)

    return (scores - minimum) / np.maximum(
        maximum - minimum, 1e-12
    )

tfidf_norm_ind = normalize_rows_ind(tfidf_scores_ind)
glamor_norm_ind = normalize_rows_ind(glamor_scores_ind)

# 9. APPLY FROZEN HYBRID WEIGHTS
TFIDF_WEIGHT = 0.775
GLAMOR_WEIGHT = 0.225

hybrid_scores_ind = (
    TFIDF_WEIGHT * tfidf_norm_ind
    + GLAMOR_WEIGHT * glamor_norm_ind
)

# 10. METRIC CALCULATION
def metrics_ind(ranked_ids, relevant_ids, k=10):
    relevant_ids = set(relevant_ids)
    top_ids = ranked_ids[:k]

    hits = [int(pid in relevant_ids) for pid in top_ids]

    precision = sum(hits) / k
    recall = (
        sum(hits) / len(relevant_ids)
        if relevant_ids else 0
    )

    hit_count = 0
    ap_sum = 0

    for rank, hit in enumerate(hits, start=1):
        if hit:
            hit_count += 1
            ap_sum += hit_count / rank

    denominator = min(len(relevant_ids), k)
    ap = ap_sum / denominator if denominator else 0

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, start=1)
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, min(len(relevant_ids), k) + 1)
    )

    ndcg = dcg / idcg if idcg else 0
    hit_rate = int(sum(hits) > 0)

    return precision, recall, ap, ndcg, hit_rate

# 11. EVALUATE LABELLED QUERIES
results_ind = []

print("\nEvaluating independent queries...")

for i, query_id in enumerate(query_ids_ind):

    valid_indices = [
        j for j, pid in enumerate(candidate_pool)
        if pid != query_id
    ]

    ranked_indices = sorted(
        valid_indices,
        key=lambda j: hybrid_scores_ind[i, j],
        reverse=True
    )[:10]

    ranked_ids = [
        candidate_pool[j] for j in ranked_indices
    ]

    result = metrics_ind(
        ranked_ids,
        relevance[query_id],
        k=10
    )

    results_ind.append(result)

    if (i + 1) % 50 == 0:
        print("Evaluated:", i + 1, "/", len(query_ids_ind))

# 12. FINAL RESULTS
average_ind = np.mean(results_ind, axis=0)

metric_names = [
    "Precision@10",
    "Recall@10",
    "MAP@10",
    "nDCG@10",
    "Hit Rate@10"
]

report = [
    "INDEPENDENT FINAL HYBRID EVALUATION",
    "====================================",
    "Fixed TF-IDF weight: 77.5%",
    "Fixed GLAMOR weight: 22.5%",
    f"Independent queries selected: 500",
    f"Queries evaluated: {len(query_ids_ind)}",
    f"Queries without relevance labels: {unlabelled_count}",
    f"Candidate papers: {len(candidate_pool)}",
    ""
]

for name, value in zip(metric_names, average_ind):
    report.append(
        f"{name}: {value:.4f} ({value * 100:.2f}%)"
    )

report_text = "\n".join(report)
print("\n" + report_text)

# 13. SAVE RESULTS
report_path = os.path.join(
    OUTPUT_DIR, "independent_final_evaluation.txt"
)

csv_path = os.path.join(
    OUTPUT_DIR, "independent_final_evaluation.csv"
)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

pd.DataFrame(
    results_ind, columns=metric_names
).to_csv(csv_path, index=False)

print("\nReports saved:")
print(report_path)
print(csv_path)


Total independent queries: 500
Queries with relevance labels: 389
Queries without labels: 111
Candidate papers in independent test: 8951
Candidate titles verified: 8951

Calculating independent TF-IDF scores...

Loading trained GLAMOR model...


Some weights of RobertaModel were not initialized from the model checkpoint at /content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Encoding candidate papers...
Encoded: 32 / 8951
Encoded: 64 / 8951
Encoded: 96 / 8951
Encoded: 128 / 8951
Encoded: 160 / 8951
Encoded: 192 / 8951
Encoded: 224 / 8951
Encoded: 256 / 8951
Encoded: 288 / 8951
Encoded: 320 / 8951
Encoded: 352 / 8951
Encoded: 384 / 8951
Encoded: 416 / 8951
Encoded: 448 / 8951
Encoded: 480 / 8951
Encoded: 512 / 8951
Encoded: 544 / 8951
Encoded: 576 / 8951
Encoded: 608 / 8951
Encoded: 640 / 8951
Encoded: 672 / 8951
Encoded: 704 / 8951
Encoded: 736 / 8951
Encoded: 768 / 8951
Encoded: 800 / 8951
Encoded: 832 / 8951
Encoded: 864 / 8951
Encoded: 896 / 8951
Encoded: 928 / 8951
Encoded: 960 / 8951
Encoded: 992 / 8951
Encoded: 1024 / 8951
Encoded: 1056 / 8951
Encoded: 1088 / 8951
Encoded: 1120 / 8951
Encoded: 1152 / 8951
Encoded: 1184 / 8951
Encoded: 1216 / 8951
Encoded: 1248 / 8951
Encoded: 1280 / 8951
Encoded: 1312 / 8951
Encoded: 1344 / 8951
Encoded: 1376 / 8951
Encoded: 1408 / 8951
Encoded: 1440 / 8951
Encoded: 1472 / 8951
Encoded: 1504 / 8951
Encoded: 1536 / 8

In [ ]:

import os

APP_DIR = (
    "/content/drive/MyDrive/GLAMOR_Project/application"
)

APP_FILE = os.path.join(APP_DIR, "app.py")

print("Checking existing Flask application...")

if not os.path.exists(APP_FILE):
    print("ERROR: app.py was not found.")
else:
    with open(APP_FILE, "r", encoding="utf-8") as f:
        app_code = f.read()

    print("\n========== CURRENT APP.PY ==========\n")
    print(app_code)

    print("\n========== TEMPLATE FILES ==========")

    template_dir = os.path.join(APP_DIR, "templates")

    if os.path.exists(template_dir):
        for filename in os.listdir(template_dir):
            print("-", filename)
    else:
        print("Templates folder not found.")


Checking existing Flask application...

========== CURRENT APP.PY ==========


import sqlite3
import re
from pathlib import Path
from flask import Flask, request, render_template, jsonify
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

app = Flask(__name__)

DB = Path(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/"
    "citation_knowledge_graph.sqlite"
)

def get_db():
    conn = sqlite3.connect(str(DB))
    conn.row_factory = sqlite3.Row
    return conn

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/search")
def search():
    query = request.args.get("q", "").strip()

    if not query:
        return jsonify([])

    conn = get_db()
    rows = conn.execute("""
        SELECT paper_id, title, year
        FROM papers
        WHERE title LIKE ?
        LIMIT 20
    """, (f"%{query}%",)).fetchall()
    conn.close()

    return jsonify([dict(row) for row in rows])

@app.route("/p

In [ ]:
APP_FILE = "/content/drive/MyDrive/GLAMOR_Project/application/app.py"

with open(APP_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("\n".join(
    f"{i+1}: {line.rstrip()}"
    for i, line in enumerate(lines[:123])
))

1: 
2: import sqlite3
3: import re
4: from pathlib import Path
5: from flask import Flask, request, render_template, jsonify
6: from sklearn.feature_extraction.text import TfidfVectorizer
7: from sklearn.metrics.pairwise import cosine_similarity
8: 
9: app = Flask(__name__)
10: 
11: DB = Path(
12:     "/content/drive/MyDrive/GLAMOR_Project/outputs/"
13:     "citation_knowledge_graph.sqlite"
14: )
15: 
16: def get_db():
17:     conn = sqlite3.connect(str(DB))
18:     conn.row_factory = sqlite3.Row
19:     return conn
20: 
21: @app.route("/")
22: def home():
23:     return render_template("index.html")
24: 
25: @app.route("/search")
26: def search():
27:     query = request.args.get("q", "").strip()
28: 
29:     if not query:
30:         return jsonify([])
31: 
32:     conn = get_db()
33:     rows = conn.execute("""
34:         SELECT paper_id, title, year
35:         FROM papers
36:         WHERE title LIKE ?
37:         LIMIT 20
38:     """, (f"%{query}%",)).fetchall()
39:     conn.clo

In [ ]:
APP_FILE = "/content/drive/MyDrive/GLAMOR_Project/application/app.py"

with open(APP_FILE, "r", encoding="utf-8") as f:
    for i in range(5):
        print(f"{i+1}: {f.readline().rstrip()}")

1: 
2: import sqlite3
3: import re
4: from pathlib import Path
5: from flask import Flask, request, render_template, jsonify


In [ ]:

# ============================================================
# INTEGRATE HYBRID GLAMOR + TF-IDF INTO EXISTING FLASK WEBSITE
# ============================================================

import os
import re
import shutil
from pathlib import Path

APP_DIR = Path(
    "/content/drive/MyDrive/GLAMOR_Project/application"
)

APP_FILE = APP_DIR / "app.py"
MODEL_DIR = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "outputs/cpu_fast_experiment/trained_model"
)

assert APP_FILE.exists(), "Existing app.py was not found."
assert os.path.isdir(MODEL_DIR), "Trained GLAMOR model was not found."

# 1. Read existing application
source = APP_FILE.read_text(encoding="utf-8")

# 2. Find recommendation route and main section
route_marker = '@app.route("/recommend/<paper_id>")'
main_marker = 'if __name__ == "__main__":'

if route_marker not in source:
    raise RuntimeError("Recommendation route marker not found.")

if main_marker not in source:
    raise RuntimeError("Flask main section not found.")

start = source.index(route_marker)
end = source.index(main_marker, start)

# 3. New hybrid recommendation route
hybrid_route = r'''
# GLAMOR model is loaded only when recommendations are requested.
from functools import lru_cache

@lru_cache(maxsize=1)
def load_glamor_encoder():
    import torch
    from transformers import AutoTokenizer, AutoModel

    model_path = (
        "/content/drive/MyDrive/GLAMOR_Project/"
        "outputs/cpu_fast_experiment/trained_model"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path)
    model.eval()
    model.to("cpu")

    return tokenizer, model, torch


def encode_glamor_titles(texts):
    tokenizer, model, torch = load_glamor_encoder()
    all_embeddings = []

    for batch_start in range(0, len(texts), 32):
        batch = texts[batch_start:batch_start + 32]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        with torch.no_grad():
            output = model(**inputs)
            hidden = output.last_hidden_state
            mask = inputs["attention_mask"].unsqueeze(-1)

            pooled = (hidden * mask).sum(dim=1)
            pooled = pooled / mask.sum(dim=1).clamp(min=1)

            pooled = torch.nn.functional.normalize(
                pooled, p=2, dim=1
            )

        all_embeddings.append(pooled.cpu())

    return torch.cat(all_embeddings, dim=0)


def normalize_recommendation_scores(scores):
    scores = scores.astype(float)
    minimum = scores.min()
    maximum = scores.max()

    if maximum - minimum < 1e-12:
        return scores * 0.0

    return (scores - minimum) / (maximum - minimum)


@app.route("/recommend/<paper_id>")
def recommend(paper_id):
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer"
    from sklearn.metrics.pairwise import cosine_similarity

    conn = get_db()

    selected = conn.execute("""
        SELECT paper_id, title
        FROM papers
        WHERE paper_id = ?
    """, (paper_id,)).fetchone()

    if selected is None:
        conn.close()
        return jsonify({"error": "Paper not found"}), 404

    title = selected["title"] or ""

    words = list(dict.fromkeys(
        word.lower()
        for word in re.findall(r"[A-Za-z0-9]+", title)
        if len(word) > 2
    ))

    if not words:
        conn.close()
        return jsonify([])

    conditions = " OR ".join(
        ["title LIKE ?" for _ in words]
    )

    rows = conn.execute(f"""
        SELECT paper_id, title
        FROM papers
        WHERE paper_id != ?
          AND ({conditions})
          AND title IS NOT NULL
          AND title != ''
        LIMIT 1000
    """, (paper_id, *[f"%{word}%" for word in words])).fetchall()

    conn.close()

    candidates = [
        dict(row) for row in rows
        if row["title"] and row["title"].strip()
    ]

    if not candidates:
        return jsonify([])

    # TF-IDF similarity
    documents = [title] + [
        item["title"] for item in candidates
    ]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=20000
    )

    vectors = vectorizer.fit_transform(documents)

    tfidf_scores = cosine_similarity(
        vectors[0:1], vectors[1:]
    ).flatten()

    # GLAMOR semantic similarity
    embeddings = encode_glamor_titles(documents)

    glamor_scores = (
        embeddings[1:] @ embeddings[0]
    ).numpy()

    # Normalize both scores
    tfidf_normalized = normalize_recommendation_scores(
        tfidf_scores
    )

    glamor_normalized = normalize_recommendation_scores(
        glamor_scores
    )

    # Frozen weights from the evaluation experiment
    hybrid_scores = (
        0.775 * tfidf_normalized
        + 0.225 * glamor_normalized
    )

    # Select top 10 recommendations
    ranked = np.argsort(-hybrid_scores)[:10]

    recommendations = []

    for index in ranked:
        item = candidates[index]

        recommendations.append({
            "paper_id": item["paper_id"],
            "title": item["title"],
            "similarity_score": round(
                float(hybrid_scores[index]), 4
            ),
            "tfidf_score": round(
                float(tfidf_normalized[index]), 4
            ),
            "glamor_score": round(
                float(glamor_normalized[index]), 4
            )
        })

    return jsonify(recommendations)


'''

# Fix accidental quote in the sklearn import line
hybrid_route = hybrid_route.replace(
    'from sklearn.feature_extraction.text import TfidfVectorizer"',
    'from sklearn.feature_extraction.text import TfidfVectorizer'
)

# 4. Create a backup before changing anything
backup_file = APP_DIR / "app_before_hybrid.py"

if not backup_file.exists():
    shutil.copy2(APP_FILE, backup_file)

# 5. Replace only the recommendation route
updated_source = (
    source[:start]
    + hybrid_route
    + source[end:]
)

APP_FILE.write_text(updated_source, encoding="utf-8")

# 6. Check Python syntax without starting the server
import ast

ast.parse(updated_source)

print("========================================")
print("HYBRID INTEGRATION COMPLETED")
print("========================================")
print("Original website backup:", backup_file)
print("Updated application:", APP_FILE)
print("TF-IDF weight: 77.5%")
print("GLAMOR weight: 22.5%")
print("Top recommendations: 10")
print("Python syntax: VALID")
print("Homepage, search and paper details routes preserved.")


HYBRID INTEGRATION COMPLETED
Original website backup: /content/drive/MyDrive/GLAMOR_Project/application/app_before_hybrid.py
Updated application: /content/drive/MyDrive/GLAMOR_Project/application/app.py
TF-IDF weight: 77.5%
GLAMOR weight: 22.5%
Top recommendations: 10
Python syntax: VALID
Homepage, search and paper details routes preserved.


In [ ]:

import sys
import os
import importlib

APP_DIR = "/content/drive/MyDrive/GLAMOR_Project/application"

if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

os.chdir(APP_DIR)

import app
importlib.reload(app)

flask_app = app.app
client = flask_app.test_client()

print("========== WEBSITE TESTING ==========")

# Test 1: Homepage
response = client.get("/")
print("Homepage:", response.status_code)

# Test 2: Search
response = client.get("/search?q=machine+learning")
print("Search:", response.status_code)

search_results = response.get_json()
print("Search results:", len(search_results))

if not search_results:
    raise RuntimeError("Search returned no papers.")

# Test 3: Paper details
paper_id = search_results[0]["paper_id"]

response = client.get(f"/paper/{paper_id}")
print("Paper details:", response.status_code)

# Test 4: Hybrid recommendations
response = client.get(f"/recommend/{paper_id}")
print("Hybrid recommendations:", response.status_code)

recommendations = response.get_json()

if response.status_code == 200:
    print("Recommendations returned:", len(recommendations))
    print("\nSample recommendations:")

    for item in recommendations[:3]:
        print(item)

print("\n========== TESTING FINISHED ==========")


========== WEBSITE TESTING ==========
Homepage: 200
Search: 200
Search results: 20
Paper details: 200


Some weights of RobertaModel were not initialized from the model checkpoint at /content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Hybrid recommendations: 200
Recommendations returned: 10

Sample recommendations:
{'glamor_score': 0.985, 'paper_id': '10000000192', 'similarity_score': 0.9687, 'tfidf_score': 0.964, 'title': 'Mobilizing the semantic web with DAML-enabled web services'}
{'glamor_score': 0.6846, 'paper_id': '10000000497', 'similarity_score': 0.929, 'tfidf_score': 1.0, 'title': 'Annotation for the Semantic Web During Website Development'}
{'glamor_score': 0.898, 'paper_id': '10000000415', 'similarity_score': 0.925, 'tfidf_score': 0.9329, 'title': 'Machine learning for adaptive image interpretation'}

========== TESTING FINISHED ==========


In [ ]:

import sys
import os
import importlib

APP_DIR = (
    "/content/drive/MyDrive/GLAMOR_Project/application"
)

sys.path.insert(0, APP_DIR)
os.chdir(APP_DIR)

import app
importlib.reload(app)

client = app.app.test_client()

topics = [
    "machine learning",
    "artificial intelligence",
    "cybersecurity",
    "natural language processing",
    "data mining"
]

print("========== FINAL TOPIC TESTING ==========")

for topic in topics:
    print("\nTOPIC:", topic)

    search_response = client.get(
        "/search",
        query_string={"q": topic}
    )

    if search_response.status_code != 200:
        print("Search failed:", search_response.status_code)
        continue

    papers = search_response.get_json()

    if not papers:
        print("No papers found.")
        continue

    print("Search results:", len(papers))

    paper_id = papers[0]["paper_id"]

    recommendation_response = client.get(
        f"/recommend/{paper_id}"
    )

    if recommendation_response.status_code != 200:
        print("Recommendation failed:",
              recommendation_response.status_code)
        continue

    recommendations = recommendation_response.get_json()

    print("Recommendations:", len(recommendations))

    for index, item in enumerate(recommendations[:3], 1):
        print(
            index,
            item["title"],
            "| Hybrid score:",
            item["similarity_score"]
        )

print("\n========== FINAL TOPIC TEST FINISHED ==========")


========== FINAL TOPIC TESTING ==========

TOPIC: machine learning
Search results: 20


Some weights of RobertaModel were not initialized from the model checkpoint at /content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Recommendations: 10
1 Mobilizing the semantic web with DAML-enabled web services | Hybrid score: 0.9687
2 Annotation for the Semantic Web During Website Development | Hybrid score: 0.929
3 Machine learning for adaptive image interpretation | Hybrid score: 0.925

TOPIC: artificial intelligence
Search results: 20
Recommendations: 10
1 Editorial - Objects, Databases, and the WWW. | Hybrid score: 0.967
2 Big Data – Challenges for Computer Science Education | Hybrid score: 0.9363
3 Multiple-Valued Logic - Guest Editor's Introduction. | Hybrid score: 0.8421

TOPIC: cybersecurity
Search results: 20
Recommendations: 10
1 Scenario-based user testing to guide consumer health informatics design. | Hybrid score: 0.9043
2 News: Where is Guide Going? | Hybrid score: 0.8347
3 A Smart TV Guide. | Hybrid score: 0.7986

TOPIC: natural language processing
Search results: 20
Recommendations: 10
1 On natural language processing and plan recognition | Hybrid score: 0.9196
2 Building an automated problem lis

In [ ]:

import sqlite3

DB_PATH = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "outputs/citation_knowledge_graph.sqlite"
)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

topics = [
    "machine learning",
    "artificial intelligence",
    "cybersecurity",
    "natural language processing",
    "data mining"
]

print("========== FTS PAPER QUALITY TEST ==========")

for topic in topics:
    words = topic.split()

    # Search for any of the topic words
    fts_query = " OR ".join(
        f'"{word}"' for word in words
    )

    print("\nTOPIC:", topic)

    cur.execute("""
        SELECT p.paper_id, p.title
        FROM papers_fts AS f
        JOIN papers AS p
          ON p.paper_id = f.paper_id
        WHERE papers_fts MATCH ?
        ORDER BY bm25(papers_fts)
        LIMIT 5
    """, (fts_query,))

    results = cur.fetchall()

    for index, (paper_id, title) in enumerate(results, 1):
        print(f"{index}. {title}")

    print("Retrieved:", len(results), "sample papers")

conn.close()


========== FTS PAPER QUALITY TEST ==========

TOPIC: machine learning
1. Do Machine-Learning Machines Learn?
2. Learning machine learning
3. Learning machines
4. Machine learning
5. Learning to learn: From smart machines to intelligent machines
Retrieved: 5 sample papers

TOPIC: artificial intelligence
1. Artificial intelligence for artificial artificial intelligence
2. Intelligent Assistance without Artificial Intelligence
3. Human intelligence needs artificial intelligence
4. Web Intelligence and Artificial Intelligence
5. Artificial intelligence and ambient intelligence
Retrieved: 5 sample papers

TOPIC: cybersecurity
1. Introduction: Cybersecurity
2. Cybersecurity dynamics
3. Demythifying Cybersecurity
4. Toward global cybersecurity collaboration: Cybersecurity operation activity model
5. The Cybersecurity Competition Experience: Perceptions from Cybersecurity Workers
Retrieved: 5 sample papers

TOPIC: natural language processing
1. Processing natural language without natural langu

In [ ]:

# ============================================================
# IMPROVE FLASK CANDIDATE RETRIEVAL USING SQLITE FTS
# Preserve existing hybrid scoring and website routes
# ============================================================

import os
import shutil
import ast
from pathlib import Path

APP_FILE = Path(
    "/content/drive/MyDrive/GLAMOR_Project/"
    "application/app.py"
)

assert APP_FILE.exists(), "app.py not found."

source = APP_FILE.read_text(encoding="utf-8")

start_marker = '    conditions = " OR ".join('
end_marker = '    conn.close()'

start = source.find(start_marker)
if start == -1:
    raise RuntimeError("Existing LIKE retrieval block not found.")

end = source.find(end_marker, start)
if end == -1:
    raise RuntimeError("Database connection closing point not found.")

# Create a separate backup before this update
backup = APP_FILE.with_name("app_before_fts_update.py")

if not backup.exists():
    shutil.copy2(APP_FILE, backup)

# Replace only the candidate retrieval section
fts_retrieval = '''
    # Retrieve candidates using SQLite Full-Text Search
    search_words = list(dict.fromkeys(
        re.findall(r"[A-Za-z0-9]+", title.lower())
    ))

    # Limit query terms to avoid excessively broad searches
    search_words = [
        word for word in search_words if len(word) > 2
    ][:12]

    candidates = []

    if search_words:
        fts_query = " OR ".join(
            '"' + word.replace('"', '""') + '"'
            for word in search_words
        )

        try:
            rows = conn.execute("""
                SELECT p.paper_id, p.title
                FROM papers_fts
                JOIN papers AS p
                  ON p.paper_id = papers_fts.paper_id
                WHERE papers_fts MATCH ?
                  AND p.paper_id != ?
                  AND p.title IS NOT NULL
                  AND p.title != ''
                ORDER BY bm25(papers_fts)
                LIMIT 1000
            """, (fts_query, paper_id)).fetchall()

            candidates = [
                dict(row) for row in rows
                if row["title"] and row["title"].strip()
            ]

        except Exception as error:
            print("FTS retrieval error:", error)

    # Fallback to the previous LIKE search if FTS finds nothing
    if not candidates and search_words:
        conditions = " OR ".join(
            ["title LIKE ?" for _ in search_words]
        )

        rows = conn.execute(f"""
            SELECT paper_id, title
            FROM papers
            WHERE paper_id != ?
              AND ({conditions})
              AND title IS NOT NULL
              AND title != ''
            LIMIT 1000
        """, (
            paper_id,
            *[f"%{word}%" for word in search_words]
        )).fetchall()

        candidates = [
            dict(row) for row in rows
            if row["title"] and row["title"].strip()
        ]

'''

updated = source[:start] + fts_retrieval + source[end:]

# Verify syntax before writing
ast.parse(updated)

APP_FILE.write_text(updated, encoding="utf-8")

print("======================================")
print("FTS INTEGRATION COMPLETED")
print("======================================")
print("FTS candidate retrieval: Integrated")
print("Fallback LIKE search: Preserved")
print("Hybrid weights: 77.5% TF-IDF + 22.5% GLAMOR")
print("Top recommendations: 10")
print("Other Flask routes: Preserved")
print("Python syntax: VALID")
print("Backup:", backup)


FTS INTEGRATION COMPLETED
FTS candidate retrieval: Integrated
Fallback LIKE search: Preserved
Hybrid weights: 77.5% TF-IDF + 22.5% GLAMOR
Top recommendations: 10
Other Flask routes: Preserved
Python syntax: VALID
Backup: /content/drive/MyDrive/GLAMOR_Project/application/app_before_fts_update.py


In [ ]:

import os
import sys
import importlib

APP_DIR = (
    "/content/drive/MyDrive/GLAMOR_Project/application"
)

sys.path.insert(0, APP_DIR)
os.chdir(APP_DIR)

import app
importlib.reload(app)

client = app.app.test_client()

topics = [
    "machine learning",
    "artificial intelligence",
    "cybersecurity",
    "natural language processing",
    "data mining"
]

print("========== FTS + HYBRID WEBSITE TEST ==========")

for topic in topics:
    print("\nTOPIC:", topic)

    response = client.get(
        "/search",
        query_string={"q": topic}
    )

    print("Search status:", response.status_code)

    if response.status_code != 200:
        continue

    papers = response.get_json()

    if not papers:
        print("No search results.")
        continue

    print("Search results:", len(papers))

    paper_id = papers[0]["paper_id"]

    rec_response = client.get(
        f"/recommend/{paper_id}"
    )

    print("Recommendation status:", rec_response.status_code)

    if rec_response.status_code == 200:
        recommendations = rec_response.get_json()

        print("Recommendations:", len(recommendations))

        for i, item in enumerate(recommendations[:3], 1):
            print(
                f"{i}. {item['title']} "
                f"| Score: {item['similarity_score']}"
            )

print("\n========== FTS WEBSITE TEST COMPLETED ==========")


========== FTS + HYBRID WEBSITE TEST ==========

TOPIC: machine learning
Search status: 200
Search results: 20


Some weights of RobertaModel were not initialized from the model checkpoint at /content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Recommendation status: 200
Recommendations: 10
1. Machine learning for information extraction | Score: 0.9611
2. Machine Learning for Information Extraction | Score: 0.9093
3. Learning, Information Extraction and the Web | Score: 0.7138

TOPIC: artificial intelligence
Search status: 200
Search results: 20
Recommendation status: 200
Recommendations: 10
1. Philosophical and psychological problems of artificial intelligence | Score: 0.8719
2. On artificial intelligence | Score: 0.8416
3. The application of algorithmic probability to problems in artificial intelligence | Score: 0.7716

TOPIC: cybersecurity
Search status: 200
Search results: 20
Recommendation status: 200
Recommendations: 10
1. IAGO: Interactive Arbitration Guide Online (Demonstration) | Score: 0.9598
2. Design recommendations for online cybersecurity courses | Score: 0.9239
3. Activity Theory to Guide Online Collaborative Learning Instructional Design | Score: 0.8431

TOPIC: natural language processing
Search status: 200
Se

In [ ]:

# ============================================================
# UPDATE FLASK SEARCH ROUTE TO USE SQLITE FTS
# Preserve all other routes and create a backup
# ============================================================

import ast
import shutil
from pathlib import Path

APP_FILE = Path(
    "/content/drive/MyDrive/GLAMOR_Project/"
    "application/app.py"
)

source = APP_FILE.read_text(encoding="utf-8")

start_marker = '@app.route("/search")'
end_marker = '@app.route("/paper/<paper_id>")'

start = source.find(start_marker)
end = source.find(end_marker, start)

if start == -1 or end == -1:
    raise RuntimeError("Could not locate the search route boundaries.")

# Backup current FTS-integrated application
backup = APP_FILE.with_name("app_before_search_fts.py")

if not backup.exists():
    shutil.copy2(APP_FILE, backup)

new_search_route = '''
@app.route("/search")
def search():
    query = request.args.get("q", "").strip()

    if not query:
        return jsonify([])

    conn = get_db()

    # Extract useful search words
    words = list(dict.fromkeys(
        re.findall(r"[A-Za-z0-9]+", query.lower())
    ))

    words = [word for word in words if len(word) > 1][:12]

    if not words:
        conn.close()
        return jsonify([])

    # Search using SQLite Full-Text Search
    fts_query = " OR ".join(
        '"' + word.replace('"', '""') + '"'
        for word in words
    )

    rows = []

    try:
        rows = conn.execute("""
            SELECT p.paper_id, p.title, p.year
            FROM papers_fts
            JOIN papers AS p
              ON p.paper_id = papers_fts.paper_id
            WHERE papers_fts MATCH ?
              AND p.title IS NOT NULL
              AND p.title != ''
            ORDER BY bm25(papers_fts)
            LIMIT 20
        """, (fts_query,)).fetchall()

    except Exception as error:
        print("FTS search error:", error)

    # Fallback to LIKE search
    if not rows:
        conditions = " OR ".join(
            ["title LIKE ?" for _ in words]
        )

        rows = conn.execute(f"""
            SELECT paper_id, title, year
            FROM papers
            WHERE ({conditions})
              AND title IS NOT NULL
              AND title != ''
            LIMIT 20
        """, tuple(f"%{word}%" for word in words)).fetchall()

    conn.close()

    return jsonify([dict(row) for row in rows])


'''

updated = source[:start] + new_search_route + source[end:]

# Validate syntax before saving
ast.parse(updated)

APP_FILE.write_text(updated, encoding="utf-8")

print("====================================")
print("FTS SEARCH ROUTE UPDATED")
print("====================================")
print("Search route: FTS with LIKE fallback")
print("Recommendation route: Preserved")
print("Paper details route: Preserved")
print("Hybrid weights: 77.5% TF-IDF + 22.5% GLAMOR")
print("Python syntax: VALID")
print("Backup:", backup)


FTS SEARCH ROUTE UPDATED
Search route: FTS with LIKE fallback
Recommendation route: Preserved
Paper details route: Preserved
Hybrid weights: 77.5% TF-IDF + 22.5% GLAMOR
Python syntax: VALID
Backup: /content/drive/MyDrive/GLAMOR_Project/application/app_before_search_fts.py


In [ ]:

import os
import sys
import importlib

APP_DIR = (
    "/content/drive/MyDrive/GLAMOR_Project/application"
)

sys.path.insert(0, APP_DIR)
os.chdir(APP_DIR)

import app
importlib.reload(app)

client = app.app.test_client()

topics = [
    "machine learning",
    "artificial intelligence",
    "cybersecurity",
    "natural language processing",
    "data mining"
]

print("========== FINAL FTS SEARCH TEST ==========")

for topic in topics:
    print("\nTOPIC:", topic)

    response = client.get(
        "/search",
        query_string={"q": topic}
    )

    print("Search status:", response.status_code)

    if response.status_code != 200:
        print("Search failed.")
        continue

    papers = response.get_json()
    print("Search results:", len(papers))

    for i, paper in enumerate(papers[:3], 1):
        print(f"Search {i}: {paper['title']}")

    if not papers:
        print("No papers found.")
        continue

    paper_id = papers[0]["paper_id"]

    details = client.get(f"/paper/{paper_id}")
    print("Paper details status:", details.status_code)

    rec_response = client.get(
        f"/recommend/{paper_id}"
    )

    print("Recommendation status:", rec_response.status_code)

    if rec_response.status_code == 200:
        recommendations = rec_response.get_json()
        print("Recommendations:", len(recommendations))

        for i, item in enumerate(recommendations[:3], 1):
            print(
                f"Recommendation {i}: {item['title']} "
                f"| Score: {item['similarity_score']}"
            )

print("\n========== FINAL FTS TEST COMPLETED ==========")


========== FINAL FTS SEARCH TEST ==========

TOPIC: machine learning
Search status: 200
Search results: 20
Search 1: Do Machine-Learning Machines Learn?
Search 2: Learning machine learning
Search 3: Learning machines
Paper details status: 200


Some weights of RobertaModel were not initialized from the model checkpoint at /content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Recommendation status: 200
Recommendations: 10
Recommendation 1: When will machines learn | Score: 0.9016
Recommendation 2: When will Machines Learn | Score: 0.8838
Recommendation 3: Why Machines Cannot Learn Mathematics, Yet. | Score: 0.6917

TOPIC: artificial intelligence
Search status: 200
Search results: 20
Search 1: Artificial intelligence for artificial artificial intelligence
Search 2: Intelligent Assistance without Artificial Intelligence
Search 3: Human intelligence needs artificial intelligence
Paper details status: 200
Recommendation status: 200
Recommendations: 10
Recommendation 1: Intelligence, Artificial and Otherwise | Score: 0.8545
Recommendation 2: The why and when of artificial intelligence | Score: 0.6671
Recommendation 3: Artificial intelligence — Where are we? | Score: 0.663

TOPIC: cybersecurity
Search status: 200
Search results: 20
Search 1: Introduction: Cybersecurity
Search 2: Cybersecurity dynamics
Search 3: Demythifying Cybersecurity
Paper details status: 200

In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/GLAMOR_Project"
)

OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)

report = """
AI-BASED RESEARCH PAPER RECOMMENDATION SYSTEM
USING GLAMOR AND TF-IDF

1. MODEL INFORMATION
--------------------
GLAMOR model: Trained and verified
Base model: DistilRoBERTa
Training platform: Google Colab CPU

2. HYBRID RECOMMENDATION
------------------------
TF-IDF weight: 77.5%
GLAMOR weight: 22.5%
Recommendations per query: 10

3. INDEPENDENT FINAL EVALUATION
--------------------------------
Independent queries selected: 500
Queries evaluated: 389
Queries without relevance labels: 111
Candidate papers: 8,951

Precision@10: 20.98%
Recall@10: 27.96%
MAP@10: 23.79%
nDCG@10: 34.48%
Hit Rate@10: 75.32%

4. PRELIMINARY WEIGHT-TUNING RESULT
-----------------------------------
Hit Rate@10: 80.00%
This is a tuning-set result, not the independent final result.

5. WEBSITE IMPLEMENTATION
-------------------------
Framework: Flask
Database: SQLite
Search: SQLite Full-Text Search (FTS)
Recommendation: TF-IDF + GLAMOR hybrid ranking

6. WEBSITE TESTING
-------------------
Machine Learning: Tested
Artificial Intelligence: Tested
Cybersecurity: Tested
Natural Language Processing: Tested
Data Mining: Tested

Search, paper details and recommendation routes
returned HTTP 200 during the latest tests.

7. IMPORTANT LIMITATIONS
-------------------------
Some recommendations may still be less relevant.
The website uses a limited candidate retrieval process.
Independent evaluation is based on 389 queries with
available relevance labels.

8. CONCLUSION
-------------
The GLAMOR-based hybrid recommendation system has been
trained, independently evaluated and integrated into
the existing Flask website.

The independent Hit Rate@10 is 75.32%.
"""

report_path = OUTPUT / "final_project_results.txt"
report_path.write_text(report, encoding="utf-8")

metrics = pd.DataFrame([
    ["Precision@10", 20.98],
    ["Recall@10", 27.96],
    ["MAP@10", 23.79],
    ["nDCG@10", 34.48],
    ["Hit Rate@10", 75.32]
], columns=["Metric", "Percentage"])

metrics_path = OUTPUT / "final_evaluation_metrics.csv"
metrics.to_csv(metrics_path, index=False)

print("FINAL PROJECT RESULTS SAVED")
print(report_path)
print(metrics_path)
print("\nIndependent Hit Rate@10: 75.32%")


FINAL PROJECT RESULTS SAVED
/content/drive/MyDrive/GLAMOR_Project/outputs/final_project_results.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/final_evaluation_metrics.csv

Independent Hit Rate@10: 75.32%


In [ ]:

import numpy as np

# Check the independent evaluation results
assert "results_ind" in globals(), "Evaluation results are missing."

results = np.asarray(results_ind)

# Each row contains:
# Precision, Recall, MAP, nDCG, Hit Rate
hit_values = results[:, 4]

total_queries = len(hit_values)
successful_queries = int(np.sum(hit_values))

actual_hit_rate = (
    successful_queries / total_queries
) * 100

print("========== HIT RATE VERIFICATION ==========")
print("Total evaluated queries:", total_queries)
print("Queries with at least one relevant recommendation:",
      successful_queries)
print("Queries without a relevant recommendation:",
      total_queries - successful_queries)
print("Calculated Hit Rate@10:", round(actual_hit_rate, 2), "%")

print("\nExpected previous result: 75.32%")
print("Does it match:",
      np.isclose(actual_hit_rate, 75.32, atol=0.02))


========== HIT RATE VERIFICATION ==========
Total evaluated queries: 389
Queries with at least one relevant recommendation: 293
Queries without a relevant recommendation: 96
Calculated Hit Rate@10: 75.32 %

Expected previous result: 75.32%
Does it match: True


In [ ]:
import sqlite3

db_path = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(db_path)

columns = conn.execute(
    "PRAGMA table_info(papers)"
).fetchall()

print("Papers table columns:")
for column in columns:
    print(column[1])

conn.close()

Papers table columns:
paper_id
title
year


In [ ]:
!pip install -q sentence-transformers

In [ ]:

import sqlite3
import numpy as np

db_path = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(db_path)

# Check test_queries table columns
print("TEST QUERIES TABLE:")
for col in conn.execute("PRAGMA table_info(test_queries)").fetchall():
    print(col[1])

# Retrieve query titles from test_queries, not papers
def fetch_query_titles(ids):
    output = {}

    for start in range(0, len(ids), 500):
        batch = [str(x) for x in ids[start:start + 500]]
        placeholders = ",".join(["?"] * len(batch))

        rows = conn.execute(
            f"""
            SELECT CAST(paper_id AS TEXT), title
            FROM test_queries
            WHERE CAST(paper_id AS TEXT) IN ({placeholders})
            """,
            batch
        ).fetchall()

        for paper_id, title in rows:
            if title and str(title).strip():
                output[str(paper_id)] = str(title).strip()

    return output

query_titles = fetch_query_titles(query_ids_ind)

# Candidate titles remain in the training papers table
def fetch_candidate_titles(ids):
    output = {}

    for start in range(0, len(ids), 500):
        batch = [str(x) for x in ids[start:start + 500]]
        placeholders = ",".join(["?"] * len(batch))

        rows = conn.execute(
            f"""
            SELECT CAST(paper_id AS TEXT), title
            FROM papers
            WHERE CAST(paper_id AS TEXT) IN ({placeholders})
            """,
            batch
        ).fetchall()

        for paper_id, title in rows:
            if title and str(title).strip():
                output[str(paper_id)] = str(title).strip()

    return output

candidate_titles = fetch_candidate_titles(candidate_pool)

conn.close()

print("\n===== TITLE RETRIEVAL CHECK =====")
print("Query titles found:", len(query_titles))
print("Candidate titles found:", len(candidate_titles))
print("Expected labelled queries: 389")


TEST QUERIES TABLE:
paper_id
title

===== TITLE RETRIEVAL CHECK =====
Query titles found: 389
Candidate titles found: 8951
Expected labelled queries: 389


In [ ]:

import os
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Prepare IDs and titles in matching order
valid_query_ids = [
    str(q) for q in query_ids_ind
    if str(q) in query_titles
]

valid_candidate_ids = [
    str(c) for c in candidate_pool
    if str(c) in candidate_titles
]

query_texts = [query_titles[q] for q in valid_query_ids]
candidate_texts = [candidate_titles[c] for c in valid_candidate_ids]

print("Queries:", len(query_texts))
print("Candidates:", len(candidate_texts))

# Load pretrained Sentence-BERT
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

# Generate title embeddings
query_embeddings = model.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Calculate semantic similarity
semantic_scores = np.matmul(
    query_embeddings,
    candidate_embeddings.T
)

# Evaluate recommendation quality
def evaluate_top10(ranked_indices, relevant_ids, candidate_ids):
    relevant_ids = set(map(str, relevant_ids))
    recommended = [candidate_ids[i] for i in ranked_indices[:10]]

    hits = [int(p in relevant_ids) for p in recommended]

    precision = sum(hits) / 10
    recall = sum(hits) / len(relevant_ids) if relevant_ids else 0

    relevant_seen = 0
    ap_sum = 0

    for rank, hit in enumerate(hits, 1):
        if hit:
            relevant_seen += 1
            ap_sum += relevant_seen / rank

    ap = (
        ap_sum / min(len(relevant_ids), 10)
        if relevant_ids else 0
    )

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, 1)
    )

    ideal_count = min(len(relevant_ids), 10)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_count + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    return [
        precision,
        recall,
        ap,
        ndcg,
        int(any(hits))
    ]

semantic_results = []

for i, query_id in enumerate(valid_query_ids):

    relevant_ids = set(
        map(str, relevance.get(query_id, set()))
    )

    if not relevant_ids:
        continue

    scores = semantic_scores[i].copy()

    # Exclude the query paper itself
    if query_id in valid_candidate_ids:
        scores[valid_candidate_ids.index(query_id)] = -np.inf

    ranking = np.argsort(-scores)

    semantic_results.append(
        evaluate_top10(
            ranking,
            relevant_ids,
            valid_candidate_ids
        )
    )

semantic_results = np.asarray(semantic_results)

if len(semantic_results) == 0:
    raise RuntimeError("No labelled queries were evaluated.")

# Display calculated metrics
successful = int(semantic_results[:, 4].sum())
total = len(semantic_results)

print("\n===== SENTENCE-BERT RESULTS =====")
print("Evaluated queries:", total)
print("Precision@10: {:.2f}%".format(semantic_results[:, 0].mean() * 100))
print("Recall@10: {:.2f}%".format(semantic_results[:, 1].mean() * 100))
print("MAP@10: {:.2f}%".format(semantic_results[:, 2].mean() * 100))
print("nDCG@10: {:.2f}%".format(semantic_results[:, 3].mean() * 100))
print("Hit Rate@10: {:.2f}%".format(successful / total * 100))
print("Successful queries:", successful, "/", total)

# Save embeddings for later experiments
save_path = (
    "/content/drive/MyDrive/GLAMOR_Project/outputs/"
    "sentence_bert_title_embeddings.npz"
)

np.savez_compressed(
    save_path,
    query_ids=np.array(valid_query_ids),
    candidate_ids=np.array(valid_candidate_ids),
    query_embeddings=query_embeddings,
    candidate_embeddings=candidate_embeddings
)

print("\nEmbeddings saved:", save_path)


Queries: 389
Candidates: 8951
Device: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/280 [00:00<?, ?it/s]


===== SENTENCE-BERT RESULTS =====
Evaluated queries: 389
Precision@10: 29.72%
Recall@10: 40.23%
MAP@10: 36.06%
nDCG@10: 48.04%
Hit Rate@10: 85.35%
Successful queries: 332 / 389

Embeddings saved: /content/drive/MyDrive/GLAMOR_Project/outputs/sentence_bert_title_embeddings.npz


In [ ]:
import os

app_path = "/content/drive/MyDrive/GLAMOR_Project/application/app.py"

print("App file exists:", os.path.exists(app_path))

if os.path.exists(app_path):
    print("App file size:", os.path.getsize(app_path), "bytes")
else:
    print("Flask app.py file not found!")

App file exists: True
App file size: 8710 bytes


In [ ]:
import socket

for port in [5000, 8000, 8080]:
    sock = socket.socket()
    result = sock.connect_ex(("127.0.0.1", port))
    sock.close()

    print(
        f"Port {port}:",
        "RUNNING" if result == 0 else "NOT RUNNING"
    )

Port 5000: RUNNING
Port 8000: NOT RUNNING
Port 8080: RUNNING


In [ ]:
from google.colab.output import eval_js

website_url = eval_js(
    "google.colab.kernel.proxyPort(5000)"
)

print("YOUR WEBSITE LINK:")
print(website_url)

YOUR WEBSITE LINK:
https://5000-m-s-use1d10-4kg8tkn486h0-d.us-east1.prod.r.colab.dev


In [ ]:

import sqlite3

db_path = "/content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite"

conn = sqlite3.connect(db_path)

print("===== DATABASE TABLES =====")

tables = conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""").fetchall()

for (table_name,) in tables:
    print("\nTABLE:", table_name)

    columns = conn.execute(
        f"PRAGMA table_info('{table_name}')"
    ).fetchall()

    print("COLUMNS:")
    for column in columns:
        print(" -", column[1], "|", column[2])

conn.close()


===== DATABASE TABLES =====

TABLE: citations
COLUMNS:
 - source_id | TEXT
 - target_id | TEXT

TABLE: papers
COLUMNS:
 - paper_id | TEXT
 - title | TEXT
 - year | TEXT

TABLE: progress
COLUMNS:
 - task | TEXT
 - last_paper_id | TEXT
 - papers_processed | INTEGER

TABLE: training_ids
COLUMNS:
 - paper_id | TEXT


In [ ]:

import os
import sqlite3

outputs = "/content/drive/MyDrive/GLAMOR_Project/outputs"

print("===== SEARCHING PROJECT DATABASES =====")

for root, dirs, files in os.walk(outputs):
    dirs[:] = [d for d in dirs if d != "__pycache__"]

    for filename in files:
        if filename.endswith((".sqlite", ".db")):
            path = os.path.join(root, filename)

            print("\nDATABASE:", path)

            try:
                conn = sqlite3.connect(path)

                tables = conn.execute("""
                    SELECT name
                    FROM sqlite_master
                    WHERE type='table'
                    ORDER BY name
                """).fetchall()

                for (table,) in tables:
                    print(" TABLE:", table)

                    columns = conn.execute(
                        f"PRAGMA table_info('{table}')"
                    ).fetchall()

                    print(
                        " COLUMNS:",
                        [column[1] for column in columns]
                    )

                conn.close()

            except Exception as e:
                print("ERROR:", e)


===== SEARCHING PROJECT DATABASES =====

DATABASE: /content/drive/MyDrive/GLAMOR_Project/outputs/citation_knowledge_graph.sqlite
 TABLE: citations
 COLUMNS: ['source_id', 'target_id']
 TABLE: papers
 COLUMNS: ['paper_id', 'title', 'year']
 TABLE: progress
 COLUMNS: ['task', 'last_paper_id', 'papers_processed']
 TABLE: training_ids
 COLUMNS: ['paper_id']


In [ ]:

import os
import sqlite3
import numpy as np

project = "/content/drive/MyDrive/GLAMOR_Project"
db_path = project + "/outputs/citation_knowledge_graph.sqlite"

embedding_path = (
    project + "/outputs/sentence_bert_title_embeddings.npz"
)

if not os.path.exists(embedding_path):
    raise FileNotFoundError("Sentence-BERT embeddings not found.")

saved = np.load(embedding_path, allow_pickle=False)

query_ids = [str(x) for x in saved["query_ids"]]
candidate_ids = [str(x) for x in saved["candidate_ids"]]

conn = sqlite3.connect(db_path)

# Check whether test query IDs have outgoing citation relationships
query_source_count = 0
query_target_count = 0
queries_with_references = set()

for start in range(0, len(query_ids), 500):
    batch = query_ids[start:start + 500]
    placeholders = ",".join(["?"] * len(batch))

    rows = conn.execute(
        f"""
        SELECT DISTINCT source_id
        FROM citations
        WHERE source_id IN ({placeholders})
        """,
        batch
    ).fetchall()

    queries_with_references.update(str(row[0]) for row in rows)

    target_rows = conn.execute(
        f"""
        SELECT DISTINCT target_id
        FROM citations
        WHERE target_id IN ({placeholders})
        """,
        batch
    ).fetchall()

    query_target_count += len(target_rows)

# Relevance candidates from outgoing citations of test queries
recovered_relevance = {}

for start in range(0, len(query_ids), 200):
    batch = query_ids[start:start + 200]
    placeholders = ",".join(["?"] * len(batch))

    rows = conn.execute(
        f"""
        SELECT source_id, target_id
        FROM citations
        WHERE source_id IN ({placeholders})
        """,
        batch
    ).fetchall()

    for source_id, target_id in rows:
        recovered_relevance.setdefault(str(source_id), set()).add(
            str(target_id)
        )

conn.close()

print("===== RELEVANCE RECOVERY CHECK =====")
print("Saved test queries:", len(query_ids))
print("Queries with outgoing citations:", len(queries_with_references))
print("Queries with no outgoing citations:",
      len(query_ids) - len(queries_with_references))
print("Queries appearing as citation targets:", query_target_count)

print(
    "Queries with recovered citation labels:",
    len(recovered_relevance)
)

# Check how many recovered relevant papers are in the candidate pool
candidate_set = set(candidate_ids)

covered = sum(
    bool(recovered_relevance.get(q, set()) & candidate_set)
    for q in query_ids
)

print("Queries with relevant candidates available:", covered)
print("Candidate pool size:", len(candidate_ids))


===== RELEVANCE RECOVERY CHECK =====
Saved test queries: 389
Queries with outgoing citations: 0
Queries with no outgoing citations: 389
Queries appearing as citation targets: 0
Queries with recovered citation labels: 0
Queries with relevant candidates available: 0
Candidate pool size: 8951


In [ ]:

import os

root = "/content/drive/MyDrive/GLAMOR_Project"

keywords = [
    "relevance",
    "evaluation",
    "test_query",
    "test_relevance",
    "ground_truth",
    "independent"
]

extensions = (
    ".csv", ".db", ".sqlite", ".json",
    ".pkl", ".parquet", ".txt"
)

print("===== SEARCHING FOR EVALUATION FILES =====")

for folder, dirs, files in os.walk(root):

    dirs[:] = [
        d for d in dirs
        if d not in [".git", "__pycache__"]
    ]

    for filename in files:

        lower = filename.lower()

        if lower.endswith(extensions) and any(
            word in lower for word in keywords
        ):
            print(os.path.join(folder, filename))

print("\n===== SEARCH COMPLETED =====")


===== SEARCHING FOR EVALUATION FILES =====
/content/drive/MyDrive/GLAMOR_Project/outputs/tfidf_500_query_evaluation.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/glamor_500_query_evaluation.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/glamor_fast_500_query_evaluation.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/independent_final_evaluation.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/independent_final_evaluation.csv
/content/drive/MyDrive/GLAMOR_Project/outputs/final_evaluation_metrics.csv

===== SEARCH COMPLETED =====


In [ ]:

import os
import pandas as pd

output_dir = "/content/drive/MyDrive/GLAMOR_Project/outputs"

files = [
    "independent_final_evaluation.txt",
    "independent_final_evaluation.csv",
    "final_evaluation_metrics.csv",
    "tfidf_500_query_evaluation.txt",
    "glamor_500_query_evaluation.txt"
]

for filename in files:
    path = os.path.join(output_dir, filename)

    print("\n" + "=" * 65)
    print("FILE:", filename)
    print("=" * 65)

    if not os.path.exists(path):
        print("File not found")
        continue

    if filename.endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            print(f.read()[:5000])

    elif filename.endswith(".csv"):
        df = pd.read_csv(path)

        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        print("\nFirst 10 rows:")
        print(df.head(10).to_string(index=False))



FILE: independent_final_evaluation.txt
INDEPENDENT FINAL HYBRID EVALUATION
Fixed TF-IDF weight: 77.5%
Fixed GLAMOR weight: 22.5%
Independent queries selected: 500
Queries evaluated: 389
Queries without relevance labels: 111
Candidate papers: 8951

Precision@10: 0.2098 (20.98%)
Recall@10: 0.2796 (27.96%)
MAP@10: 0.2379 (23.79%)
nDCG@10: 0.3448 (34.48%)
Hit Rate@10: 0.7532 (75.32%)

FILE: independent_final_evaluation.csv
Rows: 389
Columns: ['Precision@10', 'Recall@10', 'MAP@10', 'nDCG@10', 'Hit Rate@10']

First 10 rows:
 Precision@10  Recall@10   MAP@10  nDCG@10  Hit Rate@10
          0.2   0.333333 0.208333 0.398063            1
          0.7   0.212121 0.700000 0.800694            1
          0.4   0.444444 0.304233 0.514642            1
          0.3   1.000000 0.866667 0.946902            1
          0.5   0.714286 0.714286 0.810462            1
          0.0   0.000000 0.000000 0.000000            0
          0.2   0.117647 0.200000 0.358954            1
          0.1   0.166667 0.

In [ ]:

import pandas as pd
import os

path = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "outputs/independent_final_evaluation.csv"
)

if not os.path.exists(path):
    print("Independent evaluation CSV not found.")
else:
    df = pd.read_csv(path)

    print("===== INDEPENDENT EVALUATION CSV =====")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    print("\n===== FIRST 15 ROWS =====")
    print(df.head(15).to_string(index=False))

    print("\n===== DATA TYPES =====")
    print(df.dtypes)

    print("\n===== MISSING VALUES =====")
    print(df.isnull().sum())


===== INDEPENDENT EVALUATION CSV =====
Rows: 389
Columns: ['Precision@10', 'Recall@10', 'MAP@10', 'nDCG@10', 'Hit Rate@10']

===== FIRST 15 ROWS =====
 Precision@10  Recall@10   MAP@10  nDCG@10  Hit Rate@10
          0.2   0.333333 0.208333 0.398063            1
          0.7   0.212121 0.700000 0.800694            1
          0.4   0.444444 0.304233 0.514642            1
          0.3   1.000000 0.866667 0.946902            1
          0.5   0.714286 0.714286 0.810462            1
          0.0   0.000000 0.000000 0.000000            0
          0.2   0.117647 0.200000 0.358954            1
          0.1   0.166667 0.166667 0.302602            1
          0.1   0.250000 0.250000 0.390380            1
          0.1   0.142857 0.023810 0.097913            1
          0.8   0.500000 0.704206 0.836459            1
          0.0   0.000000 0.000000 0.000000            0
          0.5   0.714286 0.660714 0.790839            1
          0.2   0.333333 0.277778 0.453904            1
         

In [ ]:

import os

root = "/content/drive/MyDrive/GLAMOR_Project"

extensions = (
    ".csv", ".sqlite", ".db", ".json",
    ".jsonl", ".pkl", ".parquet", ".npz"
)

keywords = [
    "relevance", "recommendation", "prediction",
    "ranking", "query", "ground_truth",
    "result", "evaluation"
]

print("===== SEARCHING FOR ORIGINAL EVALUATION DATA =====")

for folder, dirs, files in os.walk(root):
    dirs[:] = [
        d for d in dirs
        if d not in [".git", "__pycache__"]
    ]

    for filename in files:
        lower = filename.lower()

        if lower.endswith(extensions) and any(
            word in lower for word in keywords
        ):
            print(os.path.join(folder, filename))

print("\nSearch completed.")


===== SEARCHING FOR ORIGINAL EVALUATION DATA =====
/content/drive/MyDrive/GLAMOR_Project/outputs/hybrid_recommendation_comparison.csv
/content/drive/MyDrive/GLAMOR_Project/outputs/independent_final_evaluation.csv
/content/drive/MyDrive/GLAMOR_Project/outputs/final_evaluation_metrics.csv
/content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/training_result.json

Search completed.


In [ ]:

import os
import pandas as pd
import json

output_dir = "/content/drive/MyDrive/GLAMOR_Project/outputs"

# Inspect hybrid comparison CSV
csv_path = os.path.join(
    output_dir,
    "hybrid_recommendation_comparison.csv"
)

print("===== HYBRID RECOMMENDATION COMPARISON =====")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("\nComplete contents:")
    print(df.to_string(index=False))
else:
    print("Hybrid comparison file not found.")

# Inspect training result JSON
json_path = os.path.join(
    output_dir,
    "cpu_fast_experiment",
    "training_result.json"
)

print("\n===== TRAINING RESULT =====")

if os.path.exists(json_path):
    with open(json_path, "r") as f:
        training_data = json.load(f)

    print(json.dumps(training_data, indent=2))
else:
    print("Training result JSON not found.")


===== HYBRID RECOMMENDATION COMPARISON =====
Rows: 5
Columns: ['TF-IDF Weight', 'GLAMOR Weight', 'Precision@10', 'Recall@10', 'MAP@10', 'nDCG@10', 'Hit Rate@10']

Complete contents:
 TF-IDF Weight  GLAMOR Weight  Precision@10  Recall@10   MAP@10  nDCG@10  Hit Rate@10
          1.00           0.00        0.2354   0.301107 0.276741 0.388270        0.790
          0.75           0.25        0.2382   0.303517 0.281633 0.392651        0.796
          0.50           0.50        0.2302   0.296412 0.273497 0.384716        0.794
          0.25           0.75        0.1830   0.242047 0.212933 0.319147        0.748
          0.00           1.00        0.0586   0.078733 0.054506 0.100526        0.378

===== TRAINING RESULT =====
{
  "training_walks": 500,
  "validation_walks": 100,
  "epochs": 1,
  "batch_size": 2,
  "max_length": 64,
  "training_time_minutes": 9.2,
  "model_directory": "/content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/trained_model"
}


In [ ]:

import os

root = "/content/drive/MyDrive/GLAMOR_Project"

extensions = (
    ".csv", ".sqlite", ".db", ".json",
    ".jsonl", ".pkl", ".parquet", ".npz",
    ".xlsx", ".txt"
)

keywords = [
    "relevance", "query", "test", "ground",
    "evaluation", "recommendation", "result"
]

print("===== SEARCHING ENTIRE GLAMOR PROJECT =====")

found = []

for folder, dirs, files in os.walk(root):

    # Corrected directory exclusion
    dirs[:] = [
        d for d in dirs
        if d not in (".git", "__pycache__")
    ]

    for filename in files:
        lower = filename.lower()

        if lower.endswith(extensions):
            if any(word in lower for word in keywords):
                found.append(os.path.join(folder, filename))

for path in sorted(found):
    print(path)

print("\nTotal matching files:", len(found))
print("Search completed.")


===== SEARCHING ENTIRE GLAMOR PROJECT =====
/content/drive/MyDrive/GLAMOR_Project/GLAMOR/artifacts/glamor_dblp_v12/test_ids.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/cpu_fast_experiment/training_result.json
/content/drive/MyDrive/GLAMOR_Project/outputs/final_evaluation_metrics.csv
/content/drive/MyDrive/GLAMOR_Project/outputs/final_project_results.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/glamor_500_query_evaluation.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/glamor_fast_500_query_evaluation.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/hybrid_recommendation_comparison.csv
/content/drive/MyDrive/GLAMOR_Project/outputs/hybrid_recommendation_comparison.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/independent_final_evaluation.csv
/content/drive/MyDrive/GLAMOR_Project/outputs/independent_final_evaluation.txt
/content/drive/MyDrive/GLAMOR_Project/outputs/tfidf_500_query_evaluation.txt

Total matching files: 11
Search completed.


In [ ]:

import os

files = [
    "/content/drive/MyDrive/GLAMOR_Project/GLAMOR/artifacts/glamor_dblp_v12/test_ids.txt",
    "/content/drive/MyDrive/GLAMOR_Project/outputs/final_project_results.txt",
    "/content/drive/MyDrive/GLAMOR_Project/outputs/independent_final_evaluation.txt"
]

for path in files:
    print("\n" + "=" * 65)
    print("FILE:", os.path.basename(path))
    print("=" * 65)

    if not os.path.exists(path):
        print("File not found")
        continue

    print("File size:", os.path.getsize(path), "bytes")

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        content = f.read()

    print("Total lines:", len(content.splitlines()))
    print("\nFirst 30 lines:")
    print("\n".join(content.splitlines()[:30]))



FILE: test_ids.txt
File size: 11062236 bytes
Total lines: 921853

First 30 lines:
10000000004
10000000006
10000000009
10000000011
10000000012
10000000015
10000000016
10000000018
10000000024
10000000026
10000000029
10000000039
10000000043
10000000045
10000000048
10000000049
10000000051
10000000053
10000000055
10000000057
10000000058
10000000076
10000000080
10000000081
10000000085
10000000086
10000000091
10000000092
10000000109
10000000114

FILE: final_project_results.txt
File size: 1647 bytes
Total lines: 66

First 30 lines:

AI-BASED RESEARCH PAPER RECOMMENDATION SYSTEM
USING GLAMOR AND TF-IDF

1. MODEL INFORMATION
--------------------
GLAMOR model: Trained and verified
Base model: DistilRoBERTa
Training platform: Google Colab CPU

2. HYBRID RECOMMENDATION
------------------------
TF-IDF weight: 77.5%
GLAMOR weight: 22.5%
Recommendations per query: 10

3. INDEPENDENT FINAL EVALUATION
--------------------------------
Independent queries selected: 500
Queries evaluated: 389
Queries with

In [ ]:

import os

path = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "GLAMOR/artifacts/glamor_dblp_v12/test_ids.txt"
)

if not os.path.exists(path):
    raise FileNotFoundError("test_ids.txt not found")

with open(path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

print("===== TEST IDS FILE =====")
print("Total IDs:", len(lines))
print("First 20 IDs:")

for item in lines[:20]:
    print(item)

print("\nLast 10 IDs:")
for item in lines[-10:]:
    print(item)


===== TEST IDS FILE =====
Total IDs: 921853
First 20 IDs:
10000000004
10000000006
10000000009
10000000011
10000000012
10000000015
10000000016
10000000018
10000000024
10000000026
10000000029
10000000039
10000000043
10000000045
10000000048
10000000049
10000000051
10000000053
10000000055
10000000057

Last 10 IDs:
10004609223
10004609224
10004609231
10004609232
10004609238
10004609240
10004609241
10004609256
10004609257
10004609264


In [ ]:

import os

artifact_dir = (
    "/content/drive/MyDrive/GLAMOR_Project/"
    "GLAMOR/artifacts/glamor_dblp_v12"
)

print("===== GLAMOR ARTIFACT FILES =====")

for root, dirs, files in os.walk(artifact_dir):
    dirs[:] = [
        d for d in dirs
        if d not in (".git", "__pycache__")
    ]

    for filename in sorted(files):
        path = os.path.join(root, filename)
        print(
            filename,
            "|",
            os.path.getsize(path),
            "bytes"
        )

print("\n===== SEARCH COMPLETED =====")


===== GLAMOR ARTIFACT FILES =====
citation_knowledge_graph.sqlite-shm | 2686976 bytes
test_ids.txt | 11062236 bytes
train_ids.txt | 44248944 bytes

===== SEARCH COMPLETED =====


In [ ]:

import os

dataset_dir = (
    "/content/drive/MyDrive/GLAMOR_Project/dataset"
)

print("===== ORIGINAL DATASET FILES =====")

if not os.path.exists(dataset_dir):
    print("Dataset folder not found.")
else:
    for root, dirs, files in os.walk(dataset_dir):
        dirs[:] = [
            d for d in dirs
            if d not in (".git", "__pycache__")
        ]

        for filename in sorted(files):
            path = os.path.join(root, filename)

            print(
                filename,
                "|",
                round(os.path.getsize(path) / (1024 * 1024), 2),
                "MB"
            )

print("\nSearch completed.")


===== ORIGINAL DATASET FILES =====
citation_network.csv | 1075.0 MB
citation_network_ready.csv | 1168.52 MB
citation_network_repaired.csv | 1079.4 MB
citation_network_unique.csv | 1127.75 MB
dblp_v12.txt | 988.3 MB
dblp_v12_ready.txt | 1033.87 MB
paper_id_mapping.csv | 100.63 MB

Search completed.


In [ ]:

import os
import pandas as pd

dataset_dir = (
    "/content/drive/MyDrive/GLAMOR_Project/dataset"
)

files = [
    "citation_network.csv",
    "citation_network_repaired.csv",
    "citation_network_unique.csv",
    "paper_id_mapping.csv"
]

print("===== ORIGINAL DATASET STRUCTURE =====")

for filename in files:
    path = os.path.join(dataset_dir, filename)

    if os.path.exists(path):
        print("\nFILE:", filename)

        sample = pd.read_csv(
            path,
            nrows=5,
            dtype=str,
            low_memory=False
        )

        print("Columns:", list(sample.columns))
        print("Sample records:")
        print(sample.to_string(index=False))
    else:
        print("Missing:", filename)


===== ORIGINAL DATASET STRUCTURE =====

FILE: citation_network.csv
Columns: ['id', 'title', 'year', 'n_citation', 'doc_type', 'reference_count', 'references', 'doi']
Sample records:
  id                                                                                                                                                           title   year n_citation   doc_type reference_count                                                                                                                                                           references                                          doi
1091 Preliminary Design of a Network Protocol Learning Tool Based on the Comprehension of High School Students: Design by an Empirical Study Using a Simple Mind Map 2013.0          1 Conference             2.0                                                                                                                                                2005687710;2018037215 https://doi.org/10.1007

In [ ]:

# ============================================================
# RECOVER TEST RELEVANCE LABELS FROM ORIGINAL DATASET
# ============================================================

import os
import pandas as pd
import numpy as np

project = "/content/drive/MyDrive/GLAMOR_Project"
dataset = project + "/dataset"
output = project + "/outputs"

unique_path = dataset + "/citation_network_unique.csv"
mapping_path = dataset + "/paper_id_mapping.csv"
embedding_path = output + "/sentence_bert_title_embeddings.npz"

# 1. LOAD SAVED TEST QUERY AND CANDIDATE IDS
if not os.path.exists(embedding_path):
    raise FileNotFoundError("Saved Sentence-BERT embeddings not found.")

saved = np.load(embedding_path, allow_pickle=False)

query_ids = [str(x) for x in saved["query_ids"]]
candidate_ids = set(str(x) for x in saved["candidate_ids"])

print("Test queries:", len(query_ids))
print("Candidate papers:", len(candidate_ids))

query_set = set(query_ids)

# 2. FIND ORIGINAL REFERENCES FOR TEST QUERIES
query_references = {}

print("\nReading original citation dataset...")

for chunk in pd.read_csv(
    unique_path,
    usecols=["id", "references"],
    dtype=str,
    chunksize=100000
):
    matching = chunk[chunk["id"].isin(query_set)]

    for _, row in matching.iterrows():
        query_id = str(row["id"])
        refs = row["references"]

        if pd.notna(refs) and str(refs).strip():
            reference_ids = {
                x.strip()
                for x in str(refs).split(";")
                if x.strip() and x.strip().lower() != "nan"
            }

            query_references.setdefault(
                query_id, set()
            ).update(reference_ids)

print("Queries with original references:", len(query_references))

# 3. COLLECT ALL ORIGINAL REFERENCE IDS
original_reference_ids = set()

for refs in query_references.values():
    original_reference_ids.update(refs)

print("Distinct original reference IDs:", len(original_reference_ids))

# 4. MAP ORIGINAL REFERENCE IDS TO PROJECT PAPER IDS
original_to_project_id = {}

print("\nReading paper ID mapping...")

for chunk in pd.read_csv(
    mapping_path,
    usecols=["original_id", "id"],
    dtype=str,
    chunksize=100000
):
    matching = chunk[
        chunk["original_id"].isin(original_reference_ids)
    ]

    for _, row in matching.iterrows():
        original_to_project_id[str(row["original_id"])] = (
            str(row["id"])
        )

print("Reference IDs successfully mapped:", len(original_to_project_id))

# 5. RECONSTRUCT RELEVANCE LABELS
recovered_relevance = {}

for query_id, original_refs in query_references.items():

    mapped_refs = {
        original_to_project_id[ref]
        for ref in original_refs
        if ref in original_to_project_id
    }

    # Keep only papers present in the evaluation candidate pool
    relevant_candidates = mapped_refs.intersection(candidate_ids)

    if relevant_candidates:
        recovered_relevance[query_id] = relevant_candidates

# 6. REPORT RECOVERY RESULTS
labelled_count = len(recovered_relevance)

total_relevant_pairs = sum(
    len(refs) for refs in recovered_relevance.values()
)

print("\n===== RELEVANCE RECOVERY RESULTS =====")
print("Total test queries:", len(query_ids))
print("Queries with original references:", len(query_references))
print("Queries with candidate relevance labels:", labelled_count)
print("Queries without candidate relevance:", len(query_ids) - labelled_count)
print("Total recovered relevant query-paper pairs:", total_relevant_pairs)

# 7. SAVE RECOVERED LABELS
save_path = output + "/recovered_test_relevance.csv"

records = [
    {"query_id": q, "relevant_paper_id": paper}
    for q, refs in recovered_relevance.items()
    for paper in sorted(refs)
]

pd.DataFrame(
    records,
    columns=["query_id", "relevant_paper_id"]
).to_csv(save_path, index=False)

print("\nRecovered relevance labels saved to:")
print(save_path)


Test queries: 389
Candidate papers: 8951

Reading original citation dataset...
Queries with original references: 389
Distinct original reference IDs: 4701

Reading paper ID mapping...
Reference IDs successfully mapped: 4464

===== RELEVANCE RECOVERY RESULTS =====
Total test queries: 389
Queries with original references: 389
Queries with candidate relevance labels: 389
Queries without candidate relevance: 0
Total recovered relevant query-paper pairs: 3612

Recovered relevance labels saved to:
/content/drive/MyDrive/GLAMOR_Project/outputs/recovered_test_relevance.csv


In [ ]:

# =====================================================
# RECOVER QUERY AND CANDIDATE TITLES FROM ORIGINAL CSV
# =====================================================

import os
import pandas as pd
import numpy as np

project = "/content/drive/MyDrive/GLAMOR_Project"
unique_path = project + "/dataset/citation_network_unique.csv"

# Check previous variables
if "queries" not in globals() or "candidates" not in globals():
    raise RuntimeError(
        "Query and candidate IDs are missing. "
        "Restore the Sentence-BERT embeddings first."
    )

query_ids_set = set(map(str, queries))
candidate_ids_set = set(map(str, candidates))

all_needed_ids = query_ids_set | candidate_ids_set

query_titles = {}
candidate_titles = {}

print("Queries:", len(query_ids_set))
print("Candidates:", len(candidate_ids_set))
print("\nReading original dataset...")

# Read original dataset in chunks
for chunk in pd.read_csv(
    unique_path,
    usecols=["id", "title"],
    dtype=str,
    chunksize=100000
):
    chunk["id"] = chunk["id"].astype(str)

    matching = chunk[chunk["id"].isin(all_needed_ids)]

    for _, row in matching.iterrows():
        paper_id = str(row["id"])
        title = row["title"]

        if pd.notna(title) and str(title).strip():
            if paper_id in query_ids_set:
                query_titles[paper_id] = str(title).strip()

            if paper_id in candidate_ids_set:
                candidate_titles[paper_id] = str(title).strip()

print("\n===== TITLE RECOVERY RESULTS =====")
print("Query titles found:", len(query_titles))
print("Candidate titles found:", len(candidate_titles))

print("\nSample query titles:")
for k, v in list(query_titles.items())[:5]:
    print(k, ":", v)

print("\nSample candidate titles:")
for k, v in list(candidate_titles.items())[:5]:
    print(k, ":", v)

# Stop if titles are not recovered
if not query_titles or not candidate_titles:
    raise RuntimeError(
        "Titles were not recovered. Check the ID mapping."
    )

print("\nSUCCESS: Titles recovered from original dataset.")


Queries: 389
Candidates: 8951

Reading original dataset...

===== TITLE RECOVERY RESULTS =====
Query titles found: 389
Candidate titles found: 8951

Sample query titles:
10000001915 : An Integrated Test Environment Process Model to Control Software Failures.
10000065838 : Boosting the Detection of Transposable Elements Using Machine Learning
10000078031 : Automatic pronunciation scoring of specific phone segments for language instruction.
10000089101 : Determining Reliability of Subjective and Multi-label Emotion Annotation through Novel Fuzzy Agreement Measure
10000154356 : Limit Cycle for Composited Cellar Automata

Sample candidate titles:
10000002161 : Misuse-Resistant Variants of the OMD Authenticated Encryption Mode
10000002217 : Blind identity-based encryption and simulatable oblivious transfer
10000002827 : Analysis and Synthesis of Structural Textures.
10000004433 : Cost Effectiveness of Open Systems Versus Proprietary Systems.
10000005158 : Steiner Systems for Topology-Transp

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. PREPARE IDS AND TITLES

queries = [str(q) for q in queries]
candidates = [str(c) for c in candidates]

query_map = {q: i for i, q in enumerate(queries)}
candidate_map = {c: i for i, c in enumerate(candidates)}

valid_queries = [q for q in queries if query_titles.get(q)]
valid_candidates = [c for c in candidates if candidate_titles.get(c)]

query_texts = [query_titles[q] for q in valid_queries]
candidate_texts = [candidate_titles[c] for c in valid_candidates]

print("Queries:", len(valid_queries))
print("Candidates:", len(valid_candidates))

# 2. CHARACTER-LEVEL TF-IDF

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=1,
    max_features=100000,
    sublinear_tf=True,
    dtype=np.float32
)

candidate_tfidf = vectorizer.fit_transform(candidate_texts)
query_tfidf = vectorizer.transform(query_texts)

similarity = cosine_similarity(
    query_tfidf,
    candidate_tfidf,
    dense_output=True
)

# Align similarity matrix with all IDs
tfidf_scores = np.zeros(
    (len(queries), len(candidates)),
    dtype=np.float32
)

qi = np.array([query_map[q] for q in valid_queries])
ci = np.array([candidate_map[c] for c in valid_candidates])

tfidf_scores[np.ix_(qi, ci)] = similarity

print("TF-IDF matrix:", tfidf_scores.shape)

# 3. RESTORE SEMANTIC SCORES

saved = np.load(
    "/content/drive/MyDrive/GLAMOR_Project/outputs/"
    "sentence_bert_title_embeddings.npz",
    allow_pickle=False
)

saved_queries = [str(x) for x in saved["query_ids"]]
saved_candidates = [str(x) for x in saved["candidate_ids"]]

if saved_queries != queries or saved_candidates != candidates:
    raise ValueError(
        "Embedding ID order differs from current query/candidate order."
    )

semantic_scores = np.asarray(
    saved["query_embeddings"] @ saved["candidate_embeddings"].T,
    dtype=np.float32
)

# 4. CONVERT BOTH SCORES TO RANK PERCENTILES

def rank_percentiles(matrix):
    output = np.zeros(matrix.shape, dtype=np.float32)
    n = matrix.shape[1]

    for i, row in enumerate(matrix):
        order = np.argsort(-row, kind="stable")
        ranks = np.empty(n, dtype=np.int32)
        ranks[order] = np.arange(n)
        output[i] = 1.0 - ranks / max(n - 1, 1)

    return output

semantic_rank = rank_percentiles(semantic_scores)
tfidf_rank = rank_percentiles(tfidf_scores)

# 5. LOAD RECOVERED RELEVANCE

rel_path = (
    "/content/drive/MyDrive/GLAMOR_Project/outputs/"
    "recovered_test_relevance.csv"
)

rel_df = pd.read_csv(rel_path, dtype=str)

relevance = {}

for q, p in zip(
    rel_df["query_id"],
    rel_df["relevant_paper_id"]
):
    relevance.setdefault(str(q), set()).add(str(p))

labelled_queries = [
    q for q in queries if relevance.get(q)
]

print("Queries with relevance:", len(labelled_queries))

# 6. METRICS

def calculate_metrics(scores, q):
    row = scores[query_map[q]].copy()
    relevant = relevance[q]

    if q in candidate_map:
        row[candidate_map[q]] = -np.inf

    top = np.argsort(-row, kind="stable")[:10]
    recommended = [candidates[i] for i in top]

    hits = [int(p in relevant) for p in recommended]

    precision = sum(hits) / 10
    recall = sum(hits) / len(relevant)

    seen = 0
    ap_sum = 0

    for rank, hit in enumerate(hits, 1):
        if hit:
            seen += 1
            ap_sum += seen / rank

    ap = ap_sum / min(len(relevant), 10)

    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, 1)
    )

    ideal = min(len(relevant), 10)

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    return [precision, recall, ap, ndcg, int(any(hits))]

def evaluate(scores, selected):
    values = np.array([
        calculate_metrics(scores, q)
        for q in selected
    ])

    return values.mean(axis=0), int(values[:, 4].sum())

# 7. VALIDATION AND HOLDOUT SPLIT

rng = np.random.default_rng(42)
shuffled = np.array(labelled_queries)
rng.shuffle(shuffled)

split = int(0.80 * len(shuffled))

validation = list(shuffled[:split])
holdout = list(shuffled[split:])

print("Validation queries:", len(validation))
print("Holdout queries:", len(holdout))

# 8. TEST 21 WEIGHT COMBINATIONS

experiments = []

for semantic_weight in np.arange(0, 1.01, 0.05):

    tfidf_weight = 1.0 - semantic_weight

    combined = (
        semantic_weight * semantic_rank
        + tfidf_weight * tfidf_rank
    )

    metrics, hits = evaluate(combined, validation)

    experiments.append({
        "Sentence_BERT_weight": round(float(semantic_weight), 2),
        "TFIDF_weight": round(float(tfidf_weight), 2),
        "Validation_Hit_Rate": hits / len(validation),
        "Validation_nDCG": metrics[3]
    })

best = max(
    experiments,
    key=lambda x: (
        x["Validation_Hit_Rate"],
        x["Validation_nDCG"]
    )
)

# 9. FINAL HOLDOUT EVALUATION

final_scores = (
    best["Sentence_BERT_weight"] * semantic_rank
    + best["TFIDF_weight"] * tfidf_rank
)

semantic_metrics, semantic_hits = evaluate(
    semantic_rank, holdout
)

hybrid_metrics, hybrid_hits = evaluate(
    final_scores, holdout
)

print("\n===== BEST VALIDATION WEIGHTS =====")
print(best)

print("\n===== HOLDOUT RESULTS =====")

print(
    "Sentence-BERT Hit Rate@10:",
    round(100 * semantic_hits / len(holdout), 2), "%"
)

print(
    "Hybrid Hit Rate@10:",
    round(100 * hybrid_hits / len(holdout), 2), "%"
)

print("\n===== HYBRID METRICS =====")
print("Precision@10:", round(hybrid_metrics[0] * 100, 2), "%")
print("Recall@10:", round(hybrid_metrics[1] * 100, 2), "%")
print("MAP@10:", round(hybrid_metrics[2] * 100, 2), "%")
print("nDCG@10:", round(hybrid_metrics[3] * 100, 2), "%")

# 10. SAVE RESULTS

output_dir = (
    "/content/drive/MyDrive/GLAMOR_Project/outputs"
)

pd.DataFrame(experiments).to_csv(
    output_dir + "/sentence_bert_tfidf_optimization.csv",
    index=False
)

print("\nExperiment completed.")
print("Weight results saved successfully.")


Queries: 389
Candidates: 8951
TF-IDF matrix: (389, 8951)
Queries with relevance: 389
Validation queries: 311
Holdout queries: 78

===== BEST VALIDATION WEIGHTS =====
{'Sentence_BERT_weight': 1.0, 'TFIDF_weight': 0.0, 'Validation_Hit_Rate': 0.8488745980707395, 'Validation_nDCG': np.float64(0.48599911692991665)}

===== HOLDOUT RESULTS =====
Sentence-BERT Hit Rate@10: 87.18 %
Hybrid Hit Rate@10: 87.18 %

===== HYBRID METRICS =====
Precision@10: 31.67 %
Recall@10: 35.84 %
MAP@10: 33.23 %
nDCG@10: 45.82 %

Experiment completed.
Weight results saved successfully.


In [ ]:
import pandas as pd
import numpy as np

rrf_scores = experiments["RRF_10"]

rescued = []
lost = []

for q in validation:

    baseline_row = semantic_rank[query_map[q]].copy()
    rrf_row = rrf_scores[query_map[q]].copy()

    if q in candidate_map:
        baseline_row[candidate_map[q]] = -np.inf
        rrf_row[candidate_map[q]] = -np.inf

    baseline_top = np.argsort(-baseline_row)[:10]
    rrf_top = np.argsort(-rrf_row)[:10]

    relevant = relevance[q]

    baseline_hit = any(
        candidates[i] in relevant for i in baseline_top
    )

    rrf_hit = any(
        candidates[i] in relevant for i in rrf_top
    )

    record = {
        "query_id": q,
        "title": query_titles.get(q, ""),
        "relevant_papers": len(relevant),
        "semantic_hit": int(baseline_hit),
        "rrf_hit": int(rrf_hit)
    }

    if not baseline_hit and rrf_hit:
        rescued.append(record)

    elif baseline_hit and not rrf_hit:
        lost.append(record)

print("===== RRF RESCUED QUERIES =====")
print("Count:", len(rescued))

for item in rescued:
    print(item["query_id"], "|", item["title"])

print("\n===== RRF LOST QUERIES =====")
print("Count:", len(lost))

for item in lost:
    print(item["query_id"], "|", item["title"])

# Save both groups
out = "/content/drive/MyDrive/GLAMOR_Project/outputs/"

pd.DataFrame(rescued).to_csv(
    out + "rrf_rescued_queries.csv", index=False
)

pd.DataFrame(lost).to_csv(
    out + "rrf_lost_queries.csv", index=False
)

print("\nBoth files saved successfully.")


===== RRF RESCUED QUERIES =====
Count: 8
10002642399 | Institutional Factors influencing E-Business Adoption
10002096707 | Using Discovery and Monitoring Services to Support Context-Aware Remote Assisted Living Applications
10002244113 | Performance of a traffic smoothing switching element
10004319840 | Towards Generalizing Sensorimotor Control Across Weather Conditions.
10001104369 | Integrated systems based on behaviors
10001124358 | E-ODMRP: Enhanced ODMRP with motion adaptive refresh
10002588576 | Coordinating distributed software development projects with integrated process modelling and enactment environments
10002551712 | Pupil Configuration for Extended Source Imaging with Optical Interferometry: a Computational Geometry Approach

===== RRF LOST QUERIES =====
Count: 5
10002667685 | Flexible and Dynamic Integration of Multiple Information Bases
10001550556 | Unified modeling and analysis of a proportional valve
10003380626 | “One Size Doesn’t Fit All": Helping Users Find Events 

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Existing ranking scores
rrf_scores = experiments["RRF_10"]

# Build query-level training features
def extract_features(q):

    i = query_map[q]

    s = semantic_rank[i]
    t = tfidf_rank[i]

    s_top = np.sort(s)[-20:]
    t_top = np.sort(t)[-20:]

    return [
        np.mean(s_top[:10]),
        np.mean(s_top[-10:]),
        np.std(s),
        np.max(s),
        np.mean(t_top[:10]),
        np.mean(t_top[-10:]),
        np.std(t),
        np.max(t),
        np.mean(np.abs(s - t)),
        len(query_titles.get(q, "").split())
    ]

# Label: 1 when RRF succeeds and Sentence-BERT fails
# Label: 0 otherwise
X = []
y = []

for q in validation:

    semantic_hit = query_hit(semantic_rank, q)
    rrf_hit = query_hit(rrf_scores, q)

    X.append(extract_features(q))
    y.append(int(rrf_hit > semantic_hit))

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int32)

print("Training queries:", len(X))
print("Queries where RRF helps:", int(y.sum()))

# Split validation data for internal model checking
X_train, X_check, y_train, y_check = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train adaptive selector
selector = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)

selector.fit(X_train, y_train)

predicted = selector.predict(X_check)

print("\n===== ADAPTIVE SELECTOR CHECK =====")
print(classification_report(
    y_check,
    predicted,
    zero_division=0
))

# Refit selector on all validation queries
selector.fit(X, y)

# Apply adaptive ranking to holdout queries
adaptive_scores = np.zeros_like(semantic_rank)

selected_rrf = 0
selected_semantic = 0

for q in holdout:

    features = np.asarray(
        extract_features(q),
        dtype=np.float32
    ).reshape(1, -1)

    use_rrf = selector.predict(features)[0] == 1

    i = query_map[q]

    if use_rrf:
        adaptive_scores[i] = rrf_scores[i]
        selected_rrf += 1
    else:
        adaptive_scores[i] = semantic_rank[i]
        selected_semantic += 1

# Evaluate holdout
adaptive_metrics, adaptive_hits = evaluate(
    adaptive_scores,
    holdout
)

baseline_metrics, baseline_hits = evaluate(
    semantic_rank,
    holdout
)

print("\n===== ADAPTIVE HOLDOUT RESULTS =====")
print("Queries using RRF:", selected_rrf)
print("Queries using Sentence-BERT:", selected_semantic)

print(
    "Sentence-BERT Hit Rate@10:",
    round(baseline_hits / len(holdout) * 100, 2), "%"
)

print(
    "Adaptive Hit Rate@10:",
    round(adaptive_hits / len(holdout) * 100, 2), "%"
)

print("\nSuccessful adaptive queries:",
      adaptive_hits, "/", len(holdout))

# Save the selector and results
import joblib

output_dir = (
    "/content/drive/MyDrive/GLAMOR_Project/outputs"
)

joblib.dump(
    selector,
    output_dir + "/adaptive_ranking_selector.pkl"
)

print("\nAdaptive selector saved.")


Training queries: 311
Queries where RRF helps: 8

===== ADAPTIVE SELECTOR CHECK =====
              precision    recall  f1-score   support

           0       0.97      0.97      0.97        61
           1       0.00      0.00      0.00         2

    accuracy                           0.94        63
   macro avg       0.48      0.48      0.48        63
weighted avg       0.94      0.94      0.94        63


===== ADAPTIVE HOLDOUT RESULTS =====
Queries using RRF: 4
Queries using Sentence-BERT: 74
Sentence-BERT Hit Rate@10: 87.18 %
Adaptive Hit Rate@10: 87.18 %

Successful adaptive queries: 68 / 78

Adaptive selector saved.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. VERIFY EXISTING DATA

required = [
    "queries", "candidates", "query_titles",
    "candidate_titles", "query_map", "candidate_map",
    "semantic_rank", "relevance", "validation", "holdout"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing variables: " + ", ".join(missing)
    )

valid_q = [q for q in queries if query_titles.get(q)]
valid_c = [c for c in candidates if candidate_titles.get(c)]

q_texts = [query_titles[q] for q in valid_q]
c_texts = [candidate_titles[c] for c in valid_c]

q_indices = [query_map[q] for q in valid_q]
c_indices = [candidate_map[c] for c in valid_c]

# 2. WORD-LEVEL TF-IDF
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
    min_df=1,
    max_features=100000,
    sublinear_tf=True,
    dtype=np.float32
)

word_c = word_vectorizer.fit_transform(c_texts)
word_q = word_vectorizer.transform(q_texts)

word_similarity = cosine_similarity(word_q, word_c)

word_scores = np.zeros(
    (len(queries), len(candidates)),
    dtype=np.float32
)

word_scores[np.ix_(q_indices, c_indices)] = word_similarity

print("Word TF-IDF completed:", word_scores.shape)

# 3. CHARACTER-LEVEL TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=1,
    max_features=100000,
    sublinear_tf=True,
    dtype=np.float32
)

char_c = char_vectorizer.fit_transform(c_texts)
char_q = char_vectorizer.transform(q_texts)

char_similarity = cosine_similarity(char_q, char_c)

char_scores = np.zeros(
    (len(queries), len(candidates)),
    dtype=np.float32
)

char_scores[np.ix_(q_indices, c_indices)] = char_similarity

print("Character TF-IDF completed:", char_scores.shape)

# 4. CONVERT SIMILARITIES TO RANK PERCENTILES
def to_rank_percentiles(matrix):
    result = np.zeros(matrix.shape, dtype=np.float32)
    n = matrix.shape[1]

    for i, row in enumerate(matrix):
        order = np.argsort(-row, kind="stable")
        ranks = np.empty(n, dtype=np.int32)
        ranks[order] = np.arange(n)
        result[i] = 1 - ranks / max(n - 1, 1)

    return result

word_rank = to_rank_percentiles(word_scores)
char_rank = to_rank_percentiles(char_scores)

# 5. EVALUATION FUNCTION
def hit_count(scores, selected_queries):
    successful = 0

    for q in selected_queries:
        row = scores[query_map[q]].copy()

        if q in candidate_map:
            row[candidate_map[q]] = -np.inf

        top = np.argsort(-row, kind="stable")[:10]
        recommended = {candidates[i] for i in top}

        if recommended.intersection(relevance[q]):
            successful += 1

    return successful

# 6. TEST WEIGHT COMBINATIONS ON VALIDATION ONLY
results = []

for semantic_w in [0.60, 0.70, 0.80, 0.90]:
    for word_w in [0.05, 0.10, 0.15, 0.20]:
        for char_w in [0.05, 0.10, 0.15, 0.20]:

            total = semantic_w + word_w + char_w

            if total > 1.0001:
                continue

            scores = (
                semantic_w * semantic_rank
                + word_w * word_rank
                + char_w * char_rank
            )

            hits = hit_count(scores, validation)

            results.append({
                "Semantic_weight": semantic_w,
                "Word_weight": word_w,
                "Character_weight": char_w,
                "Validation_hits": hits,
                "Validation_total": len(validation),
                "Validation_Hit_Rate": hits / len(validation)
            })

results_df = pd.DataFrame(results).sort_values(
    "Validation_Hit_Rate",
    ascending=False
)

print("\n===== TOP 10 WORD + CHARACTER HYBRIDS =====")
print(results_df.head(10).to_string(index=False))

# 7. SAVE VALIDATION RESULTS
output_dir = (
    "/content/drive/MyDrive/GLAMOR_Project/outputs"
)

results_df.to_csv(
    output_dir + "/word_character_hybrid_results.csv",
    index=False
)

print("\nResults saved successfully.")


Word TF-IDF completed: (389, 8951)
Character TF-IDF completed: (389, 8951)

===== TOP 10 WORD + CHARACTER HYBRIDS =====
 Semantic_weight  Word_weight  Character_weight  Validation_hits  Validation_total  Validation_Hit_Rate
             0.7         0.10              0.05              256               311             0.823151
             0.9         0.05              0.05              256               311             0.823151
             0.7         0.05              0.05              256               311             0.823151
             0.6         0.10              0.05              256               311             0.823151
             0.8         0.05              0.05              256               311             0.823151
             0.8         0.10              0.05              256               311             0.823151
             0.6         0.05              0.20              255               311             0.819936
             0.6         0.05              0.05 

In [ ]:
import numpy as np
import pandas as pd
import os

# 1. VERIFY VARIABLES
required = [
    "semantic_rank", "experiments", "queries",
    "candidates", "query_map", "candidate_map",
    "relevance", "validation", "holdout"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing variables: " + ", ".join(missing)
        + ". Run previous ranking cells first."
    )

rrf_scores = experiments["RRF_10"]

# 2. CALCULATE SENTENCE-BERT CONFIDENCE
def get_confidence(q):

    row = semantic_rank[query_map[q]].copy()

    if q in candidate_map:
        row[candidate_map[q]] = -np.inf

    top = np.sort(row)[-10:][::-1]

    return {
        "mean": float(np.mean(top)),
        "minimum": float(np.min(top)),
        "maximum": float(np.max(top)),
        "gap": float(top[0] - top[-1])
    }

# 3. CALCULATE HIT FOR EACH QUERY
def get_hit(scores, q):

    row = scores[query_map[q]].copy()

    if q in candidate_map:
        row[candidate_map[q]] = -np.inf

    top_indices = np.argsort(-row, kind="stable")[:10]

    recommended = {
        candidates[i] for i in top_indices
    }

    return int(bool(recommended.intersection(relevance[q])))

# 4. BASELINE VALIDATION RESULT
baseline_validation = sum(
    get_hit(semantic_rank, q) for q in validation
)

print("Sentence-BERT validation:",
      baseline_validation, "/", len(validation))

# 5. TEST ADAPTIVE THRESHOLDS
records = []

for feature in ["mean", "minimum", "maximum", "gap"]:

    confidence = {
        q: get_confidence(q)[feature]
        for q in validation
    }

    thresholds = np.unique(
        np.quantile(
            list(confidence.values()),
            np.linspace(0.05, 0.95, 37)
        )
    )

    for threshold in thresholds:

        hits = 0
        rrf_used = 0

        for q in validation:

            if confidence[q] < threshold:
                hits += get_hit(rrf_scores, q)
                rrf_used += 1
            else:
                hits += get_hit(semantic_rank, q)

        records.append({
            "feature": feature,
            "threshold": float(threshold),
            "successful_queries": hits,
            "total_queries": len(validation),
            "hit_rate": hits / len(validation),
            "rrf_queries": rrf_used
        })

results = pd.DataFrame(records)

# 6. SELECT BEST VALIDATION CONFIGURATION
results = results.sort_values(
    ["hit_rate", "successful_queries"],
    ascending=False
)

best = results.iloc[0]

print("\n===== BEST ADAPTIVE CONFIGURATION =====")
print("Feature:", best["feature"])
print("Threshold:", best["threshold"])
print("Validation successes:",
      int(best["successful_queries"]), "/", len(validation))
print("Validation Hit Rate:",
      round(best["hit_rate"] * 100, 2), "%")
print("Queries using RRF:", int(best["rrf_queries"]))

# 7. APPLY SELECTED CONFIGURATION TO HOLDOUT
feature = best["feature"]
threshold = best["threshold"]

holdout_hits = 0
rrf_count = 0
semantic_count = 0

for q in holdout:

    confidence = get_confidence(q)[feature]

    if confidence < threshold:
        hit = get_hit(rrf_scores, q)
        rrf_count += 1
    else:
        hit = get_hit(semantic_rank, q)
        semantic_count += 1

    holdout_hits += hit

holdout_rate = holdout_hits / len(holdout) * 100

print("\n===== ADAPTIVE HOLDOUT RESULT =====")
print("RRF queries:", rrf_count)
print("Sentence-BERT queries:", semantic_count)
print("Successful queries:", holdout_hits, "/", len(holdout))
print("Adaptive Hit Rate@10:", round(holdout_rate, 2), "%")

print("\n===== BASELINE COMPARISON =====")
print("Sentence-BERT: 87.18%")
print("Adaptive:", round(holdout_rate, 2), "%")

# 8. SAVE RESULTS
output_dir = (
    "/content/drive/MyDrive/GLAMOR_Project/outputs"
)

results.to_csv(
    output_dir + "/adaptive_threshold_optimization.csv",
    index=False
)

print("\nAdaptive optimization completed.")
print("Results saved successfully.")


Sentence-BERT validation: 264 / 311

===== BEST ADAPTIVE CONFIGURATION =====
Feature: mean
Threshold: 0.9994972348213196
Validation successes: 264 / 311
Validation Hit Rate: 84.89 %
Queries using RRF: 0

===== ADAPTIVE HOLDOUT RESULT =====
RRF queries: 0
Sentence-BERT queries: 78
Successful queries: 68 / 78
Adaptive Hit Rate@10: 87.18 %

===== BASELINE COMPARISON =====
Sentence-BERT: 87.18%
Adaptive: 87.18 %

Adaptive optimization completed.
Results saved successfully.
